# 📊 Análise de Carteira — Ações e FIIs (sinais de entrada/saída)
### Notebook interativo — sem necessidade de programar

Este notebook analisa sua carteira de **ações** e **FIIs** combinando dois tipos de critério:

1. **Fundamentalista** (o ativo está "caro" ou "barato" pelos seus números: P/L, P/VP, ROE, Dividend Yield...)
2. **Técnico** (o preço está numa região historicamente boa de compra/venda: médias móveis, RSI, distância de máximas/mínimas)

No final, cada ativo recebe um **sinal simples**: 🟢 possível entrada, 🟡 neutro/monitorar, 🔴 possível saída/cautela.

Com a carteira importada (Seção 9.3), a **Seção 11** mostra **cenários comparativos por ativo** — a situação em relação às faixas de preço sobre o seu preço médio e os caminhos possíveis lado a lado, sem recomendação —, com a evidência histórica de cada cenário gerada pelo backtest da **Seção 12**.

> ⚠️ **Aviso importante**: Isso é uma ferramenta educacional de apoio à análise, baseada em regras simples e transparentes.
> **Não é recomendação de investimento.** Os critérios usados são heurísticas (regras de bolso) amplamente conhecidas,
> não uma análise profissional. Sempre complemente com seu próprio julgamento, relatórios de analistas e, se possível,
> um assessor/consultor de investimentos licenciado (CVM) antes de decidir.

**Como usar**: rode as células em ordem (▶ em cada uma, ou "Ambiente de execução → Executar tudo"), preencha os tickers
na Seção 2 e clique nos botões.


In [ ]:
#@title 🔧 1. Preparar o ambiente (clique para rodar)

!uv pip install -q yfinance ipywidgets

# --- Pacote carteira_analise: o MESMO código do painel ------------------------
# Desde a Fase 1 (11/2026), as regras (motor de pontuação, regularidade dos
# dividendos, preço atrativo dos FIIs, cenários, base da B3...) não são mais
# copiadas nas células: vêm do repositório do projeto, com os testes
# automatizados do pacote. Para reproduzir exatamente a versão entregue,
# troque "main" pela tag da versão (ex.: "v1.0-pfc").
VERSAO_PACOTE = "main"  #@param {type:"string"}
REPO_PACOTE = "https://github.com/paulolmenezes-hub/carteira"
import os, sys, subprocess
PASTA_PACOTE = "/content/carteira" if os.path.isdir("/content") else os.path.abspath("carteira_pacote")
if os.path.isdir(os.path.join(PASTA_PACOTE, ".git")):
    subprocess.run(["git", "-C", PASTA_PACOTE, "fetch", "-q", "--depth", "1", "origin", VERSAO_PACOTE], check=True)
    subprocess.run(["git", "-C", PASTA_PACOTE, "checkout", "-q", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", VERSAO_PACOTE, REPO_PACOTE, PASTA_PACOTE],
                   check=True)
if PASTA_PACOTE not in sys.path:
    sys.path.insert(0, PASTA_PACOTE)
for _m in [m for m in sys.modules if m.startswith("carteira_analise")]:
    del sys.modules[_m]  # rodou de novo na mesma sessão: carrega a versão recém-baixada
_versao = subprocess.run(["git", "-C", PASTA_PACOTE, "log", "-1", "--format=%h de %cd", "--date=format:%d/%m/%Y"],
                         capture_output=True, text=True).stdout.strip()
import carteira_analise
print(f"Pacote carteira_analise ({VERSAO_PACOTE}, commit {_versao}) carregado.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import datetime
import time
from zoneinfo import ZoneInfo
import warnings
warnings.filterwarnings("ignore")

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# --- Cache local (em memória, válido durante esta sessão do Colab) -----------
# Evita repetir a mesma chamada de rede se você rodar a análise mais de uma vez
# na mesma sessão (ex: Seção 7 e depois Seção 8 pro mesmo ticker, ou clicar em
# "Analisar carteira" de novo sem mudar nada). NÃO persiste entre sessões —
# isso exigiria montar o Google Drive, o que o Colab não faz automaticamente;
# ver Seção 8.1.1 do manual sobre o cache persistente (SQLite) do pacote Python.
_cache_dados = {}
_CACHE_TTL_SEGUNDOS = 30 * 60  # 30 minutos

def _cache_get(chave):
    if chave in _cache_dados:
        valor, salvo_em = _cache_dados[chave]
        if time.time() - salvo_em < _CACHE_TTL_SEGUNDOS:
            return valor
    return None

def _cache_set(chave, valor):
    _cache_dados[chave] = (valor, time.time())

def limpar_cache():
    """Force a busca de dados atualizados, ignorando o que já está em cache
    nesta sessão. Rode esta função manualmente se desconfiar de dado defasado."""
    _cache_dados.clear()
    print(f"Cache limpo às {datetime.datetime.now(ZoneInfo('America/Sao_Paulo')).strftime('%H:%M')}.")

print("Bibliotecas carregadas.")


## 📝 2. Sua carteira

Preencha os tickers separados por vírgula, em cinco grupos: ações da B3, FIIs da B3, ações americanas,
ETFs da B3 e ETFs americanos. Ações e FIIs/ETFs da B3 usam sufixo `.SA` (ex: `PETR4.SA`, `BOVA11.SA`);
ativos americanos não usam sufixo (ex: `AAPL`, `SPY`).

- **Ações (B3)**: PETR4.SA, ITUB4.SA, WEGE3.SA ...
- **FIIs**: HGLG11.SA, KNRI11.SA, MXRF11.SA ...
- **Ações (EUA)**: AAPL, MSFT, GOOGL ...
- **ETFs (B3)**: BOVA11.SA, IVVB11.SA, SMAL11.SA, ou de renda fixa (ex: IMAB11.SA, B5P211.SA) ...
- **ETFs (EUA)**: SPY, QQQ, VOO ...

As duas caixas de ETF aceitam **qualquer tipo de ETF** (de ações, de renda fixa/juros, multimercado) —
não separamos por classe de ativo dentro de ETF, só por bolsa (B3 x EUA). Isso importa porque a
terminologia do que é distribuído muda: ETFs de ações costumam chamar de "dividendo/rendimento" (quando
distribuem — muitos são "de acumulação" e não distribuem nada, reinvestindo internamente), enquanto ETFs
de renda fixa repassam os juros dos títulos que carregam (às vezes chamado de "cupom"). Para efeito de
cálculo, o notebook trata ambos da mesma forma: soma o que a fonte de dados registrou como provento pago
nos últimos 12 meses, seja qual for o nome técnico.

Os critérios de avaliação são diferentes por grupo. Ações usam P/L, P/VP, ROE e dividend yield, com
limiares próprios para B3 e EUA (o mercado americano historicamente negocia a múltiplos mais altos).
ETFs não têm P/L, P/VP nem ROE — não são empresas, são cestas de ativos — então o critério é só técnico
(tendência, RSI, distância do período) mais yield. Em ambos os casos, ativos sem histórico de distribuição
são classificados como "não aplicável" (ações de crescimento ou ETFs de acumulação), não como sinal
negativo — é comum e esperado nesses perfis.

> ⚠️ **Não suportado: títulos de renda fixa individuais (bonds corporativos, ex: "Citigroup 6,27% 2034")**.
> O Yahoo Finance não tem cotação diária por ticker pra bonds individuais — eles negociam em balcão (OTC),
> identificados por CUSIP, não por um ticker de bolsa. Mesmo que houvesse o dado, os critérios técnicos
> desta ferramenta (RSI, médias móveis) não fazem sentido pra um bond específico, que pode passar dias sem
> negociar — a análise correta seria yield-to-maturity, duration e spread de crédito, algo fora do escopo
> deste protótipo. ETFs de renda fixa (ex: TLT, IEF, AGG, IMAB11.SA) são diferentes e **são suportados**
> normalmente nas caixas de ETF acima, porque negociam como qualquer ETF, com ticker e liquidez diária.


In [ ]:
#@title 📝 2. Digite os tickers da sua carteira

caixa_acoes = widgets.Textarea(
    value='VALE3.SA, ABCB4.SA, BBSE3.SA, BBDC4.SA, LEVE3.SA, POMO4.SA, ITSA4.SA, CMIG4.SA, BBAS3.SA, KLBN11.SA, CXSE3.SA, PSSA3.SA',
    description='Ações (B3):',
    placeholder='Ex: PETR4.SA, ITUB4.SA, WEGE3.SA',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px', height='60px')
)

caixa_fiis = widgets.Textarea(
    value='HGCR11.SA, RECR11.SA, KNSC11.SA, CPTS11.SA, VGIP11.SA, KNIP11.SA, BTHF11.SA, VRTA11.SA, BTLG11.SA, HGRE11.SA, HGLG11.SA, XPML11.SA, TRXF11.SA, JURO11.SA, TGAR11.SA, XPLG11.SA, BRCO11.SA, PVBI11.SA, BRCR11.SA, RECT11.SA, VISC11.SA, KNRI11.SA, RBRF11.SA, KNCR11.SA, MXRF11.SA, HCTR11.SA, DEVA11.SA, VGHF11.SA, IRDM11.SA, URPR11.SA',
    description='FIIs:',
    placeholder='Ex: HGLG11.SA, KNRI11.SA, MXRF11.SA',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px', height='110px')
)

caixa_acoes_us = widgets.Textarea(
    value='BRK-B',
    description='Ações (EUA):',
    placeholder='Ex: AAPL, MSFT, GOOGL',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px', height='60px')
)

caixa_etf_br = widgets.Textarea(
    value='BOVA11.SA, IVVB11.SA',
    description='ETFs (B3):',
    placeholder='Ex: BOVA11.SA, IVVB11.SA, SMAL11.SA',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px', height='60px')
)

caixa_etf_us = widgets.Textarea(
    value='QQQ, TLT, IAU, IVV, IEF, AGG, MCHI, EMXC, REMX, XME, IXG, XLE',
    description='ETFs (EUA):',
    placeholder='Ex: SPY, QQQ, VOO',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px', height='60px')
)

periodo_historico = widgets.Dropdown(
    options=['6mo', '1y', '2y', '5y'],
    value='2y',
    description='Histórico p/ análise técnica:',
    style={'description_width': 'initial'}
)

display(widgets.VBox([
    widgets.HTML("<h3>📝 Carteira</h3>"),
    caixa_acoes,
    caixa_fiis,
    caixa_acoes_us,
    caixa_etf_br,
    caixa_etf_us,
    periodo_historico
]))


## ⚙️ 3. Indicadores usados

**Técnicos** (calculados a partir do histórico de preços — Seção 4):
- Preço em relação à média móvel de 200 dias (tendência de longo prazo)
- RSI de 14 dias (sobrecomprado/sobrevendido)
- Distância em relação à faixa alta/baixa do período (percentil 98%/2%, não o máximo/mínimo
  absoluto — isso evita que um único dado ruim do provedor distorça o indicador inteiro)

**Fundamentalistas** (obtidos via Yahoo Finance — Seção 5), critérios diferentes para ações e FIIs:
- **Ações**: P/L, P/VP, ROE, Dividend Yield
- **FIIs**: P/VP, Dividend Yield, liquidez média diária

**Renda passiva** (histórico real de dividendos/rendimentos pagos, Seção 5 e 6):
- Yield efetivo dos últimos 12 meses (soma real paga ÷ preço atual — mais confiável que um yield "de tela")
- Crescimento dos dividendos em relação ao mesmo período do ano anterior
- Regularidade dos pagamentos (FIIs costumam pagar mensalmente, ações 1-2x/ano)

O resultado final traz **três colunas**: um *sinal de timing* (bom momento de comprar/vender), uma
*qualidade da renda* (dividendos crescentes e consistentes ou não) e uma **estratégia sugerida**, que combina
as duas coisas — por exemplo: se o preço subiu muito mas a renda continua saudável, o ativo não precisa ser
mantido a qualquer preço; pode valer a pena vender e recomprar depois, se/quando o preço voltar a um ponto
melhor. Já se a renda estiver piorando, isso pesa mais do que o preço, porque é sinal de que a tese original
pode ter mudado.

Cada indicador soma ou subtrai pontos de uma pontuação simples (Seção 6), que gera os sinais finais.
**Os limites usados são regras de bolso genéricas — ajuste-os na célula de pontuação se quiser refletir sua estratégia.**


In [ ]:
#@title 📈 4. Funções de indicadores técnicos

def calcular_rsi(precos, janela=14):
    delta = precos.diff()
    ganho = delta.clip(lower=0)
    perda = -delta.clip(upper=0)
    media_ganho = ganho.rolling(janela).mean()
    media_perda = perda.rolling(janela).mean()
    rs = media_ganho / media_perda.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi

def agrupar_datas_em_ciclos(datas, limite_dias=45):
    """Agrupa uma lista de datas em 'ciclos' — datas próximas (menos de `limite_dias`
    de intervalo) ficam no mesmo grupo; um intervalo maior que isso indica que o preço
    se afastou e voltou depois, ou seja, um novo ciclo. Usado para distinguir um ativo
    fazendo mínima/máxima pela primeira vez de um ativo revisitando um patamar que já
    tinha visitado antes (e recuperado no meio)."""
    datas_ordenadas = sorted(datas)
    ciclos = [[datas_ordenadas[0]]]
    for d in datas_ordenadas[1:]:
        if (d - ciclos[-1][-1]).days > limite_dias:
            ciclos.append([d])
        else:
            ciclos[-1].append(d)
    return ciclos

def pregao_provavelmente_aberto(data_ultima_cotacao):
    """Estimativa simples (não oficial) de se o pregão da B3 pode ainda estar em
    andamento no momento em que a análise é rodada. Usada só pra avisar que o preço
    mais recente pode ainda não ser o fechamento definitivo do dia — não substitui o
    calendário oficial da B3 (não considera feriados, por exemplo)."""
    agora = datetime.datetime.now(ZoneInfo('America/Sao_Paulo'))
    if data_ultima_cotacao.date() != agora.date():
        return False  # a cotação mais recente já é de um dia anterior (pregão daquele dia encerrado)
    if agora.weekday() >= 5:
        return False  # sábado ou domingo, não tem pregão
    return datetime.time(10, 0) <= agora.time() <= datetime.time(18, 0)

def limpar_outliers_precos(fechamento, janela=21, limite=0.35, iteracoes=3, janela_salto=10, fator_salto=3.0):
    """Trata dois tipos de problema de dados, de formas diferentes, NESTA ORDEM:

    1) TICKS ISOLADOS: comparação com a mediana de uma janela local (21 dias), em
       várias passadas. Roda PRIMEIRO de propósito — corrige por interpolação os
       erros pontuais de poucos dias antes de qualquer outra análise. Isso importa
       porque, se um tick isolado no MEIO do histórico for avaliado antes por um
       critério global (etapa 2), ele pode ser confundido com o início de um bloco
       inteiro fora de escala, levando a truncar (descartar) uma quantidade enorme
       de dados bons só por causa de um erro de poucos dias.

    2) BLOCO ANTIGO FORA DE ESCALA (ex: agrupamento/desdobramento de cotas que o
       provedor de dados não ajustou retroativamente): SÓ DEPOIS de (1), comparamos
       cada preço remanescente com a mediana dos ÚLTIMOS 60 DIAS (a cotação atual de
       verdade). Um trecho antigo severamente fora dessa faixa — e que sobreviveu à
       limpeza local porque é consistente com seus próprios vizinhos — é DESCARTADO
       por completo; interpolar seria arriscado quando o bloco é grande. Indicadores
       que precisam de mais dados do que sobrou (ex: MM200) ficam automaticamente
       como 'não disponível' em vez de mostrar um número calculado sobre dado ruim."""
    serie = fechamento.copy()

    total_corrigidos = 0
    for _ in range(iteracoes):
        mediana_movel = serie.rolling(janela, center=True, min_periods=1).median()
        desvio_relativo = (serie - mediana_movel).abs() / mediana_movel
        suspeitos = desvio_relativo > limite
        n_suspeitos = int(suspeitos.sum())
        if n_suspeitos == 0:
            break
        serie[suspeitos] = np.nan
        serie = serie.interpolate().ffill().bfill()
        total_corrigidos += n_suspeitos

    # 2) Descontinuidade de patamar (grupamento/desdobramento de cotas não
    #    ajustado pelo provedor): o preço muda de escala de um dia para o outro
    #    e PERMANECE no novo patamar. É detectada comparando a mediana dos
    #    `janela_salto` pregões antes e depois de cada ponto: uma mudança de
    #    `fator_salto` vezes ou mais marca o corte, e o trecho anterior é
    #    descartado. Quedas e altas REAIS, mesmo grandes, acontecem ao longo de
    #    semanas ou meses e são mantidas. A versão anterior comparava todo o
    #    histórico com o preço atual e descartava qualquer trecho 60% acima ou
    #    abaixo dele — o que apagava justamente a história dos ativos que mais
    #    caíram (viés de sobrevivência no backtest).
    n_truncados = 0
    k = janela_salto
    if len(serie) >= 2 * k:
        antes = serie.rolling(k).median().shift(1)
        depois = serie[::-1].rolling(k).median()[::-1]
        razao = np.maximum(antes / depois, depois / antes)
        saltos = np.where((razao >= fator_salto).values)[0]
        if len(saltos):
            corte = int(saltos.max())
            alvo = float(depois.iloc[corte])
            # avança até o primeiro ponto já no novo patamar (a etapa 1 pode ter
            # suavizado a fronteira em alguns pregões)
            while corte < len(serie) - 1 and abs(serie.iloc[corte] - alvo) / alvo > limite:
                corte += 1
            if 0 < corte < len(serie) - 1:
                n_truncados = corte
                serie = serie.iloc[corte:]
                total_corrigidos += n_truncados

    return serie, total_corrigidos, n_truncados


def indicadores_tecnicos(ticker, periodo):
    """Baixa histórico e calcula indicadores técnicos de um ativo."""
    chave_cache = f"precos_sem_ajuste:{ticker}:{periodo}"
    fechamento_bruto_cache = _cache_get(chave_cache)
    if fechamento_bruto_cache is not None:
        fechamento = fechamento_bruto_cache
    else:
        # auto_adjust=False: preço SEM ajuste por proventos (o padrão do yfinance
        # passou a ser True, o que contaria os proventos duas vezes no backtest).
        hist = yf.download(ticker, period=periodo, progress=False, auto_adjust=False)
        if hist.empty:
            return None
        fechamento = hist['Close']
        if isinstance(fechamento, pd.DataFrame):
            fechamento = fechamento.squeeze(axis=1)
        fechamento = fechamento.dropna()
        if not fechamento.empty:
            _cache_set(chave_cache, fechamento)

    if len(fechamento) < 30:
        return None

    fechamento, n_outliers, n_truncados = limpar_outliers_precos(fechamento)

    if len(fechamento) < 30:
        return None

    preco_atual = float(fechamento.iloc[-1])
    sma50 = fechamento.rolling(50).mean().iloc[-1] if len(fechamento) >= 50 else np.nan
    sma200 = fechamento.rolling(200).mean().iloc[-1] if len(fechamento) >= 200 else np.nan
    rsi14 = calcular_rsi(fechamento, 14).iloc[-1]
    # Percentil 5%/95%, não mínimo/máximo absolutos, e com folga maior que o normal (2%)
    # de propósito: alguns tickers da B3 têm um BLOCO inteiro de dias com preço numa escala
    # errada no histórico do Yahoo Finance (ex: um agrupamento/desdobramento de cotas que não
    # foi reajustado retroativamente). Como esses dias são consistentes ENTRE SI, o filtro de
    # outliers acima (que compara cada dia com seus vizinhos) não os pega — só um corte por
    # percentil com folga suficiente resolve.
    maxima_periodo = fechamento.quantile(0.95)
    minima_periodo = fechamento.quantile(0.05)
    dist_maxima = (preco_atual - maxima_periodo) / maxima_periodo * 100
    dist_minima = (preco_atual - minima_periodo) / minima_periodo * 100

    return {
        'preco_atual': preco_atual,
        'sma50': float(sma50) if not pd.isna(sma50) else None,
        'sma200': float(sma200) if not pd.isna(sma200) else None,
        'rsi14': float(rsi14) if not pd.isna(rsi14) else None,
        'dist_maxima_pct': float(dist_maxima),
        'dist_minima_pct': float(dist_minima),
        'n_outliers_corrigidos': n_outliers,
        'n_dias_truncados': n_truncados,
        'dias_usados': len(fechamento),
        'data_ultima_cotacao': fechamento.index[-1],
        'historico': fechamento
    }

print("Funções técnicas prontas.")


In [ ]:
#@title 💰 5. Funções de indicadores fundamentalistas

def _info_cache(ticker):
    """Busca yf.Ticker(ticker).info com cache — compartilhado entre ações e
    FIIs, já que ambos leem o mesmo dicionário 'info' por baixo."""
    chave = f"info:{ticker}"
    em_cache = _cache_get(chave)
    if em_cache is not None:
        return em_cache
    try:
        info = yf.Ticker(ticker).info
    except Exception:
        return {}
    if info:
        _cache_set(chave, info)
    return info

def indicadores_fundamentalistas_acao(ticker):
    """Busca múltiplos fundamentalistas de uma ação. Não usamos o campo
    'dividendYield' bruto do Yahoo aqui de propósito: em vários tickers da
    B3 ele vem com escala inconsistente (às vezes já em %, às vezes fração),
    o que gerava valores absurdos como '1224%'. O yield real é calculado à
    parte, em historico_dividendos(), a partir dos dividendos efetivamente
    pagos.

    Para AÇÕES DA B3, P/L, P/VP e ROE vêm preferencialmente do Fundamentus
    (sua base original e mais consolidada — mais confiável que o Yahoo
    Finance para tickers da B3); se indisponível, cai de volta no Yahoo,
    sem quebrar a análise. Dados extras do Fundamentus (ROIC, margem
    líquida, dívida bruta/patrimônio, liquidez corrente) também entram no
    dicionário. Não se aplica a ações dos EUA (Fundamentus cobre só B3)."""
    info = _info_cache(ticker)
    fund = {
        'pl': info.get('trailingPE'),
        'pvp': info.get('priceToBook'),
        'roe': info.get('returnOnEquity'),
        'divida_patrimonio': info.get('debtToEquity'),
    }
    if ticker.upper().endswith('.SA'):  # Fundamentus só cobre B3, não ações EUA
        fundamentus = buscar_fundamentos_acao_fundamentus(ticker)
        if fundamentus is not None:
            if fundamentus.get('pl') is not None:
                fund['pl'] = fundamentus['pl']
            if fundamentus.get('pvp') is not None:
                fund['pvp'] = fundamentus['pvp']
            if fundamentus.get('roe') is not None:
                fund['roe'] = fundamentus['roe']
            fund['roic'] = fundamentus.get('roic')
            fund['margem_liquida'] = fundamentus.get('margem_liquida')
            fund['divida_bruta_patrimonio'] = fundamentus.get('divida_bruta_patrimonio')
            fund['liquidez_corrente'] = fundamentus.get('liquidez_corrente')
    return fund

import io
import re
import urllib.request

_URL_FII_RESULTADO = "https://www.fundamentus.com.br/fii_resultado.php"
_USER_AGENT_FUNDAMENTUS = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
)
_cache_tabela_fundamentus = None  # cache em memória da tabela inteira (1 chamada por sessão)

def _ler_tabela_fundamentus_como_texto(html):
    """Lê a tabela mantendo todas as células como texto no formato
    brasileiro original; a conversão fica a cargo de
    _parse_numero_br_fundamentus. Com thousands='.'/decimal=',', o pandas
    convertia '0,45' em '0.45' mas mantinha como texto quando a coluna tinha
    algum '-' (bancos), e o ponto era removido como milhar: 0,45 virava 45
    (mesma correção do pacote, Sprint 3)."""
    n_colunas = len(pd.read_html(io.StringIO(html))[0].columns)
    return pd.read_html(io.StringIO(html), thousands=None,
                        converters={i: str for i in range(n_colunas)})[0]


def _parse_numero_br_fundamentus(valor):
    """Converte número em formato brasileiro ('1.234,56' ou '10,80%') para
    float. Também aceita valor já convertido para float/int pelo pandas
    (algumas colunas vêm assim, outras não, dependendo de terem '%' junto —
    reprocessar um float como se fosse texto gera erro, ver teste do pacote)."""
    if valor is None:
        return None
    if isinstance(valor, (int, float)):
        return None if (isinstance(valor, float) and pd.isna(valor)) else float(valor)
    t = str(valor).strip()
    if not t or t in ('-', 'N/A', 'nan'):
        return None
    eh_percentual = t.endswith('%')
    t = t.rstrip('%').strip().replace('.', '').replace(',', '.')
    try:
        n = float(t)
    except ValueError:
        return None
    return n / 100 if eh_percentual else n

def _baixar_html_fii_resultado():
    req = urllib.request.Request(_URL_FII_RESULTADO, headers={'User-Agent': _USER_AGENT_FUNDAMENTUS})
    with urllib.request.urlopen(req, timeout=20) as resp:
        return resp.read().decode('latin-1')

def buscar_todos_fiis_fundamentus():
    """Busca a tabela completa de FIIs do Fundamentus (site de terceiros —
    fundamentos que o Yahoo Finance não fornece de forma confiável para
    FIIs: P/VP, Dividend Yield, Segmento, FFO Yield, Vacância, Cap Rate).
    Uma chamada de rede para ~500+ fundos, reaproveitada via cache em
    memória para todos os FIIs analisados na mesma sessão."""
    html = _baixar_html_fii_resultado()
    df = _ler_tabela_fundamentus_como_texto(html)
    colunas = {c: str(c).strip().lower() for c in df.columns}

    def achar(*pistas):
        for col, nome in colunas.items():
            if any(p in nome for p in pistas):
                return col
        return None

    col_papel, col_segmento = achar('papel'), achar('segmento')
    col_pvp, col_dy, col_ffo = achar('p/vp', 'pvp'), achar('dividend'), achar('ffo')
    col_vac, col_cap, col_imoveis = achar('vac'), achar('cap rate'), achar('qtd de im', 'imóveis', 'imoveis')

    resultado = {}
    for _, linha in df.iterrows():
        ticker_bruto = linha.get(col_papel)
        if ticker_bruto is None or (isinstance(ticker_bruto, float) and pd.isna(ticker_bruto)):
            continue
        ticker = re.sub(r'\s+', '', str(ticker_bruto)).upper()
        if not ticker:
            continue
        resultado[ticker] = {
            'segmento': str(linha.get(col_segmento, '')).strip() if col_segmento else '',
            'pvp': _parse_numero_br_fundamentus(linha.get(col_pvp)) if col_pvp else None,
            'dividend_yield_fundamentus': _parse_numero_br_fundamentus(linha.get(col_dy)) if col_dy else None,
            'ffo_yield': _parse_numero_br_fundamentus(linha.get(col_ffo)) if col_ffo else None,
            'vacancia_media': _parse_numero_br_fundamentus(linha.get(col_vac)) if col_vac else None,
            'cap_rate': _parse_numero_br_fundamentus(linha.get(col_cap)) if col_cap else None,
            'qtd_imoveis': (int(_parse_numero_br_fundamentus(linha.get(col_imoveis)))
                             if col_imoveis and _parse_numero_br_fundamentus(linha.get(col_imoveis)) is not None else None),
        }
    return resultado

def buscar_fundamentos_fii_fundamentus(ticker):
    """Busca os fundamentos de um FII específico no Fundamentus, com cache
    em memória da tabela completa. Devolve None em qualquer falha (rede
    indisponível, ticker não encontrado, mudança de layout) — quem chama
    (indicadores_fundamentalistas_fii, abaixo) cai de volta no Yahoo
    Finance nesse caso, sem quebrar a análise."""
    global _cache_tabela_fundamentus
    ticker_normalizado = re.sub(r'\.SA$', '', ticker.strip().upper())
    try:
        if _cache_tabela_fundamentus is None:
            _cache_tabela_fundamentus = buscar_todos_fiis_fundamentus()
        return _cache_tabela_fundamentus.get(ticker_normalizado)
    except Exception:
        return None


def _achar_coluna_acao(colunas, *pistas):
    """Localiza uma coluna por substring, tolerando acento (a tabela de
    ações do Fundamentus tem nomes como 'Mrg. Líq.' e 'Dív.Brut/ Patrim.')."""
    def normalizar(s):
        s = s.lower()
        for de, para in (('í', 'i'), ('î', 'i'), ('ó', 'o'), ('ô', 'o'), ('á', 'a'), ('â', 'a'), ('ã', 'a')):
            s = s.replace(de, para)
        return s
    for col, nome in colunas.items():
        nome_norm = normalizar(nome)
        if any(normalizar(p) in nome_norm for p in pistas):
            return col
    return None

_URL_ACAO_RESULTADO = "https://www.fundamentus.com.br/resultado.php"
_cache_tabela_acoes_fundamentus = None  # cache em memória, independente do cache de FIIs

def _baixar_html_acao_resultado():
    req = urllib.request.Request(_URL_ACAO_RESULTADO, headers={'User-Agent': _USER_AGENT_FUNDAMENTUS})
    with urllib.request.urlopen(req, timeout=20) as resp:
        return resp.read().decode('latin-1')

def buscar_todas_acoes_fundamentus():
    """Busca a tabela completa de ações da B3 do Fundamentus — sua base
    original e mais consolidada (mais confiável que o Yahoo Finance para
    tickers da B3). Não cobre ETFs (não são empresas com balanço
    patrimonial próprio) nem ações dos EUA (cobertura restrita à B3)."""
    html = _baixar_html_acao_resultado()
    df = _ler_tabela_fundamentus_como_texto(html)
    colunas = {c: str(c).strip().lower() for c in df.columns}

    col_papel = _achar_coluna_acao(colunas, 'papel')
    col_pl = _achar_coluna_acao(colunas, 'p/l')
    col_pvp = _achar_coluna_acao(colunas, 'p/vp')
    col_roe = _achar_coluna_acao(colunas, 'roe')
    col_roic = _achar_coluna_acao(colunas, 'roic')
    col_margem = _achar_coluna_acao(colunas, 'mrg.liq', 'mrg. liq', 'margem liq')
    col_divida = _achar_coluna_acao(colunas, 'div.brut', 'dív.brut', 'divida bruta')
    col_liquidez = _achar_coluna_acao(colunas, 'liq.corr', 'liq. corr')

    resultado = {}
    for _, linha in df.iterrows():
        ticker_bruto = linha.get(col_papel)
        if ticker_bruto is None or (isinstance(ticker_bruto, float) and pd.isna(ticker_bruto)):
            continue
        ticker = re.sub(r'\s+', '', str(ticker_bruto)).upper()
        if not ticker:
            continue
        resultado[ticker] = {
            'pl': _parse_numero_br_fundamentus(linha.get(col_pl)) if col_pl else None,
            'pvp': _parse_numero_br_fundamentus(linha.get(col_pvp)) if col_pvp else None,
            'roe': _parse_numero_br_fundamentus(linha.get(col_roe)) if col_roe else None,
            'roic': _parse_numero_br_fundamentus(linha.get(col_roic)) if col_roic else None,
            'margem_liquida': _parse_numero_br_fundamentus(linha.get(col_margem)) if col_margem else None,
            'divida_bruta_patrimonio': _parse_numero_br_fundamentus(linha.get(col_divida)) if col_divida else None,
            'liquidez_corrente': _parse_numero_br_fundamentus(linha.get(col_liquidez)) if col_liquidez else None,
        }
    return resultado

def buscar_fundamentos_acao_fundamentus(ticker):
    """Busca os fundamentos de uma ação B3 específica no Fundamentus, com
    cache em memória. Devolve None em qualquer falha — quem chama
    (indicadores_fundamentalistas_acao) cai de volta no Yahoo Finance."""
    global _cache_tabela_acoes_fundamentus
    ticker_normalizado = re.sub(r'\.SA$', '', ticker.strip().upper())
    try:
        if _cache_tabela_acoes_fundamentus is None:
            _cache_tabela_acoes_fundamentus = buscar_todas_acoes_fundamentus()
        return _cache_tabela_acoes_fundamentus.get(ticker_normalizado)
    except Exception:
        return None


def indicadores_fundamentalistas_fii(ticker):
    """Busca múltiplos relevantes de um FII. P/VP e Dividend Yield vêm
    preferencialmente do Fundamentus (mais confiável para FIIs do que o
    campo 'priceToBook' do Yahoo Finance, frequentemente ausente/impreciso
    para fundos imobiliários); se o Fundamentus estiver indisponível, cai
    de volta no Yahoo Finance, sem quebrar a análise. Dados extras do
    Fundamentus (Segmento, FFO Yield, Vacância, Cap Rate) também entram no
    dicionário, mesmo sem equivalente no Yahoo."""
    info = _info_cache(ticker)
    fund = {
        'pvp': info.get('priceToBook'),
        'volume_medio': info.get('averageVolume'),
    }
    fundamentus = buscar_fundamentos_fii_fundamentus(ticker)
    if fundamentus is not None:
        if fundamentus.get('pvp') is not None:
            fund['pvp'] = fundamentus['pvp']
        fund['dividend_yield_fundamentus'] = fundamentus.get('dividend_yield_fundamentus')
        fund['segmento'] = fundamentus.get('segmento')
        fund['ffo_yield'] = fundamentus.get('ffo_yield')
        fund['vacancia_media'] = fundamentus.get('vacancia_media')
        fund['cap_rate'] = fundamentus.get('cap_rate')
        fund['qtd_imoveis'] = fundamentus.get('qtd_imoveis')
    return fund

def historico_dividendos(ticker, preco_atual):
    """Busca o histórico real de dividendos/rendimentos pagos e calcula
    yield efetivo (12m), crescimento ano contra ano e regularidade dos pagamentos.
    Isso é mais confiável que o campo 'dividendYield' isolado, que só reflete um instante."""
    chave_cache = f"dividendos:{ticker}"
    divs = _cache_get(chave_cache)
    if divs is None:
        try:
            divs = yf.Ticker(ticker).dividends
        except Exception:
            return None
        if divs is None or divs.empty:
            return None
        _cache_set(chave_cache, divs)

    if divs is None or divs.empty:
        return None

    if divs.index.tz is not None:
        divs.index = divs.index.tz_localize(None)

    hoje = divs.index.max()
    ult_12m = divs[divs.index > hoje - pd.Timedelta(days=365)]
    anterior_12m = divs[(divs.index <= hoje - pd.Timedelta(days=365)) &
                         (divs.index > hoje - pd.Timedelta(days=730))]

    soma_12m = float(ult_12m.sum())
    soma_anterior_12m = float(anterior_12m.sum())

    crescimento_yoy = None
    if soma_anterior_12m > 0:
        crescimento_yoy = (soma_12m - soma_anterior_12m) / soma_anterior_12m

    dy_12m_real = (soma_12m / preco_atual) if preco_atual else None

    return {
        'soma_12m': soma_12m,
        'soma_anterior_12m': soma_anterior_12m,
        'crescimento_yoy': crescimento_yoy,
        'dy_12m_real': dy_12m_real,
        'n_pagamentos_12m': int(len(ult_12m)),
    }

print("Funções fundamentalistas prontas.")
print("⚠️ Nem todo FII/ação da B3 tem todos os campos preenchidos no Yahoo Finance —")
print("   quando faltar um dado, o indicador aparece como 'N/D' e simplesmente não pontua.")


In [ ]:
#@title 🧮 6. Motor de pontuação (o mesmo do painel)

# Desde a Fase 1 (11/2026), o motor vem do pacote carteira_analise
# (carteira_analise/pontuacao.py) — o mesmo código do painel, coberto pelos
# testes automatizados. Esta célula só liga o motor ao CDI do Banco Central e
# aos indicadores em dicionário usados pelo notebook.
#
# Regras, em resumo:
#   • Sinal do momento: MM200, RSI e piso/topo usuais do período (percentis 5% e
#     95%), mais fundamentos (P/L, P/VP, ROE, dívida, liquidez; nos FIIs, P/VP,
#     vacância, segmento...). Com piso e topo a menos de 10% um do outro
#     (AMPLITUDE_MINIMA_FAIXA), nenhum dos dois pontua.
#   • Dividend yield "atrativo": 80% do CDI líquido dos últimos 12 meses (ações,
#     FIIs e ETFs da B3), com piso; ações e ETFs dos EUA, limites fixos.
#   • Qualidade da renda: yield, crescimento e regularidade dos pagamentos; nas
#     ações da B3, o selo de regularidade dos 5 anos fechados (Seção 14).
#   • Leitura combinada: só fundamentos e renda. Os sinais técnicos ficaram
#     próximos de "não fazer nada" no backtest (Seção 10) e aparecem só como
#     informação no detalhamento.
#
# Para experimentar outros limites nesta sessão, altere as constantes do pacote,
# por exemplo: FATOR_DY_CDI['acao'] = 0.9 (volta ao padrão ao rodar a Seção 1 de novo).

from carteira_analise import pontuacao as motor
from carteira_analise.pontuacao import (
    FATOR_DY_CDI, PISO_DY, LIMITE_DY_FIXO, AMPLITUDE_MINIMA_FAIXA,
    classificar, classificar_renda, leitura_fundamentos,
)
from carteira_analise.contexto import cdi_liquido_12m
from carteira_analise.fontes.bcb import buscar_cdi_diario
from carteira_analise.tecnicos import IndicadoresTecnicos

_CDI_MOTOR = globals().get('_CDI_MOTOR', {})  # CDI líquido de 12 meses, baixado uma vez por dia


def cdi_liquido_motor():
    """CDI acumulado dos últimos 12 meses, líquido de IR (17,5%), ou None
    (sem o Banco Central, valem os limites fixos)."""
    hoje = pd.Timestamp.today().normalize()
    if _CDI_MOTOR.get('data') != hoje:
        try:
            valor = cdi_liquido_12m(buscar_cdi_diario(hoje - pd.Timedelta(days=400), hoje))
        except Exception:
            valor = None
        _CDI_MOTOR.update(data=hoje, valor=valor)
    return _CDI_MOTOR.get('valor')


def limite_dy(tipo):
    return motor.limite_dy(tipo, cdi_liquido_motor())


def _texto_limite(tipo):
    return motor._texto_limite(tipo, cdi_liquido_motor())


def tec_para_pacote(tec):
    """Indicadores técnicos em dicionário (Seções 4 e 10) -> formato do pacote."""
    return IndicadoresTecnicos(
        preco_atual=tec['preco_atual'], sma50=tec.get('sma50'), sma200=tec.get('sma200'),
        rsi14=tec.get('rsi14'), dist_maxima_pct=tec['dist_maxima_pct'], dist_minima_pct=tec['dist_minima_pct'],
        n_outliers_corrigidos=tec.get('n_outliers_corrigidos', 0), n_dias_truncados=tec.get('n_dias_truncados', 0),
        dias_usados=tec.get('dias_usados', 0), historico=tec.get('historico', pd.Series(dtype=float)))


def pontos_tecnicos(tec):
    """Só a parte técnica do motor (MM200, RSI, piso/topo) — usada no backtest da Seção 10."""
    return motor._pontos_tecnicos(tec_para_pacote(tec), [])


_cdi = cdi_liquido_motor()
print("Motor de pontuação carregado do pacote carteira_analise (o mesmo do painel).")
print(f"Yield mínimo hoje: ação {limite_dy('acao')*100:.1f}%, FII {limite_dy('fii')*100:.1f}%, "
      f"ETF B3 {limite_dy('etf_br')*100:.1f}%".replace('.', ',')
      + (f" (80% do CDI líquido de {_cdi*100:.1f}%)".replace('.', ',') if _cdi else " (limites fixos: sem o CDI)"))


In [ ]:
#@title 🚀 7. Rodar a análise da carteira inteira

botao_analisar = widgets.Button(description="🚀 Analisar carteira", button_style='primary', icon='play')
saida_analise = widgets.Output()

import re


def parse_tickers(texto):
    """Quebra a lista de tickers (vírgula, ponto e vírgula, espaço ou quebra de
    linha) e remove repetições. Corrige automaticamente um erro de
    digitação comum: vírgula no lugar do ponto antes do sufixo de bolsa
    (ex: 'VALE3, SA' vira, sem querer, dois tokens 'VALE3' e 'SA' — aqui a gente
    detecta esse padrão e reconstrói como 'VALE3.SA')."""
    # Aceita vírgula, ponto e vírgula, espaço ou quebra de linha como separador
    brutos = [t.strip().upper() for t in re.split(r'[,;\s]+', texto) if t.strip()]

    tickers = []
    i = 0
    while i < len(brutos):
        atual = brutos[i]
        proximo = brutos[i + 1] if i + 1 < len(brutos) else None
        if proximo in ('SA', 'US') and not atual.endswith('.' + proximo):
            tickers.append(f"{atual}.{proximo}")
            i += 2
        else:
            tickers.append(atual)
            i += 1
    return list(dict.fromkeys(tickers))  # remove repetições, mantendo a ordem

from carteira_analise.carteira import analisar_ativo
from carteira_analise.fontes import yahoo as fonte_yahoo
from carteira_analise.fontes.fundamentus import buscar_proventos_fii
from carteira_analise.fontes.rendimentos import FonteComRendimentosConferidos
from carteira_analise.fundamentalistas import texto_regularidade

_SELO_REGULARIDADE = {'regular': '🟢 Regular', 'oscilante': '🟡 Oscilante', 'irregular': '🔴 Irregular',
                      'incompleto': '⚪ Sem selo (lacunas na fonte)'}


def analisar_grupo(tickers, tipo, periodo):
    """Análise de cada ativo pelo pacote carteira_analise (``analisar_ativo``),
    a mesma do painel: indicadores, fundamentos (Yahoo + Fundamentus), renda,
    regularidade dos dividendos (ações) e, nos FIIs, os rendimentos conferidos
    entre Fundamentus e Yahoo."""
    linhas = []
    detalhes_por_ativo = {}
    moeda = 'US$' if tipo in ('acao_us', 'etf_us') else 'R$'
    fonte = FonteComRendimentosConferidos(fonte_yahoo, {t: tipo for t in tickers},
                                          lambda t: {"fundamentus": buscar_proventos_fii(t)})
    cdi = cdi_liquido_motor()
    for ticker in tickers:
        try:
            res = analisar_ativo(ticker, tipo, periodo, fonte, cdi_liquido_aa=cdi)
        except Exception as erro:
            print(f"  ⚠️ {ticker}: falha ao buscar os dados ({erro})")
            res = None
        if res is None:
            linhas.append({'Ticker': ticker, 'Sinal de timing': '⚠️ sem dados suficientes', 'Pontos (timing)': None})
            continue
        tec, fund, div = res.tec, res.fund, res.div
        data_ultima = tec.historico.index[-1]
        detalhes = list(res.detalhes_timing)
        detalhes_renda = list(res.detalhes_renda)

        # Pregão da B3 só se aplica a ativos negociados na B3 (fuso e horário de
        # negociação diferentes para ativos americanos).
        pregao_aberto = pregao_provavelmente_aberto(data_ultima) if tipo in ('acao', 'fii', 'etf_br') else False

        if tec.n_dias_truncados:
            detalhes.append(
                f"⚠️ {tec.n_dias_truncados} dia(s) mais antigos do histórico foram descartados "
                f"(mudança de patamar do preço de um dia para o outro — provável grupamento ou "
                f"desdobramento de cotas não ajustado pelo provedor). Indicadores calculados com os "
                f"{tec.dias_usados} dias restantes, mais confiáveis."
            )
        elif tec.n_outliers_corrigidos:
            detalhes.append(f"⚠️ {tec.n_outliers_corrigidos} ponto(s) de preço suspeito(s) corrigido(s) no histórico")

        if pregao_aberto:
            detalhes.append(
                "📊 Pregão provavelmente em andamento — este preço é o mais recente disponível, "
                "não necessariamente o fechamento definitivo do dia. Os indicadores podem mudar "
                "se a análise for rodada de novo após o fechamento (~18h, horário de Brasília)."
            )

        if res.regularidade:
            detalhes_renda.append("Dividendos regulares? " + texto_regularidade(res.regularidade))

        cotacao_texto = data_ultima.strftime('%d/%m/%Y')
        if pregao_aberto:
            cotacao_texto += ' (pregão aberto)'

        linha = {
            'Ticker': ticker + (' ⚠️' if (tec.n_outliers_corrigidos or tec.n_dias_truncados) else ''),
            'Preço': f"{moeda} {tec.preco_atual:.2f}",
            'Última cotação': cotacao_texto,
            'RSI(14)': round(tec.rsi14, 1) if tec.rsi14 else None,
            'Dist. mín. (p5)': f"{tec.dist_minima_pct:.1f}%",
            'Dist. máx. (p95)': f"{tec.dist_maxima_pct:.1f}%",
            'P/VP': round(fund.get('pvp'), 2) if fund.get('pvp') else 'N/D',
            'Yield 12m (real)': f"{div.dy_12m_real*100:.1f}%" if div and div.dy_12m_real is not None else 'N/D',
            'Dividendos 12m': f"{moeda} {div.soma_12m:.2f}" if div else 'N/D',
            'Cresc. dividendos (a/a)': f"{div.crescimento_yoy*100:+.1f}%" if div and div.crescimento_yoy is not None else 'N/D',
            'Nº pagamentos 12m': div.n_pagamentos_12m if div else 'N/D',
        }
        if tipo == 'acao':
            linha['Dividendos regulares?'] = (_SELO_REGULARIDADE.get(res.regularidade['selo'], res.regularidade['selo'])
                                              if res.regularidade else 'N/D')
        linha.update({
            'Sinal de timing': res.sinal_timing,
            'Pontos (timing)': res.pontos_timing,  # pontuação numérica por trás do sinal (🟢/🟡/🔴)
            'Qualidade da renda': res.qualidade_renda,
            'Leitura combinada': res.leitura_combinada_texto,
        })
        linhas.append(linha)
        detalhes_por_ativo[ticker] = {'timing': detalhes, 'renda': detalhes_renda}

    tabela = pd.DataFrame(linhas)
    for coluna in ('Pontos (timing)', 'Nº pagamentos 12m'):  # inteiros, mesmo com um ativo sem dados
        if coluna in tabela:
            try:
                tabela[coluna] = tabela[coluna].astype('Int64')
            except (TypeError, ValueError):
                pass
    return tabela, detalhes_por_ativo

def analisar_carteira(b):
    with saida_analise:
        clear_output()
        tickers_acoes = parse_tickers(caixa_acoes.value)
        tickers_fiis = parse_tickers(caixa_fiis.value)
        tickers_acoes_us = parse_tickers(caixa_acoes_us.value)
        tickers_etf_br = parse_tickers(caixa_etf_br.value)
        tickers_etf_us = parse_tickers(caixa_etf_us.value)
        periodo = periodo_historico.value

        agora = datetime.datetime.now(ZoneInfo('America/Sao_Paulo')).strftime('%d/%m/%Y às %H:%M (horário de Brasília)')
        print(f"🕒 Análise executada em {agora}")
        print(f"Analisando {len(tickers_acoes)} ações BR, {len(tickers_fiis)} FIIs, {len(tickers_acoes_us)} ações EUA, "
              f"{len(tickers_etf_br)} ETFs BR e {len(tickers_etf_us)} ETFs EUA...\n")

        def imprimir_detalhes(det):
            for ticker, grupos in det.items():
                print(f"\n  {ticker}:")
                if grupos.get('timing'):
                    print("    Timing (compra/venda):")
                    for d in grupos['timing']:
                        print(f"      {d}")
                if grupos.get('renda'):
                    print("    Qualidade da renda:")
                    for d in grupos['renda']:
                        print(f"      {d}")

        if tickers_acoes:
            df_acoes, det_acoes = analisar_grupo(tickers_acoes, 'acao', periodo)
            print("📈 AÇÕES (B3)")
            display(df_acoes)
            imprimir_detalhes(det_acoes)
            print("\n" + "-"*70 + "\n")

        if tickers_fiis:
            df_fiis, det_fiis = analisar_grupo(tickers_fiis, 'fii', periodo)
            print("🏢 FIIs")
            display(df_fiis)
            imprimir_detalhes(det_fiis)
            print("\n" + "-"*70 + "\n")

        if tickers_acoes_us:
            df_acoes_us, det_acoes_us = analisar_grupo(tickers_acoes_us, 'acao_us', periodo)
            print("🇺🇸 AÇÕES (EUA)")
            display(df_acoes_us)
            imprimir_detalhes(det_acoes_us)
            print("\n" + "-"*70 + "\n")

        if tickers_etf_br:
            df_etf_br, det_etf_br = analisar_grupo(tickers_etf_br, 'etf_br', periodo)
            print("📊 ETFs (B3)")
            display(df_etf_br)
            imprimir_detalhes(det_etf_br)
            print("\n" + "-"*70 + "\n")

        if tickers_etf_us:
            df_etf_us, det_etf_us = analisar_grupo(tickers_etf_us, 'etf_us', periodo)
            print("📊 ETFs (EUA)")
            display(df_etf_us)
            imprimir_detalhes(det_etf_us)

        print("\n📌 Como ler esta tabela: 'Sinal de timing' descreve a tendência predominante dos")
        print("   indicadores técnicos no momento (não é uma recomendação de compra/venda).")
        print("   'Qualidade da renda' descreve se os dividendos pagos têm sido crescentes e")
        print("   consistentes. 'Leitura combinada' resume fundamentos e renda (os sinais técnicos")
        print("   ficaram próximos de 'não fazer nada' no backtest da Seção 10 e aparecem só no")
        print("   detalhamento) — continua sendo descrição do que os números mostram, não uma")
        print("   indicação do que fazer. A decisão de comprar, manter ou vender é sempre do usuário.")
        print("   ⚠️ ao lado de um ticker NÃO é um sinal — é só aviso de que o histórico de preço")
        print("   desse ativo precisou de algum ajuste (dado ruim do provedor), sem relação com os")
        print("   sinais coloridos. Detalhes de cada ajuste aparecem no detalhamento abaixo.")
        print("   ⚪ 'Não aplicável' na qualidade da renda (comum em ações EUA de crescimento e em")
        print("   ETFs de acumulação) não é um sinal negativo — só indica que o ativo não distribui")
        print("   proventos regularmente. ETFs não têm P/L/P/VP/ROE (não são empresas), por isso o")
        print("   componente técnico deles usa só tendência/RSI/distância do período mais yield.")
        print("   Esta ferramenta é estritamente informativa: não constitui recomendação de")
        print("   investimento, consultoria ou análise de valores mobiliários.")

botao_analisar.on_click(analisar_carteira)
display(botao_analisar, saida_analise)


## 🔍 8. Analisar um ativo específico

Quer olhar só um ativo, com todos os critérios (timing, qualidade da renda, estratégia sugerida)
e o gráfico de preço/médias móveis/RSI, sem rodar a carteira inteira de novo? Use a célula abaixo.
Ela usa exatamente a mesma lógica e os mesmos dados da Seção 7 — os números batem 100% com a tabela
principal, é só uma forma mais direta de olhar um ativo isolado.


In [ ]:
#@title 🔍 8. Analisar um ativo específico

caixa_ticker_individual = widgets.Text(
    value='PETR4.SA',
    description='Ticker:',
    style={'description_width': 'initial'}
)
tipo_ticker_individual = widgets.Dropdown(
    options=[('Ação (B3)', 'acao'), ('FII', 'fii'), ('Ação (EUA)', 'acao_us'),
             ('ETF (B3)', 'etf_br'), ('ETF (EUA)', 'etf_us')],
    value='acao',
    description='Tipo:',
    style={'description_width': 'initial'}
)
botao_individual = widgets.Button(description="🔍 Analisar ativo", button_style='info', icon='search')
saida_individual = widgets.Output()

def analisar_ativo_individual(b):
    with saida_individual:
        clear_output()
        ticker = caixa_ticker_individual.value.strip().upper()
        tipo = tipo_ticker_individual.value
        periodo = periodo_historico.value

        # Reaproveita a MESMA função usada na análise da carteira inteira (Seção 7),
        # só que com uma lista de 1 ticker — garante que os números batem exatamente
        # com os da tabela principal, sem duplicar lógica de cálculo em dois lugares.
        agora = datetime.datetime.now(ZoneInfo('America/Sao_Paulo')).strftime('%d/%m/%Y às %H:%M (horário de Brasília)')
        print(f"🕒 Análise executada em {agora}")
        df, detalhes_por_ativo = analisar_grupo([ticker], tipo, periodo)

        if df.empty or 'Sinal de timing' not in df.columns or df.iloc[0].get('Sinal de timing') == '⚠️ sem dados suficientes':
            print(f"⚠️ Não foi possível obter dados suficientes para {ticker}.")
            return

        linha = df.iloc[0]
        nomes_tipo = {'acao': 'Ação (B3)', 'fii': 'FII', 'acao_us': 'Ação (EUA)',
                      'etf_br': 'ETF (B3)', 'etf_us': 'ETF (EUA)'}
        print(f"📌 {ticker}  —  {nomes_tipo.get(tipo, tipo)}\n")
        for coluna in df.columns:
            if coluna == 'Ticker':
                continue
            print(f"  {coluna}: {linha[coluna]}")

        print()
        grupos = detalhes_por_ativo.get(ticker, {})
        if grupos.get('timing'):
            print("  Timing (compra/venda):")
            for d in grupos['timing']:
                print(f"    {d}")
        if grupos.get('renda'):
            print("  Qualidade da renda:")
            for d in grupos['renda']:
                print(f"    {d}")

        tec = indicadores_tecnicos(ticker, periodo)
        precos = tec['historico']
        mm50 = precos.rolling(50).mean()
        mm200 = precos.rolling(200).mean()
        rsi = calcular_rsi(precos, 14)

        # Diagnóstico automático: mostra os preços mais extremos do histórico BRUTO
        # (antes de qualquer limpeza), pra você conferir rapidamente se uma queda ou
        # alta forte no gráfico é dado real ou coisa de provedor de dados. Reaproveita
        # o cache que indicadores_tecnicos() já preencheu logo acima — evita baixar o
        # mesmo dado duas vezes no mesmo clique.
        fechamento_bruto = _cache_get(f"precos_sem_ajuste:{ticker}:{periodo}")
        if fechamento_bruto is None:
            hist_bruto = yf.download(ticker, period=periodo, progress=False, auto_adjust=False)
            fechamento_bruto = hist_bruto['Close']
            if isinstance(fechamento_bruto, pd.DataFrame):
                fechamento_bruto = fechamento_bruto.squeeze(axis=1)
            fechamento_bruto = fechamento_bruto.dropna()

        print("\n🔍 Diagnóstico do histórico bruto (antes de qualquer limpeza):")
        print(f"  Total de dias no histórico: {len(fechamento_bruto)}")
        print(f"  Preço atual: {fechamento_bruto.iloc[-1]:.2f}")
        print("\n  10 MENORES preços do período (data e valor):")
        print(fechamento_bruto.nsmallest(10))
        print("\n  10 MAIORES preços do período (data e valor):")
        print(fechamento_bruto.nlargest(10))
        print("\n  Se algum desses valores parecer fora do padrão dos vizinhos, é sinal de dado ruim")
        print("  do provedor (o que o notebook já tenta corrigir automaticamente); se os valores")
        print("  parecerem consistentes com uma queda/alta real e gradual, é movimento genuíno do ativo.")

        # Leitura automática do padrão dos preços extremos: só faz sentido analisar ciclos
        # do lado (mínima ou máxima) que o preço ATUAL está de fato perto — analisar o lado
        # errado (ex: procurar padrão na mínima quando o preço está no meio do range) gera
        # leitura confusa/alarmista à toa.
        data_mais_recente = fechamento_bruto.index.max()
        ciclos_minimas = agrupar_datas_em_ciclos(list(fechamento_bruto.nsmallest(10).index))
        ciclos_maximas = agrupar_datas_em_ciclos(list(fechamento_bruto.nlargest(10).index))
        dias_desde_ciclo_recente_min = (data_mais_recente - ciclos_minimas[-1][-1]).days
        dias_desde_ciclo_recente_max = (data_mais_recente - ciclos_maximas[-1][-1]).days

        perto_da_minima = tec['dist_minima_pct'] < 15
        perto_da_maxima = tec['dist_maxima_pct'] > -5

        print("\n🧭 Leitura automática:")
        if perto_da_minima:
            if len(ciclos_minimas) == 1 and dias_desde_ciclo_recente_min <= 90:
                print("  O preço está perto da mínima do período, e os menores preços do histórico estão")
                print("  concentrados nos últimos ~3 meses — queda recente e aparentemente real, sem")
                print("  precedente parecido dentro do período analisado.")
            elif len(ciclos_minimas) > 1:
                primeiro = ciclos_minimas[0]
                ultimo = ciclos_minimas[-1]
                print(f"  O preço está perto da mínima do período, e o ativo já esteve num patamar parecido")
                print(f"  antes: identifiquei {len(ciclos_minimas)} período(s) de mínima separados por")
                print(f"  recuperações no meio. O mais antigo foi por volta de {primeiro[0].date()}, e o mais")
                print(f"  recente começou por volta de {ultimo[0].date()}. Ou seja, essa não é a primeira vez")
                print("  que o ativo chega perto desse preço — já recuperou dessa faixa ao menos uma vez")
                print("  antes. Vale entender se o motivo daquela recuperação anterior (resultado, notícia,")
                print("  ciclo do setor) ainda se aplica agora, ou se dessa vez é diferente.")
            else:
                print("  O preço está perto da mínima do período, mas os menores preços do histórico não")
                print("  seguem um padrão simples de queda contínua nem de ciclos repetidos claros. Vale")
                print("  conferir as datas acima manualmente.")
        elif perto_da_maxima:
            if len(ciclos_maximas) == 1 and dias_desde_ciclo_recente_max <= 90:
                print("  O preço está perto da máxima do período, e os maiores preços do histórico estão")
                print("  concentrados nos últimos ~3 meses — alta recente e aparentemente real, sem")
                print("  precedente parecido dentro do período analisado.")
            elif len(ciclos_maximas) > 1:
                primeiro = ciclos_maximas[0]
                ultimo = ciclos_maximas[-1]
                print(f"  O preço está perto da máxima do período, e o ativo já chegou perto desse patamar")
                print(f"  alto antes: identifiquei {len(ciclos_maximas)} período(s) de máxima separados por")
                print(f"  quedas no meio. O mais antigo foi por volta de {primeiro[0].date()}, e o mais recente")
                print(f"  começou por volta de {ultimo[0].date()}. Ou seja, o ativo já esbarrou nesse teto antes")
                print("  e recuou depois — vale considerar se esse nível costuma ser um ponto de resistência")
                print("  pra esse ativo.")
            else:
                print("  O preço está perto da máxima do período, mas os maiores preços do histórico não")
                print("  seguem um padrão simples de alta contínua nem de ciclos repetidos claros. Vale")
                print("  conferir as datas acima manualmente.")
        else:
            data_minima_abs = fechamento_bruto.idxmin()
            valor_minima_abs = fechamento_bruto.min()
            data_maxima_abs = fechamento_bruto.idxmax()
            valor_maxima_abs = fechamento_bruto.max()
            moeda_leitura = 'US$' if tipo in ('acao_us', 'etf_us') else 'R$'
            print(f"  O preço atual ({moeda_leitura} {tec['preco_atual']:.2f}) não está perto nem da mínima nem da máxima")
            print(f"  do período — está numa faixa intermediária. A mínima do período foi {moeda_leitura}")
            print(f"  {valor_minima_abs:.2f} em {data_minima_abs.date()}, e a máxima foi {moeda_leitura} {valor_maxima_abs:.2f}")
            print(f"  em {data_maxima_abs.date()}. Nada de especial pra verificar aqui.")

        if tec['dist_minima_pct'] < 15 and tec.get('rsi14') is not None and tec['rsi14'] < 35:
            print("\n  ⚠️ Atenção: o ativo está próximo da mínima do período E com RSI baixo ao mesmo")
            print("  tempo. Isso pode ser uma barganha genuína (preço descontado, prestes a reagir),")
            print("  ou pode ser uma queda estrutural que ainda não acabou — RSI baixo não garante que")
            print("  o fundo já foi atingido. Este notebook não distingue as duas coisas; vale checar")
            print("  se há notícias recentes (resultados, dívida, setor) que expliquem a queda antes")
            print("  de decidir.")

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                                        gridspec_kw={'height_ratios': [3, 1]})
        ax1.plot(precos.index, precos.values, label='Preço', color='#1f77b4')
        ax1.plot(mm50.index, mm50.values, label='MM50', color='#ff7f0e', linewidth=1)
        ax1.plot(mm200.index, mm200.values, label='MM200', color='#d62728', linewidth=1)
        titulo_grafico = f"{ticker} — preço e médias móveis"
        if tec.get('n_dias_truncados'):
            titulo_grafico += f" (histórico limpo: {tec['dias_usados']} dias confiáveis)"
        ax1.set_title(titulo_grafico)
        ax1.legend(); ax1.grid(alpha=0.3)

        ax2.plot(rsi.index, rsi.values, color='#9467bd')
        ax2.axhline(70, color='red', linestyle='--', alpha=0.5)
        ax2.axhline(30, color='green', linestyle='--', alpha=0.5)
        ax2.set_title("RSI (14)"); ax2.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

botao_individual.on_click(analisar_ativo_individual)
display(widgets.VBox([caixa_ticker_individual, tipo_ticker_individual, botao_individual, saida_individual]))


## 💹 9. Ganhos e proventos da posição

Duas formas de acompanhar o ganho de um ativo, conforme o quanto de detalhe você tiver à mão:

- **9.1 Ganho de posição aberta**: só quantidade, preço médio e data de início — não exige nenhum histórico de operações.
- **9.2 Registro de operações**: se você quiser registrar compras e vendas ao longo do tempo, a partir de agora, a ferramenta calcula o ganho já realizado nas vendas, o que ainda está em aberto, e a renda recebida — sem precisar reconstruir operações antigas.

In [ ]:
#@title 💹 9. Ganhos e proventos da posição

def _dividendos_brutos_cache(ticker):
    """Busca (com cache) o histórico bruto de dividendos de um ticker — mesma
    fonte usada por historico_dividendos() na Seção 5, reaproveitada aqui pra
    somar proventos recebidos num período qualquer."""
    chave_cache = f"dividendos:{ticker}"
    divs = _cache_get(chave_cache)
    if divs is None:
        try:
            divs = yf.Ticker(ticker).dividends
        except Exception:
            return pd.Series(dtype=float)
        if divs is None or divs.empty:
            return pd.Series(dtype=float)
        _cache_set(chave_cache, divs)
    if divs.index.tz is not None:
        divs = divs.copy()
        divs.index = divs.index.tz_localize(None)
    return divs


def calcular_ganho_posicao_aberta(quantidade, preco_medio, data_inicio, preco_atual, dividendos=None):
    """Ganho de uma posição ainda aberta, a partir de dado agregado mínimo
    (quantidade, preço médio, data de início) — não exige nenhum histórico de
    operações. A renda recebida é somada automaticamente a partir do
    histórico real de dividendos do ativo desde a data de início informada."""
    if quantidade <= 0:
        raise ValueError("quantidade deve ser positiva")
    if preco_medio <= 0:
        raise ValueError("preco_medio deve ser positivo")

    valor_investido = quantidade * preco_medio
    valor_atual = quantidade * preco_atual
    ganho_preco = valor_atual - valor_investido
    ganho_preco_pct = ganho_preco / valor_investido

    renda_recebida = 0.0
    if dividendos is not None and not dividendos.empty:
        corte = pd.Timestamp(data_inicio)
        renda_recebida = float(dividendos[dividendos.index >= corte].sum()) * quantidade

    ganho_total = ganho_preco + renda_recebida
    ganho_total_pct = ganho_total / valor_investido

    return {
        'valor_investido': valor_investido, 'valor_atual': valor_atual,
        'ganho_preco': ganho_preco, 'ganho_preco_pct': ganho_preco_pct,
        'renda_recebida': renda_recebida,
        'ganho_total': ganho_total, 'ganho_total_pct': ganho_total_pct,
    }


def avaliar_operacoes(operacoes, preco_atual=None, dividendos=None):
    """Processa uma lista de operações de compra/venda (dicts com 'data',
    'tipo', 'quantidade', 'preco') em ordem cronológica, usando custo médio
    ponderado — o mesmo método usado pela Receita Federal para apuração de
    ganho de capital no Brasil. Registro incremental e opcional: não exige
    reconstrução do histórico anterior ao uso desta funcionalidade."""
    if not operacoes:
        raise ValueError("é necessário informar ao menos uma operação")

    ops = sorted(operacoes, key=lambda o: o['data'])
    quantidade_aberta = 0.0
    custo_medio = 0.0
    ganho_realizado = 0.0
    detalhes = []

    for op in ops:
        if op['tipo'] == 'compra':
            novo_custo_total = custo_medio * quantidade_aberta + op['preco'] * op['quantidade']
            quantidade_aberta += op['quantidade']
            custo_medio = novo_custo_total / quantidade_aberta if quantidade_aberta > 0 else 0.0
            detalhes.append(f"{op['data']}: compra de {op['quantidade']:.0f} a {op['preco']:.2f} "
                             f"— posição passa a {quantidade_aberta:.0f} (custo médio {custo_medio:.2f})")
        elif op['tipo'] == 'venda':
            if op['quantidade'] > quantidade_aberta + 1e-9:
                raise ValueError(f"venda de {op['quantidade']} em {op['data']} excede a posição em "
                                  f"aberto ({quantidade_aberta}) — confira as operações informadas")
            ganho_da_venda = (op['preco'] - custo_medio) * op['quantidade']
            ganho_realizado += ganho_da_venda
            quantidade_aberta -= op['quantidade']
            detalhes.append(f"{op['data']}: venda de {op['quantidade']:.0f} a {op['preco']:.2f} "
                             f"(custo médio {custo_medio:.2f}) — ganho realizado de {ganho_da_venda:+.2f}")
            if quantidade_aberta <= 1e-9:
                quantidade_aberta = 0.0
                custo_medio = 0.0
        else:
            raise ValueError(f"tipo de operação inválido: {op['tipo']!r} (use 'compra' ou 'venda')")

    ganho_nao_realizado = None
    if quantidade_aberta > 0 and preco_atual is not None:
        ganho_nao_realizado = (preco_atual - custo_medio) * quantidade_aberta

    renda_recebida = 0.0
    if dividendos is not None and not dividendos.empty:
        data_inicio = ops[0]['data']
        corte = pd.Timestamp(data_inicio)
        soma_por_cota = float(dividendos[dividendos.index >= corte].sum())
        qtd_acum, quantidades = 0.0, []
        for op in ops:
            qtd_acum += op['quantidade'] if op['tipo'] == 'compra' else -op['quantidade']
            quantidades.append(max(qtd_acum, 0.0))
        quantidade_media_periodo = sum(quantidades) / len(quantidades) if quantidades else 0.0
        renda_recebida = soma_por_cota * quantidade_media_periodo

    ganho_total = ganho_realizado + (ganho_nao_realizado or 0.0) + renda_recebida

    return {
        'quantidade_aberta': quantidade_aberta,
        'preco_medio_aberto': custo_medio if quantidade_aberta > 0 else None,
        'ganho_realizado': ganho_realizado, 'ganho_nao_realizado': ganho_nao_realizado,
        'renda_recebida': renda_recebida, 'ganho_total': ganho_total, 'detalhes': detalhes,
    }


# --- Painel 9.1: Ganho de posição aberta (dado agregado, sem histórico) ---
ticker_ganho = widgets.Text(value='PETR4.SA', description='Ticker:', style={'description_width': 'initial'})
qtd_ganho = widgets.FloatText(value=100, description='Quantidade:', style={'description_width': 'initial'})
preco_medio_ganho = widgets.FloatText(value=30.0, description='Preço médio:', style={'description_width': 'initial'})
data_inicio_ganho = widgets.Text(value='2024-01-01', description='Data início (AAAA-MM-DD):', style={'description_width': 'initial'})
botao_ganho = widgets.Button(description="💹 Calcular ganho", button_style='success', icon='calculator')
saida_ganho = widgets.Output()

def _calcular_ganho_click(b):
    with saida_ganho:
        clear_output()
        ticker = ticker_ganho.value.strip().upper()
        tec = indicadores_tecnicos(ticker, periodo_historico.value)
        if tec is None:
            print(f"⚠️ Não foi possível obter dados suficientes para {ticker}.")
            return
        divs = _dividendos_brutos_cache(ticker)
        r = calcular_ganho_posicao_aberta(
            quantidade=qtd_ganho.value, preco_medio=preco_medio_ganho.value,
            data_inicio=data_inicio_ganho.value, preco_atual=tec['preco_atual'], dividendos=divs,
        )
        print(f"📌 {ticker} — {qtd_ganho.value:.0f} cotas desde {data_inicio_ganho.value}\n")
        print(f"  Valor investido: R$ {r['valor_investido']:.2f}")
        print(f"  Valor atual: R$ {r['valor_atual']:.2f}")
        print(f"  Ganho de preço: R$ {r['ganho_preco']:+.2f} ({r['ganho_preco_pct']*100:+.1f}%)")
        print(f"  Renda recebida (proventos desde o início): R$ {r['renda_recebida']:.2f}")
        print(f"  Ganho total: R$ {r['ganho_total']:+.2f} ({r['ganho_total_pct']*100:+.1f}%)")

botao_ganho.on_click(_calcular_ganho_click)
display(widgets.HTML("<h3>9.1 Ganho de posição aberta (dado agregado)</h3>"))
display(widgets.VBox([ticker_ganho, qtd_ganho, preco_medio_ganho, data_inicio_ganho, botao_ganho, saida_ganho]))


# --- Painel 9.2: Registro de operações (compra/venda) e posições encerradas ---
ticker_ops = widgets.Text(value='PETR4.SA', description='Ticker:', style={'description_width': 'initial'})
texto_ops = widgets.Textarea(
    value='2023-05-10,compra,100,25.00\n2023-11-20,compra,100,30.00\n2024-08-05,venda,80,35.00',
    description='Operações:', placeholder='data,tipo,quantidade,preco (uma por linha)',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='500px', height='90px'),
)
botao_ops = widgets.Button(description="📋 Avaliar operações", button_style='success', icon='list')
saida_ops = widgets.Output()

def _avaliar_operacoes_click(b):
    with saida_ops:
        clear_output()
        ticker = ticker_ops.value.strip().upper()
        linhas = [l.strip() for l in texto_ops.value.split('\n') if l.strip()]
        operacoes = []
        for linha in linhas:
            try:
                data_str, tipo, qtd_str, preco_str = [p.strip() for p in linha.split(',')]
                operacoes.append({
                    'data': pd.Timestamp(data_str), 'tipo': tipo,
                    'quantidade': float(qtd_str), 'preco': float(preco_str),
                })
            except Exception:
                print(f"⚠️ Linha inválida, ignorada: {linha!r} (formato esperado: data,tipo,quantidade,preco)")

        if not operacoes:
            print("⚠️ Nenhuma operação válida informada.")
            return

        tec = indicadores_tecnicos(ticker, periodo_historico.value)
        preco_atual = tec['preco_atual'] if tec is not None else None
        divs = _dividendos_brutos_cache(ticker)

        try:
            r = avaliar_operacoes(operacoes, preco_atual=preco_atual, dividendos=divs)
        except ValueError as e:
            print(f"⚠️ {e}")
            return

        print(f"📌 {ticker} — {len(operacoes)} operação(ões) registrada(s)\n")
        for d in r['detalhes']:
            print(" ", d)
        print()
        if r['quantidade_aberta'] > 0:
            print(f"  Posição ainda aberta: {r['quantidade_aberta']:.0f} cotas a custo médio R$ {r['preco_medio_aberto']:.2f}")
            if r['ganho_nao_realizado'] is not None:
                print(f"  Ganho não realizado: R$ {r['ganho_nao_realizado']:+.2f}")
        else:
            print("  Posição totalmente encerrada.")
        print(f"  Ganho realizado (vendas já feitas): R$ {r['ganho_realizado']:+.2f}")
        print(f"  Renda recebida no período: R$ {r['renda_recebida']:.2f}")
        print(f"  Ganho total: R$ {r['ganho_total']:+.2f}")
        print("\n  📌 A renda recebida é aproximada pela quantidade média mantida ao longo do")
        print("  período coberto pelas operações informadas — não célula a célula por operação.")

botao_ops.on_click(_avaliar_operacoes_click)
display(widgets.HTML("<h3>9.2 Registro de operações e posições encerradas</h3>"))
display(widgets.HTML("<p>Uma operação por linha, no formato <code>data,tipo,quantidade,preco</code> "
                      "(ex: <code>2024-01-15,compra,100,25.50</code>). Não é preciso reconstruir "
                      "operações antigas — comece a registrar a partir de agora.</p>"))
display(widgets.VBox([ticker_ops, texto_ops, botao_ops, saida_ops]))


## 💼 9.3 Ganho consolidado da carteira (via planilha)

Importa a carteira de um arquivo Excel/CSV ou Google Sheets, combinando duas abas por ativo:

- **Resumo** — a posição inicial de cada ativo, o ponto de partida: `ticker`, `quantidade`, `preco_medio` (ou `valor_investido`), `data_inicio`.
- **Operações** — as movimentações feitas a partir dali: `ticker`, `tipo` (compra/venda), `quantidade`, `preco`, `data`.

Um ativo pode ter só Resumo, só Operações, ou as duas — quando tiver as duas, a posição inicial (Resumo) entra como o primeiro registro, e cada operação lançada depois é aplicada **em cima** dela, ajustando quantidade, preço médio e ganho automaticamente. Não existe "sincronizar manualmente" — a cada execução, tudo é recalculado do zero a partir do que estiver nas duas abas naquele momento.

In [ ]:
#@title 💼 9.3 Ganho consolidado da carteira (via planilha)
# Funções e constantes iguais às do pacote carteira_analise (o mesmo código do painel),
# importadas do pacote em vez de copiadas aqui (Fase 1, 11/2026).
from carteira_analise.planilha import (
    _PADRAO_TICKER_B3, interpretar_quantidade_operacao, _TIPOS_OPERACAO, EVENTOS_SOCIETARIOS,
    aplicar_eventos_societarios, _fator_redondo, detectar_saltos_de_patamar,
    verificar_eventos_nao_registrados,
)


import re

_carteira_importada = None  # preenchida ao importar; lida pela Seção 11


def _normalizar_ticker(ticker_bruto):
    """Limpa espaços e, se o ticker parecer um código B3 (4 letras + 1-2
    dígitos) sem o sufixo '.SA', completa automaticamente — ex: 'POMO4' ->
    'POMO4.SA'. Devolve (ticker_normalizado, foi_corrigido: bool)."""
    t = str(ticker_bruto).strip().upper()
    if t.endswith('.SA') or '.' in t or '-' in t:
        return t, False
    if _PADRAO_TICKER_B3.match(t):
        return t + '.SA', True
    return t, False


def _normalizar_aba_resumo(df):
    """Colunas: ticker, quantidade, data_inicio, e preco_medio OU
    valor_investido (pelo menos um dos dois — se só valor_investido vier
    preenchido, o preço médio é calculado como valor_investido/quantidade)."""
    mapa = {}
    for col in df.columns:
        c = str(col).strip().lower()
        c = (c.replace('í', 'i').replace('é', 'e').replace('ó', 'o')
               .replace('ã', 'a').replace('ç', 'c').replace('_', ' '))
        if c in ('ticker', 'ativo', 'codigo', 'papel'):
            mapa[col] = 'ticker'
        elif c in ('quantidade', 'qtd', 'qtde', 'quantity'):
            mapa[col] = 'quantidade'
        elif c in ('preco medio', 'pm', 'preco medio r$', 'preco', 'preco entrada', 'preco compra'):
            mapa[col] = 'preco_medio'
        elif c in ('valor investido', 'valor total', 'valor aplicado', 'total investido'):
            mapa[col] = 'valor_investido'
        elif c in ('data inicio', 'data', 'data compra', 'data aquisicao', 'data entrada'):
            mapa[col] = 'data_inicio'
    df_novo = df.rename(columns=mapa)
    faltando = [c for c in ('ticker', 'quantidade', 'data_inicio') if c not in df_novo.columns]
    if faltando:
        raise ValueError(f"Colunas obrigatórias não encontradas na aba Resumo: {faltando}.")
    if 'preco_medio' not in df_novo.columns and 'valor_investido' not in df_novo.columns:
        raise ValueError("A aba Resumo precisa ter 'preco_medio' OU 'valor_investido' preenchido.")
    for opcional in ('preco_medio', 'valor_investido'):
        if opcional not in df_novo.columns:
            df_novo[opcional] = None
    return df_novo[['ticker', 'quantidade', 'preco_medio', 'valor_investido', 'data_inicio']]


def _normalizar_aba_operacoes(df):
    """Colunas: ticker, tipo (compra/venda), quantidade, preco, data."""
    mapa = {}
    for col in df.columns:
        c = str(col).strip().lower()
        c = (c.replace('í', 'i').replace('é', 'e').replace('ó', 'o')
               .replace('ã', 'a').replace('ç', 'c').replace('_', ' '))
        if c in ('ticker', 'ativo', 'codigo', 'papel'):
            mapa[col] = 'ticker'
        elif c in ('tipo', 'operacao', 'movimento', 'tipo de operacao'):
            mapa[col] = 'tipo'
        elif c in ('quantidade', 'qtd', 'qtde', 'quantity'):
            mapa[col] = 'quantidade'
        elif c in ('preco', 'preco unitario', 'valor', 'valor unitario'):
            mapa[col] = 'preco'
        elif c in ('data', 'data operacao', 'data da operacao'):
            mapa[col] = 'data'
    df_novo = df.rename(columns=mapa)
    faltando = [c for c in ('ticker', 'tipo', 'quantidade', 'preco', 'data') if c not in df_novo.columns]
    if faltando:
        raise ValueError(f"Colunas obrigatórias não encontradas na aba Operações: {faltando}.")
    return df_novo[['ticker', 'tipo', 'quantidade', 'preco', 'data']]


def _construir_operacao_da_posicao_inicial(row):
    """Converte uma linha da aba Resumo (posição inicial/ponto de partida)
    numa operação de compra sintética, que entra como a PRIMEIRA operação
    do ativo, antes de qualquer movimentação da aba Operações."""
    quantidade = float(row['quantidade'])
    preco_medio = row.get('preco_medio')
    if preco_medio is None or (isinstance(preco_medio, float) and pd.isna(preco_medio)):
        valor_investido = row.get('valor_investido')
        if valor_investido is None or (isinstance(valor_investido, float) and pd.isna(valor_investido)):
            raise ValueError("informe preco_medio ou valor_investido na aba Resumo")
        preco_medio = float(valor_investido) / quantidade
    return {'data': pd.Timestamp(row['data_inicio']), 'tipo': 'compra',
            'quantidade': quantidade, 'preco': float(preco_medio)}


def normalizar_tipo_operacao(valor) -> str | None:
    """'Compra', 'Bonificação', 'Desdobro'... -> tipo interno, ou None."""
    import unicodedata
    texto = "".join(ch for ch in unicodedata.normalize("NFD", str(valor).strip().lower())
                    if unicodedata.category(ch) != "Mn").replace("_", " ")
    return _TIPOS_OPERACAO.get(texto)


def operacoes_do_ticker(row_resumo, grupo_operacoes: pd.DataFrame | None) -> tuple[list[dict], list[str], int]:
    """Operações válidas de um ativo (posição inicial da aba Resumo + aba
    Operações), já com os eventos societários aplicados. Devolve
    (operações, avisos sobre linhas ignoradas, nº de linhas aproveitadas)."""
    ops: list[dict] = []
    avisos: list[str] = []
    if row_resumo is not None:
        try:
            op = _construir_operacao_da_posicao_inicial(row_resumo)
            ops.append({**op, "data": pd.Timestamp(op["data"])})
        except (ValueError, TypeError) as e:
            avisos.append(f"posição inicial (aba Resumo) inválida — {e}")
    n = 0
    if grupo_operacoes is not None:
        for _, row in grupo_operacoes.iterrows():
            tipo = normalizar_tipo_operacao(row["tipo"])
            if tipo is None:
                avisos.append(f"tipo inválido {row['tipo']!r} numa operação (use compra, venda, "
                              f"bonificação, desdobramento ou grupamento)")
                continue
            try:
                data_op = pd.Timestamp(row["data"])
                if pd.isna(data_op):
                    raise ValueError
                preco_bruto = row["preco"]
                if tipo in ("desdobramento", "grupamento"):
                    preco_op = 0.0  # não há preço: só muda o número de ações
                elif tipo == "bonificacao" and (preco_bruto is None or pd.isna(preco_bruto)):
                    preco_op = 0.0  # custo atribuído não informado
                else:
                    preco_op = float(preco_bruto)
            except (ValueError, TypeError):
                avisos.append(f"operação com data/preço ilegível (data={row['data']!r}) — linha ignorada")
                continue
            quantidade, motivo = interpretar_quantidade_operacao(
                "venda" if tipo == "grupamento" else tipo, row["quantidade"])
            if quantidade is None:
                avisos.append(f"{tipo} de {data_op:%d/%m/%Y} com {motivo} — linha ignorada")
                continue
            if preco_op < 0 or (preco_op == 0 and tipo == "venda") or preco_op != preco_op:
                avisos.append(f"{tipo} de {data_op:%d/%m/%Y} com preço inválido ({preco_bruto!r}) — linha ignorada")
                continue
            ops.append({"data": data_op, "tipo": tipo, "quantidade": quantidade, "preco": preco_op})
            n += 1
    return ops, avisos, n


def _eventos_e_precos_brutos(ticker, periodo='5y'):
    """Desdobramentos/grupamentos do Yahoo e histórico de fechamento (com cache),
    para conferir os eventos lançados na planilha."""
    chave = f"eventos_precos:{ticker}:{periodo}"
    valor = _cache_get(chave)
    if valor is None:
        try:
            eventos = yf.Ticker(ticker).splits
        except Exception:
            eventos = pd.Series(dtype=float)
        try:
            hist = yf.download(ticker, period=periodo, progress=False, auto_adjust=False)
            precos = hist['Close'].squeeze(axis=1) if isinstance(hist['Close'], pd.DataFrame) else hist['Close']
        except Exception:
            precos = pd.Series(dtype=float)
        valor = (eventos if eventos is not None else pd.Series(dtype=float), precos.dropna())
        _cache_set(chave, valor)
    return valor


def processar_carteira_combinada(df_resumo, df_operacoes):
    """Uma linha final por ativo, combinando a posição inicial (aba Resumo,
    quando existir) com as movimentações (aba Operações, quando existirem).
    O Resumo funciona como ponto de partida; cada operação lançada depois
    afeta a quantidade, o preço médio e o ganho — igual um extrato real."""
    tickers_resumo = {}
    if df_resumo is not None:
        for _, row in df_resumo.iterrows():
            ticker, foi_corrigido = _normalizar_ticker(row['ticker'])
            if foi_corrigido:
                print(f"  📌 [Resumo] Ticker '{str(row['ticker']).strip()}' interpretado como '{ticker}'.")
            tickers_resumo[ticker] = row

    tickers_operacoes = {}
    if df_operacoes is not None and not df_operacoes.empty:
        tickers_normalizados = df_operacoes['ticker'].apply(lambda t: _normalizar_ticker(t)[0])
        for ticker, grupo in df_operacoes.groupby(tickers_normalizados):
            tickers_operacoes[ticker] = grupo

    todos_tickers = sorted(set(tickers_resumo) | set(tickers_operacoes))
    linhas = []

    for ticker in todos_tickers:
        moeda = 'R$' if ticker.endswith('.SA') else 'US$'
        operacoes, problemas, n_operacoes_lancadas = operacoes_do_ticker(
            tickers_resumo.get(ticker), tickers_operacoes.get(ticker))
        ops_brutas = list(operacoes)
        if ops_brutas:
            eventos_yf, precos_brutos = _eventos_e_precos_brutos(ticker)
            for aviso_evento in verificar_eventos_nao_registrados(ops_brutas, eventos_yf, precos_brutos):
                problemas.insert(0, 'possível evento não lançado — ' + aviso_evento)
        try:
            operacoes = aplicar_eventos_societarios(operacoes)
        except ValueError as e:
            problemas.append(str(e))
            operacoes = []

        if problemas:
            for p in problemas:
                print(f"  ⚠️ {ticker}: {p}")

        if not operacoes:
            linhas.append({'Ticker': ticker, 'Situação': 'erro',
                            'Ganho total': '⚠️ nenhuma posição/operação válida (ver avisos acima)'})
            continue

        tec = indicadores_tecnicos(ticker, periodo_historico.value)
        preco_atual = tec['preco_atual'] if tec is not None else None
        divs = _dividendos_brutos_cache(ticker)

        try:
            r = avaliar_operacoes(operacoes, preco_atual=preco_atual, dividendos=divs)
        except ValueError as e:
            linhas.append({'Ticker': ticker, 'Situação': 'erro', 'Ganho total': f'⚠️ {e}'})
            continue

        situacao = 'Aberta' if r['quantidade_aberta'] > 0 else 'Encerrada'
        aviso = ' ⚠️' if problemas else ''
        origem = []
        if ticker in tickers_resumo:
            origem.append('posição inicial')
        if n_operacoes_lancadas:
            origem.append(f"{n_operacoes_lancadas} operação(ões)")
        ganho_nao_real = r['ganho_nao_realizado']

        linhas.append({
            'Ticker': ticker, 'Situação': situacao + aviso, 'Origem': ' + '.join(origem),
            'Quantidade aberta': f"{r['quantidade_aberta']:.0f}",
            'Preço médio atual': f"{moeda} {r['preco_medio_aberto']:.2f}" if r['preco_medio_aberto'] else 'N/A',
            'Ganho realizado': f"{moeda} {r['ganho_realizado']:+.2f}",
            'Ganho não realizado': f"{moeda} {ganho_nao_real:+.2f}" if ganho_nao_real is not None else 'N/A',
            'Renda recebida': f"{moeda} {r['renda_recebida']:.2f}",
            'Ganho total': f"{moeda} {r['ganho_total']:+.2f}",
            '_moeda': moeda, '_ganho_total_num': r['ganho_total'],
            '_ganho_realizado_num': r['ganho_realizado'],
        })

    return pd.DataFrame(linhas)


def _achar_aba(abas_dict, nomes_possiveis):
    """Procura uma aba pelo nome, tolerando maiúsculas/minúsculas e acento."""
    def normalizar_nome(n):
        n = n.strip().lower()
        n = n.replace('í', 'i').replace('é', 'e').replace('ç', 'c').replace('õ', 'o').replace('ã', 'a')
        n = n.replace('ª', 'a').replace('º', 'o').replace('.', '').replace('_', ' ')
        return ' '.join(n.split())
    alvo = {normalizar_nome(n) for n in nomes_possiveis}
    for nome_real, df in abas_dict.items():
        if normalizar_nome(nome_real) in alvo:
            return df
    return None


# --- Painel: importar carteira (abas Resumo + Operações) e calcular ---
origem_planilha = widgets.Dropdown(
    options=[('Upload de arquivo (Excel/CSV)', 'upload'), ('Google Sheets (link)', 'sheets')],
    value='upload', description='Fonte:', style={'description_width': 'initial'},
)
link_sheets = widgets.Text(
    value='', description='Link da planilha:',
    placeholder='https://docs.google.com/spreadsheets/d/...',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'),
)
botao_importar = widgets.Button(description="📂 Importar e calcular", button_style='success', icon='upload')
saida_carteira_planilha = widgets.Output()

def _ler_todas_as_abas(origem):
    if origem == 'upload':
        from google.colab import files
        print("Selecione o arquivo (Excel .xlsx ou CSV) na janela que vai abrir...")
        enviados = files.upload()
        if not enviados:
            return None
        nome_arquivo = list(enviados.keys())[0]
        if nome_arquivo.lower().endswith('.csv'):
            return {'Operações': pd.read_csv(nome_arquivo)}  # CSV = 1 tabela só, tratada como Operações
        return pd.read_excel(nome_arquivo, sheet_name=None)
    else:
        if not link_sheets.value.strip():
            print("⚠️ Cole o link da planilha do Google Sheets no campo acima.")
            return None
        from google.colab import auth
        import gspread
        from google.auth import default
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
        planilha = gc.open_by_url(link_sheets.value.strip())
        return {aba.title: pd.DataFrame(aba.get_all_records()) for aba in planilha.worksheets()}


def _importar_carteira_click(b):
    with saida_carteira_planilha:
        clear_output()
        try:
            abas = _ler_todas_as_abas(origem_planilha.value)
            if abas is None:
                return
            if not abas:
                print("⚠️ Nenhuma aba encontrada na planilha.")
                return

            print(f"Abas encontradas: {', '.join(abas.keys())}\n")

            df_resumo_bruto = _achar_aba(abas, ['Resumo', 'Resumo da carteira', 'Posicao', 'Posição', 'Posição inicial', '1ª compra', '1a compra', '1º compra', '1o compra', 'Primeira compra'])
            df_operacoes_bruto = _achar_aba(abas, ['Operações', 'Operacoes', 'Movimentações', 'Movimentacoes', 'Compras e vendas posteriores'])

            df_resumo = _normalizar_aba_resumo(df_resumo_bruto) if df_resumo_bruto is not None else None
            df_operacoes = _normalizar_aba_operacoes(df_operacoes_bruto) if df_operacoes_bruto is not None else None

            global _carteira_importada
            _carteira_importada = (df_resumo, df_operacoes)  # usada pela Seção 11 (posições sugeridas)

            if df_resumo is None and df_operacoes is None:
                print("⚠️ Não encontrei nenhuma aba chamada 'Resumo' ou 'Operações' (nem variações). "
                      "Renomeie as abas do arquivo e tente de novo.")
                return

            print(f"Aba Resumo: {'não encontrada' if df_resumo is None else f'{len(df_resumo)} ativo(s)'}")
            print(f"Aba Operações: {'não encontrada' if df_operacoes is None else f'{len(df_operacoes)} operação(ões)'}\n")

            resultado = processar_carteira_combinada(df_resumo, df_operacoes)
            colunas_exibir = [c for c in resultado.columns if not c.startswith('_')]

            situacao = resultado['Situação'].astype(str) if 'Situação' in resultado.columns else pd.Series(dtype=str)
            abertas = resultado[situacao.str.startswith('Aberta')]
            encerradas = resultado[situacao.str.startswith('Encerrada')]
            com_erro = resultado[situacao == 'erro']

            print(f"\n📌 CARTEIRA ATUAL — {len(abertas)} posição(ões) em aberto "
                  f"(inclui ativos novos, exclui os totalmente vendidos):")
            if not abertas.empty:
                display(abertas[colunas_exibir])
            else:
                print("   (nenhuma posição em aberto)")

            if not encerradas.empty:
                print(f"\n📁 POSIÇÕES ENCERRADAS NO PERÍODO — {len(encerradas)} ativo(s) totalmente "
                      f"vendido(s) (fora da carteira atual, histórico do ganho já realizado):")
                display(encerradas[colunas_exibir])

            if not com_erro.empty:
                print(f"\n⚠️ {len(com_erro)} ativo(s) com erro — ver avisos acima e a tabela abaixo:")
                display(com_erro[colunas_exibir])

            if '_ganho_total_num' in resultado.columns:
                print("\n📊 TOTAIS POR MOEDA:")
                for rotulo, subset in [
                    ("Carteira atual — ganho total (inclui realizado de vendas parciais + não realizado + renda)", abertas),
                    ("Posições encerradas — ganho total (100% realizado)", encerradas),
                ]:
                    df_valido = subset[subset['_ganho_total_num'].notna()]
                    if df_valido.empty:
                        continue
                    for moeda, grupo in df_valido.groupby('_moeda'):
                        total_ganho = grupo['_ganho_total_num'].sum()
                        print(f"   {rotulo}\n      {moeda}: {moeda} {total_ganho:+,.2f}")

                if '_ganho_realizado_num' in resultado.columns:
                    df_valido_real = resultado[resultado['_ganho_realizado_num'].notna()]
                    if not df_valido_real.empty:
                        print("\n💰 GANHO JÁ REALIZADO NO TOTAL (soma vendas parciais + posições encerradas, "
                              "por moeda):")
                        for moeda, grupo in df_valido_real.groupby('_moeda'):
                            total_realizado = grupo['_ganho_realizado_num'].sum()
                            print(f"   {moeda}: {moeda} {total_realizado:+,.2f}")

                print("\n📌 Os totais são calculados separadamente por moeda (R$ e US$), sem conversão")
                print("   cambial entre elas.")
        except ImportError:
            print("⚠️ Este recurso depende de bibliotecas específicas do Google Colab "
                  "(google.colab, gspread) — só funciona quando executado no Colab, não localmente.")
        except Exception as e:
            print(f"⚠️ Erro ao processar a planilha: {e}")

botao_importar.on_click(_importar_carteira_click)
display(widgets.HTML("<h3>9.3 Ganho consolidado da carteira (via planilha)</h3>"))
display(widgets.HTML(
    "<p>Importa a carteira de um arquivo Excel/CSV ou Google Sheets, combinando duas abas:</p>"
    "<ul>"
    "<li><b>Resumo</b> — a posição inicial de cada ativo (ponto de partida): "
    "<code>ticker</code>, <code>quantidade</code>, <code>preco_medio</code> (ou "
    "<code>valor_investido</code>), <code>data_inicio</code>.</li>"
    "<li><b>Operações</b> — as movimentações feitas a partir dali: <code>ticker</code>, "
    "<code>tipo</code> (compra/venda), <code>quantidade</code>, <code>preco</code>, <code>data</code>.</li>"
    "</ul>"
    "<p>Um ativo pode ter só Resumo, só Operações, ou as duas — as operações são sempre "
    "aplicadas EM CIMA da posição inicial, ajustando quantidade, preço médio e ganho, "
    "igual um extrato.</p>"
))
display(widgets.VBox([origem_planilha, link_sheets, botao_importar, saida_carteira_planilha]))


## 📐 10. Backtesting do sinal de timing (técnico)

Reavalia retroativamente o componente **técnico** do motor de pontuação (tendência, RSI, distância do período) sobre o histórico de cada ativo da carteira, comparando o sinal que teria sido dado no passado com o retorno realmente observado depois — F1-score (macro, 3 classes: compra/manter/venda).

Todos os parâmetros (horizonte de avaliação, limiar de retorno, e os limiares de pontuação que definem compra/venda em dois cenários) são **selecionáveis nos controles abaixo**, sem precisar editar código — os valores padrão refletem os parâmetros definidos na Seção 3.1 do Relatório do Projeto.

Desde 11/2026 o backtest usa a **mesma regra técnica do motor** (pacote `carteira_analise`): MM200, RSI e piso/topo usuais do período, que não pontuam quando estão a menos de 10% um do outro (faixa estreita).

**Limitação conhecida**: nem o Yahoo Finance nem o Fundamentus (Seção 5) fornecem uma série histórica gratuita de fundamentos (P/VP, vacância, dividend yield de datas passadas) — os dois só dão o valor de **hoje**. Por isso este backtest avalia só o componente técnico, não o motor completo usado no dia a dia da ferramenta. É uma validação parcial, não do sistema inteiro. Para compensar isso, a Seção mostra, ao final, o retrato fundamentalista **atual** de cada FII (via Fundamentus) como contexto complementar — não como parte do cálculo retroativo — e uma interpretação em linguagem simples do resultado técnico.

In [ ]:
#@title 📐 10. Backtesting do sinal de timing (técnico)

def _indicadores_de_serie(fechamento_bruto):
    """Mesma lógica de cálculo de indicadores técnicos da Seção 4, mas
    recebendo a série de preços diretamente (em vez de baixar por ticker) —
    permite aplicar sobre um RECORTE truncado do histórico, "congelando"
    uma data de corte no passado, sem espiar dado futuro (look-ahead bias)."""
    fechamento = fechamento_bruto.dropna()
    if len(fechamento) < 30:
        return None
    fechamento, n_outliers, n_truncados = limpar_outliers_precos(fechamento)
    if len(fechamento) < 30:
        return None
    preco_atual = float(fechamento.iloc[-1])
    sma200 = fechamento.rolling(200).mean().iloc[-1] if len(fechamento) >= 200 else np.nan
    rsi14 = calcular_rsi(fechamento, 14).iloc[-1]
    maxima_periodo = fechamento.quantile(0.95)
    minima_periodo = fechamento.quantile(0.05)
    dist_maxima = (preco_atual - maxima_periodo) / maxima_periodo * 100
    dist_minima = (preco_atual - minima_periodo) / minima_periodo * 100
    return {
        'preco_atual': preco_atual,
        'sma200': float(sma200) if not pd.isna(sma200) else None,
        'rsi14': float(rsi14) if not pd.isna(rsi14) else None,
        'dist_maxima_pct': float(dist_maxima),
        'dist_minima_pct': float(dist_minima),
    }


def _pontuar_tecnico_backtest(tec):
    """Componente técnico do motor de pontuação, isolado do fundamentalista —
    é o único componente reconstruível retroativamente, já que o Yahoo Finance
    não fornece fundamentos históricos (P/L, P/VP de datas passadas). Usa a
    MESMA regra do motor (pacote: MM200, RSI e piso/topo usuais, que não
    pontuam quando a faixa do período é estreita — revisão de 06/11/2026)."""
    return pontos_tecnicos(tec)


def _pontuar_tecnico_regra_anterior(tec):
    """Regra técnica ANTERIOR a 06/11/2026 (piso e topo pontuam mesmo com a
    faixa estreita) — só para medir, com os mesmos dados, o efeito da mudança."""
    original = motor.AMPLITUDE_MINIMA_FAIXA
    try:
        motor.AMPLITUDE_MINIMA_FAIXA = 0.0
        return pontos_tecnicos(tec)
    finally:
        motor.AMPLITUDE_MINIMA_FAIXA = original


def _classe_a_partir_de_pontos(pontos, limiar_compra=3, limiar_venda=-2):
    if pontos >= limiar_compra:
        return 'compra'
    if pontos <= limiar_venda:
        return 'venda'
    return 'manter'


def _rotular_retorno_futuro(fechamento, data_corte, horizonte_dias, limiar):
    pos_corte = fechamento.index.searchsorted(data_corte)
    if pos_corte >= len(fechamento):
        return None
    preco_corte = float(fechamento.iloc[pos_corte])
    data_alvo = data_corte + pd.Timedelta(days=horizonte_dias)
    pos_futuro = fechamento.index.searchsorted(data_alvo)
    if pos_futuro >= len(fechamento):
        return None
    preco_futuro = float(fechamento.iloc[pos_futuro])
    retorno = (preco_futuro - preco_corte) / preco_corte
    if retorno > limiar:
        return 'compra'
    if retorno < -limiar:
        return 'venda'
    return 'manter'


def _yield_ultimos_12m_backtest(dividendos_ate_data, preco, data_ref=None):
    """Yield 'trailing' dos 12 meses que terminam na data de corte — previsão
    'ingênua' (persistência) do yield dos 12 meses seguintes. Correção de
    04/11/2026: a janela era contada a partir do ÚLTIMO pagamento, e não da
    data de corte (uma empresa que parou de pagar ainda aparecia com yield).
    Sem pagamento nos 12 meses até a data, o yield é 0."""
    if dividendos_ate_data is None or dividendos_ate_data.empty or not preco:
        return None
    fim = pd.Timestamp(data_ref) if data_ref is not None else dividendos_ate_data.index.max()
    ult_12m = dividendos_ate_data[(dividendos_ate_data.index > fim - pd.Timedelta(days=365))
                                  & (dividendos_ate_data.index <= fim)]
    return float(ult_12m.sum()) / preco


def _baixar_fechamento_cache(ticker, periodo='5y'):
    """Busca o histórico de preço de um ticker com cache (mesmo padrão já
    usado pra dividendos nesta seção) — evita rebuscar o mesmo ticker
    quando o backtest é rodado várias vezes com horizontes diferentes
    (sensibilidade ao horizonte, mais abaixo nesta seção)."""
    chave = f"precos_backtest_sem_ajuste:{ticker}:{periodo}"
    fechamento = _cache_get(chave)
    if fechamento is not None:
        return fechamento
    # auto_adjust=False: preço SEM ajuste por proventos (o padrão do yfinance
    # passou a ser True, o que contaria os proventos duas vezes no backtest).
    hist = yf.download(ticker, period=periodo, progress=False, auto_adjust=False)
    if hist.empty:
        fechamento = pd.Series(dtype=float)
    else:
        fechamento = hist['Close']
        if isinstance(fechamento, pd.DataFrame):
            fechamento = fechamento.squeeze(axis=1)
        fechamento = fechamento.dropna()
    _cache_set(chave, fechamento)
    return fechamento


def executar_backtest_ticker(ticker, periodo='5y', passo_dias=30, horizonte_dias=90, limiar_retorno=0.05,
                              limiar_compra=3, limiar_venda=-2, dividendos=None, fechamento_completo=None):
    """Roda o backtest retroativo de um ticker: percorre datas de corte a
    cada `passo_dias`, calcula o sinal técnico só com dado disponível até
    ali, e compara com o retorno realmente observado `horizonte_dias`
    depois. Parâmetros padrão: horizonte de 90 dias e limiar de ±5%,
    conforme definido na Seção 3.1 (Objetivo SMART) do Relatório do Projeto.
    Também registra, em cada ponto de corte, o yield trailing (12m até ali)
    como previsão 'ingênua' e o yield efetivo pago nos 12 meses seguintes —
    usados depois pra calcular MAE/R² da previsão de renda.

    `fechamento_completo` pode ser passado pronto (já buscado antes) pra
    evitar rebuscar o mesmo ticker ao testar múltiplos horizontes."""
    if fechamento_completo is None:
        fechamento_completo = _baixar_fechamento_cache(ticker, periodo)
    if fechamento_completo.empty:
        return []

    if dividendos is None:
        dividendos = pd.Series(dtype=float, index=pd.DatetimeIndex([]))
    elif dividendos.index.tz is not None:
        dividendos = dividendos.copy()
        dividendos.index = dividendos.index.tz_localize(None)

    registros = []
    for idx in range(200, len(fechamento_completo) - 1, passo_dias):
        data_corte = fechamento_completo.index[idx]
        fechamento_ate_corte = fechamento_completo.iloc[:idx + 1]
        tec = _indicadores_de_serie(fechamento_ate_corte)
        if tec is None:
            continue
        classe_real = _rotular_retorno_futuro(fechamento_completo, data_corte, horizonte_dias, limiar_retorno)
        if classe_real is None:
            continue
        pontos_tecnicos = _pontuar_tecnico_backtest(tec)
        pontos_regra_anterior = _pontuar_tecnico_regra_anterior(tec)
        classe_prevista = _classe_a_partir_de_pontos(pontos_tecnicos, limiar_compra, limiar_venda)

        dividendos_ate_corte = dividendos[dividendos.index <= data_corte]
        dy_trailing = _yield_ultimos_12m_backtest(dividendos_ate_corte, tec['preco_atual'], data_corte)
        data_futuro_12m = data_corte + pd.Timedelta(days=365)
        dividendos_futuros = dividendos[(dividendos.index > data_corte) & (dividendos.index <= data_futuro_12m)]
        dy_futuro = float(dividendos_futuros.sum()) / tec['preco_atual'] if tec['preco_atual'] else None

        registros.append({'ticker': ticker, 'data_corte': data_corte, 'pontos_tecnicos': pontos_tecnicos,
                          'pontos_regra_anterior': pontos_regra_anterior,
                           'classe_prevista': classe_prevista, 'classe_real': classe_real,
                           'dy_trailing_previsto': dy_trailing, 'dy_futuro_realizado': dy_futuro})
    return registros


def reclassificar(registros, limiar_compra=3, limiar_venda=-2):
    """Reclassifica um conjunto de registros já calculados, com limiares
    diferentes, sem precisar baixar dado nem recalcular indicadores de novo
    — só reaplica a régua de classificação sobre a pontuação já guardada."""
    novos = []
    for r in registros:
        nova_classe = _classe_a_partir_de_pontos(r['pontos_tecnicos'], limiar_compra, limiar_venda)
        novos.append({**r, 'classe_prevista': nova_classe})
    return novos


def calcular_f1_macro(registros):
    classes = ('compra', 'manter', 'venda')
    por_classe = {}
    for c in classes:
        tp = sum(1 for r in registros if r['classe_prevista'] == c and r['classe_real'] == c)
        fp = sum(1 for r in registros if r['classe_prevista'] == c and r['classe_real'] != c)
        fn = sum(1 for r in registros if r['classe_prevista'] != c and r['classe_real'] == c)
        precisao = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precisao * recall / (precisao + recall) if (precisao + recall) > 0 else 0.0
        por_classe[c] = {'precisao': precisao, 'recall': recall, 'f1': f1, 'n_real': tp + fn}
    f1_macro = sum(m['f1'] for m in por_classe.values()) / len(classes) if registros else 0.0
    return f1_macro, por_classe


def calcular_mae_r2_renda(registros):
    """MAE e R² comparando o yield trailing (previsão 'ingênua' de
    persistência) contra o yield efetivo realmente pago nos 12 meses
    seguintes à data de corte — mesma lógica de backtesting.py (pacote)."""
    pares = [(r['dy_trailing_previsto'], r['dy_futuro_realizado']) for r in registros
             if r.get('dy_trailing_previsto') is not None and r.get('dy_futuro_realizado') is not None]
    if not pares:
        return {'mae': None, 'r2': None, 'n': 0}
    previstos = np.array([p[0] for p in pares])
    reais = np.array([p[1] for p in pares])
    mae = float(np.mean(np.abs(previstos - reais)))
    media_real = reais.mean()
    ss_tot = float(np.sum((reais - media_real) ** 2))
    ss_res = float(np.sum((reais - previstos) ** 2))
    r2 = (1 - ss_res / ss_tot) if ss_tot > 0 else None
    return {'mae': mae, 'r2': r2, 'n': len(pares)}


# --- Painel: parâmetros selecionáveis + rodar o backtest ---
horizonte_backtest = widgets.IntSlider(value=90, min=30, max=180, step=15,
    description='Horizonte (dias):', style={'description_width': 'initial'})
limiar_retorno_backtest = widgets.FloatSlider(value=0.05, min=0.02, max=0.15, step=0.01,
    description='Limiar de retorno (±):', readout_format='.0%', style={'description_width': 'initial'})

limiar_compra_a = widgets.IntSlider(value=3, min=-3, max=3, step=1,
    description='Cenário A — limiar de compra:', style={'description_width': 'initial'})
limiar_venda_a = widgets.IntSlider(value=-2, min=-3, max=3, step=1,
    description='Cenário A — limiar de venda:', style={'description_width': 'initial'})
limiar_compra_b = widgets.IntSlider(value=2, min=-3, max=3, step=1,
    description='Cenário B — limiar de compra:', style={'description_width': 'initial'})
limiar_venda_b = widgets.IntSlider(value=-2, min=-3, max=3, step=1,
    description='Cenário B — limiar de venda:', style={'description_width': 'initial'})

botao_backtest = widgets.Button(description="📐 Rodar backtesting", button_style='warning', icon='line-chart')
saida_backtest = widgets.Output()

def _imprimir_resultado_cenario(nome_cenario, registros):
    f1_macro, por_classe = calcular_f1_macro(registros)
    print(f"\n📊 {nome_cenario}: F1-score macro = {f1_macro:.3f}")
    for c, m in por_classe.items():
        print(f"   {c}: precisão={m['precisao']:.2f}  recall={m['recall']:.2f}  f1={m['f1']:.2f}  (n={m['n_real']})")
    return f1_macro



def _interpretar_em_linguagem_simples(registros, f1_macro, por_classe):
    """Traduz o resultado técnico do backtest para linguagem simples, sem
    jargão estatístico — pensado para o perfil de conhecimento intermediário
    (Persona 2, Seção 2.2 do Relatório), que pode não ter familiaridade com
    F1-score/precisão/recall."""
    from collections import Counter
    n_total = len(registros)
    if n_total == 0:
        return
    contagem_real = Counter(r['classe_real'] for r in registros)
    classe_maj, n_maj = contagem_real.most_common(1)[0]
    p_maj = n_maj / n_total
    r_maj = 1.0
    f1_maj_classe = 2 * p_maj * r_maj / (p_maj + r_maj) if (p_maj + r_maj) > 0 else 0
    f1_majoritaria = f1_maj_classe / 3

    recall_compra = por_classe.get('compra', {}).get('recall', 0)
    recall_venda = por_classe.get('venda', {}).get('recall', 0)

    print("\n" + "=" * 66)
    print("💬 O QUE ISSO SIGNIFICA, EM PALAVRAS SIMPLES:")
    print("=" * 66)
    print(f"\nDas vezes em que REALMENTE teria valido a pena COMPRAR (olhando pra trás,")
    print(f"sabendo o que aconteceu depois), o sinal técnico identificou corretamente")
    print(f"apenas {recall_compra*100:.0f}% delas.")
    print(f"\nDas vezes em que REALMENTE teria valido a pena VENDER, identificou")
    print(f"corretamente apenas {recall_venda*100:.0f}% delas.")
    print(f"\nComparando com um critério \"preguiçoso\" que nunca sugere nada (sempre")
    print(f"\"manter\"), o sinal técnico pontuou {f1_macro:.2f} contra {f1_majoritaria:.2f}")
    print(f"do critério preguiçoso — diferença de {(f1_macro - f1_majoritaria):+.2f}.")
    if abs(f1_macro - f1_majoritaria) < 0.05:
        print(f"\n👉 Isso indica que, sozinho, o sinal técnico teve desempenho muito parecido")
        print(f"   com simplesmente não fazer nada. Não deve ser usado como única base pra")
        print(f"   decidir comprar ou vender — funciona melhor como mais um dado a considerar,")
        print(f"   junto com os fundamentos (Seção 5) e o contexto do ativo.")
    elif f1_macro > f1_majoritaria:
        print(f"\n👉 O sinal técnico teve desempenho melhor que simplesmente não fazer nada,")
        print(f"   mas ainda está longe de ser perfeito — use como um dado a mais na sua")
        print(f"   análise, não como resposta pronta.")
    else:
        print(f"\n👉 Neste teste, o sinal técnico teve desempenho PIOR que simplesmente não")
        print(f"   fazer nada — um sinal de que, pelo menos neste histórico e com estes")
        print(f"   parâmetros, ele não ajudou a prever o que aconteceria depois.")


def _exibir_contexto_fundamentus_atual(tickers):
    """Mostra o retrato ATUAL (não histórico) dos fundamentos de FIIs via
    Fundamentus, como contexto complementar ao backtest técnico acima — não
    faz parte do cálculo retroativo, porque o Fundamentus só fornece o valor
    de HOJE desses indicadores, não uma série histórica gratuita (ver nota
    desta seção)."""
    linhas_fii = []
    for ticker in tickers:
        f = buscar_fundamentos_fii_fundamentus(ticker)
        if f is not None:
            linhas_fii.append((ticker, f))

    if not linhas_fii:
        return

    print("\n" + "=" * 66)
    print("🏢 CONTEXTO FUNDAMENTALISTA ATUAL DOS FIIs (via Fundamentus, hoje)")
    print("=" * 66)
    print("Não faz parte do backtest acima (que é só técnico e retroativo) — é o")
    print("retrato mais recente disponível, para complementar a leitura histórica")
    print("com a situação atual de cada fundo.\n")
    for ticker, f in linhas_fii:
        pvp = f.get('pvp')
        dy = f.get('dividend_yield_fundamentus')
        vac = f.get('vacancia_media')
        segm = f.get('segmento') or 'N/D'
        pvp_txt = f"P/VP={pvp:.2f}" if pvp is not None else "P/VP=N/D"
        dy_txt = f"Dividend Yield={dy*100:.1f}%" if dy is not None else "Dividend Yield=N/D"
        # FII de papel (sem imóveis físicos) não tem vacância no sentido usual.
        # Três sinais combinados (nenhum sozinho é sempre confiável — o
        # RECR11.SA já voltou a mostrar qtd_imoveis preenchido incorretamente
        # depois da primeira correção, Seção 4.3 do Relatório): segmento
        # classificado como papel/recebíveis/títulos, qtd_imoveis ausente/
        # zero, ou vacância extrema (100% exato — implausível num FII de
        # tijolo real).
        segmento_lower = segm.lower()
        segmento_indica_papel = any(
            p in segmento_lower for p in ('papel', 'recebív', 'recebiv', 'título', 'titulo', 'renda fixa'))
        qtd_imoveis_indica_papel = f.get('qtd_imoveis') in (None, 0)
        vacancia_extrema_suspeita = vac is not None and vac >= 0.999
        eh_fii_de_papel = qtd_imoveis_indica_papel or segmento_indica_papel or vacancia_extrema_suspeita
        vac_txt = "Vacância=N/A (FII de papel)" if eh_fii_de_papel else (
            f"Vacância={vac*100:.1f}%" if vac is not None else "Vacância=N/D")
        print(f"  {ticker} ({segm}): {pvp_txt} | {dy_txt} | {vac_txt}")


def _rodar_backtest_click(b):
    with saida_backtest:
        clear_output()
        # Mapa ticker -> tipo de ativo, usado na quebra de F1 por categoria mais abaixo.
        tipo_por_ticker = {}
        for t in parse_tickers(caixa_acoes.value):
            tipo_por_ticker[t] = 'Ação B3'
        for t in parse_tickers(caixa_fiis.value):
            tipo_por_ticker[t] = 'FII'
        for t in parse_tickers(caixa_acoes_us.value):
            tipo_por_ticker[t] = 'Ação EUA'
        for t in parse_tickers(caixa_etf_br.value):
            tipo_por_ticker[t] = 'ETF B3'
        for t in parse_tickers(caixa_etf_us.value):
            tipo_por_ticker[t] = 'ETF EUA'
        todos_tickers = list(tipo_por_ticker.keys())
        if not todos_tickers:
            print("⚠️ Nenhum ticker configurado na Seção 2.")
            return

        horizonte = horizonte_backtest.value
        limiar_ret = limiar_retorno_backtest.value
        lc_a, lv_a = limiar_compra_a.value, limiar_venda_a.value
        lc_b, lv_b = limiar_compra_b.value, limiar_venda_b.value

        print(f"Rodando backtest (horizonte {horizonte} dias, limiar de retorno ±{limiar_ret*100:.0f}%) "
              f"para {len(todos_tickers)} ativo(s)... isso pode demorar um pouco.\n")
        todos_registros = []
        for ticker in todos_tickers:
            # Preço e dividendos com cache (mesmo padrão já usado no resto do notebook) —
            # o cache de preço evita rebuscar o mesmo ticker na sensibilidade ao horizonte,
            # mais abaixo, que roda o backtest de novo com horizontes diferentes.
            fechamento_ticker = _baixar_fechamento_cache(ticker)
            chave_cache_div = f"dividendos:{ticker}"
            divs_ticker = _cache_get(chave_cache_div)
            if divs_ticker is None:
                try:
                    divs_ticker = yf.Ticker(ticker).dividends
                except Exception:
                    divs_ticker = pd.Series(dtype=float, index=pd.DatetimeIndex([]))
                _cache_set(chave_cache_div, divs_ticker)

            # Roda uma vez com os limiares do Cenário A — os pontos_tecnicos ficam guardados
            # no registro, permitindo reclassificar pro Cenário B (e pro sweep, mais abaixo)
            # sem baixar o dado de novo.
            registros = executar_backtest_ticker(ticker, horizonte_dias=horizonte, limiar_retorno=limiar_ret,
                                                  limiar_compra=lc_a, limiar_venda=lv_a, dividendos=divs_ticker,
                                                  fechamento_completo=fechamento_ticker)
            for r in registros:
                r['tipo_ativo'] = tipo_por_ticker[ticker]
            print(f"  {ticker}: {len(registros)} ponto(s) de corte avaliado(s)")
            todos_registros.extend(registros)

        if not todos_registros:
            print("\n⚠️ Não foi possível gerar nenhum ponto de backtest (histórico curto demais pra algum ativo).")
            return

        f1_a, por_classe_a = calcular_f1_macro(todos_registros)
        _imprimir_resultado_cenario(f"Cenário A (compra ≥ {lc_a}, venda ≤ {lv_a})", todos_registros)

        registros_b = reclassificar(todos_registros, limiar_compra=lc_b, limiar_venda=lv_b)
        f1_b, por_classe_b = calcular_f1_macro(registros_b)
        _imprimir_resultado_cenario(f"Cenário B (compra ≥ {lc_b}, venda ≤ {lv_b})", registros_b)

        print(f"\n🔍 Comparação: F1 macro Cenário A = {f1_a:.3f}  |  F1 macro Cenário B = {f1_b:.3f}")
        print(f"   Diferença (B − A): {f1_b - f1_a:+.3f}")

        # Efeito da regra da faixa estreita (06/11/2026), com os MESMOS dados e limiares do Cenário A
        registros_antes = reclassificar([{**r, 'pontos_tecnicos': r['pontos_regra_anterior']}
                                         for r in todos_registros], limiar_compra=lc_a, limiar_venda=lv_a)
        f1_antes, _ = calcular_f1_macro(registros_antes)
        n_mudou = sum(1 for r in todos_registros if r['pontos_tecnicos'] != r['pontos_regra_anterior'])
        print(f"\n🧪 Regra da faixa estreita (piso e topo a menos de {AMPLITUDE_MINIMA_FAIXA:.0%} um do outro não pontuam):")
        print(f"   mudou a pontuação em {n_mudou} de {len(todos_registros)} pontos de corte; F1 macro do Cenário A "
              f"{f1_antes:.3f} com a regra anterior → {f1_a:.3f} com a atual ({f1_a - f1_antes:+.3f}).")

        # Interpretação em linguagem simples do cenário A (o padrão/principal)
        _interpretar_em_linguagem_simples(todos_registros, f1_a, por_classe_a)

        # --- Previsão de renda: MAE/R² (persistência do yield trailing vs. realizado) ---
        resultado_renda = calcular_mae_r2_renda(todos_registros)
        print("\n" + "=" * 66)
        print("💰 PREVISÃO DE RENDA (yield trailing como previsão 'ingênua')")
        print("=" * 66)
        if resultado_renda['n'] == 0:
            print("Não há pares suficientes (yield trailing + yield futuro realizado) pra")
            print("calcular MAE/R² — comum se os ativos da carteira pagarem pouco ou")
            print("irregularmente dividendo/rendimento no histórico disponível.")
        else:
            mae_txt = f"{resultado_renda['mae']*100:.2f} pontos percentuais" if resultado_renda['mae'] is not None else "N/D"
            r2_txt = f"{resultado_renda['r2']:.3f}" if resultado_renda['r2'] is not None else "N/D (variância real zero)"
            print(f"N = {resultado_renda['n']} pontos de corte com par válido (yield trailing, yield futuro)")
            print(f"MAE (erro médio absoluto): {mae_txt}")
            print(f"R² (quanto da variação do yield futuro a previsão ingênua explica): {r2_txt}")
            print(f"\n💬 Em palavras simples: usar o yield dos últimos 12 meses como \"aposta\" pro")
            print(f"   yield dos PRÓXIMOS 12 meses erra, em média, {mae_txt} — e um R² baixo (perto")
            print(f"   de 0 ou negativo) indica que essa aposta simples não captura bem as mudanças")
            print(f"   reais de renda; um R² mais alto (perto de 1) indica que o yield passado é um")
            print(f"   bom indício do yield futuro, para os ativos e período testados.")

        # --- Quebra do F1-score por tipo de ativo ---
        # O F1 macro geral é uma média de ativos bem diferentes entre si — pode esconder
        # o sinal funcionando bem numa categoria e mal noutra.
        print("\n" + "=" * 66)
        print("📂 F1-SCORE POR TIPO DE ATIVO (Cenário A)")
        print("=" * 66)
        for tipo in sorted(set(tipo_por_ticker.values())):
            registros_tipo = [r for r in todos_registros if r['tipo_ativo'] == tipo]
            if not registros_tipo:
                continue
            f1_tipo, _ = calcular_f1_macro(registros_tipo)
            print(f"  {tipo}: F1 macro = {f1_tipo:.3f}  (n={len(registros_tipo)})")

        # --- Sweep sistemático de limiares ---
        # Em vez de só os 2 cenários escolhidos à mão acima, testa uma grade de combinações
        # de limiar de compra/venda sobre os MESMOS registros já calculados (reclassificar()
        # não rebusca dado, só reaplica a régua de classificação) — mostra se existe uma
        # combinação melhor que a testada, ou se o teto é baixo mesmo com qualquer limiar.
        print("\n" + "=" * 66)
        print("🔬 SWEEP SISTEMÁTICO DE LIMIARES (compra: 1 a 4 · venda: -1 a -4)")
        print("=" * 66)
        resultados_sweep = []
        for lc_sweep in range(1, 5):
            for lv_sweep in range(-4, 0):
                registros_sweep = reclassificar(todos_registros, limiar_compra=lc_sweep, limiar_venda=lv_sweep)
                f1_sweep, _ = calcular_f1_macro(registros_sweep)
                resultados_sweep.append((lc_sweep, lv_sweep, f1_sweep))
        resultados_sweep.sort(key=lambda x: -x[2])
        print(f"Testadas {len(resultados_sweep)} combinações. Top 5 (compra, venda, F1 macro):")
        for lc_r, lv_r, f1_r in resultados_sweep[:5]:
            print(f"  compra≥{lc_r}, venda≤{lv_r}: F1 = {f1_r:.3f}")
        pior = resultados_sweep[-1]
        melhor = resultados_sweep[0]
        print(f"\nFaixa de F1 no sweep inteiro: {pior[2]:.3f} (pior: compra≥{pior[0]}, venda≤{pior[1]}) "
              f"a {melhor[2]:.3f} (melhor: compra≥{melhor[0]}, venda≤{melhor[1]})")
        if melhor[2] - f1_a < 0.03:
            print("👉 Nenhuma combinação testada supera de forma relevante o Cenário A original —")
            print("   o teto do sinal técnico parece ser mesmo baixo neste histórico, não uma questão")
            print("   de limiar mal escolhido.")
        else:
            print(f"👉 A melhor combinação do sweep (compra≥{melhor[0]}, venda≤{melhor[1]}) supera o Cenário A")
            print(f"   em {melhor[2] - f1_a:+.3f} — pode valer adotar esses limiares como novo padrão.")

        # --- Sensibilidade ao horizonte ---
        # Reaproveita preço e dividendo já em cache (buscados uma vez, acima) — só recalcula
        # os pontos de corte/rótulos pra cada horizonte, sem nova chamada de rede.
        print("\n" + "=" * 66)
        print("⏱️ SENSIBILIDADE AO HORIZONTE (mesmos limiares do Cenário A)")
        print("=" * 66)
        for h_teste in (30, 60, 90, 120, 180):
            registros_h = []
            for ticker in todos_tickers:
                fechamento_ticker = _baixar_fechamento_cache(ticker)
                divs_ticker = _cache_get(f"dividendos:{ticker}")
                if divs_ticker is None:
                    divs_ticker = pd.Series(dtype=float, index=pd.DatetimeIndex([]))
                regs_h = executar_backtest_ticker(ticker, horizonte_dias=h_teste, limiar_retorno=limiar_ret,
                                                   limiar_compra=lc_a, limiar_venda=lv_a, dividendos=divs_ticker,
                                                   fechamento_completo=fechamento_ticker)
                registros_h.extend(regs_h)
            if not registros_h:
                continue
            f1_h, _ = calcular_f1_macro(registros_h)
            marcador_atual = "  ← horizonte atual" if h_teste == horizonte else ""
            print(f"  {h_teste:>3} dias: F1 macro = {f1_h:.3f}  (n={len(registros_h)}){marcador_atual}")

        # Contexto fundamentalista atual (Fundamentus) — complementar, não retroativo
        _exibir_contexto_fundamentus_atual(todos_tickers)

        print("\n📌 Este backtest avalia SÓ o componente técnico do motor — nem o Yahoo Finance")
        print("   nem o Fundamentus fornecem fundamentos históricos gratuitos (ver nota acima) —")
        print("   é uma validação parcial, não do motor completo.")
        print("   Todos os parâmetros acima são ajustáveis nos controles desta seção, sem editar código.")

botao_backtest.on_click(_rodar_backtest_click)
display(widgets.HTML("<h3>10. Backtesting do sinal de timing</h3>"))
display(widgets.HTML("<p>Reavalia retroativamente o componente técnico do motor de pontuação sobre "
                      "o histórico de cada ativo da carteira (Seção 2), comparando o sinal que teria "
                      "sido dado no passado com o retorno realmente observado depois. Ajuste os "
                      "parâmetros abaixo e compare dois cenários de limiar lado a lado.</p>"))
display(widgets.VBox([
    horizonte_backtest, limiar_retorno_backtest,
    widgets.HTML("<b>Cenário A</b>"), limiar_compra_a, limiar_venda_a,
    widgets.HTML("<b>Cenário B</b>"), limiar_compra_b, limiar_venda_b,
    botao_backtest, saida_backtest,
]))


## 🧭 11. Cenários por ativo

Para cada ativo da carteira importada na **Seção 9.3**, mostra a **situação** em relação às faixas de preço sobre o **seu preço médio** e compara os caminhos possíveis, lado a lado — sem recomendação: a decisão é sempre do usuário. São os mesmos cards do painel.

| Variação sobre o preço médio | Situação | Cenários comparados |
|---|---|---|
| Alta de 25% ou mais | Faixa de realização de lucro | Manter tudo × realizar 10% / 20% / 30% / 40% (altas de 25% / 35% / 45% / 60%) ou venda total (a partir de +100%) |
| Entre −15% e +25% | Fora das faixas | — (o card mostra a que preço cada faixa começa) |
| Queda de 15% ou mais | Faixa de queda | Manter tudo × aumentar a posição em 10% (−15%) ou 25% (−25%) |

- **Tabela padronizada**: as mesmas 8 linhas em todos os cards (valor investido, valor da posição, movimento de caixa, resultado realizado e não realizado, preço médio, variação e renda por mês), em valores e percentuais.
- **O que o histórico mostra**: cada card cita o resultado do backtest do seu próprio cenário (Seção 12).
- **FIIs**: o que teria acontecido nos últimos 12 meses com os rendimentos (sem reinvestir, reinvestindo 50% ou 100%, ou guardando no CDI para comprar na queda), com preços, rendimentos e CDI reais.
- **Proteções** (limite de 2 aumentos de posição na queda; alertas de fundamentos e de rendimentos) aparecem como avisos no card.


In [ ]:
#@title 🧭 11. Cenários por ativo (regra de faixas de preço)
# Funções e constantes iguais às do pacote carteira_analise (o mesmo código do painel),
# importadas do pacote em vez de copiadas aqui (Fase 1, 11/2026).
from carteira_analise.regra_posicao import (
    CONFIG_REGRA_PADRAO, _EPS, novo_estado, aplicar_resets_por_preco, decidir_posicao,
    aplicar_compra, aplicar_venda, estado_a_partir_de_operacoes, quantidade_para_acao,
    filtro_renda_compra, _eh_fii_de_papel, filtro_fundamentos_compra, _dividendos_em_pregoes,
    simular_estrategia, VARIANTES_BACKTEST_REGRA, comparar_variantes, resumir_por_tipo,
    texto_simples_por_tipo, formatar_valor, montar_card_posicao,
)
from carteira_analise.contexto import _cfg
from carteira_analise.backtest_cenarios import (
    TABELA_IOF, aliquota_ir_renda_fixa, aliquota_iof, indice_cdi_acumulado, CaixaRendaFixa,
    DIAS_ATE_PAGAMENTO, _pagamentos_por_pregao, _retorno_anual, ESPACAMENTO_MINIMO_ENTRADAS,
    datas_de_entrada, VARIANTES_ACOES, simular_cenario_regra, ESTRATEGIAS_FII,
    QUEDA_SOBRE_PRECO_MEDIO, QUEDA_SOBRE_MAXIMA_12M, simular_rendimentos_fii,
    rodar_em_varias_datas, TOLERANCIA_EMPATE, _referencia_ajustada, resumir_cenarios,
    frase_evidencia, exportar_evidencias, _sem_fuso,
)
from carteira_analise.posicoes import (
    renda_12m_por_cota, _ORDEM_MERCADO, _ORDEM_GRUPO, chave_ordem_card, operacoes_por_ticker,
)
from carteira_analise.planilha import _TIPO_PARA_GRUPO
from carteira_analise.cenarios import (
    situacao_por_variacao, precos_das_faixas, distancia_da_maxima_12m, LINHAS_CENARIO, _coluna,
    tabela_cenario, _posicao_na_data, retrospecto_reinvestimento, carregar_evidencias,
    frase_evidencia_card, frase_evidencia_fiis, gerar_cenarios,
)


import html
import json as _json
import urllib.request as _urlreq
from pathlib import Path


import math


# No notebook, as evidências vêm do arquivo gerado pela Seção 12 nesta sessão
# (evidencias_cenarios.json) ou, se ele não existir, das evidências publicadas
# com o pacote na época desta versão do notebook.
ARQUIVO_EVIDENCIAS = Path("evidencias_cenarios.json")
EVIDENCIAS_PADRAO = {'periodo': '2022 e 2026', 'gerado_em': '2026-10-05', 'fonte': 'Seção 12 do notebook: backtest dos cenários com 55 ativos; até 13 datas de entrada por ativo; caixa em CDB de 100% do CDI real (Banco Central, série 12, de 05/10/2021 a 02/10/2026), lote a lote, com IR e IOF.', 'acoes': {'Ações B3': {'ativos': 12, 'realizar': {'cenario': 293, 'referencia': 312, 'pct_agiu': 71, 'pct_vence_quando_agiu': 44}, 'aumentar': {'cenario': 312, 'referencia': 310, 'pct_agiu': 38, 'pct_vence_quando_agiu': 82}}, 'Ações EUA': {'ativos': 1, 'realizar': {'cenario': 156, 'referencia': 181, 'pct_agiu': 62, 'pct_vence_quando_agiu': 25}, 'aumentar': {'cenario': 181, 'referencia': 181, 'pct_agiu': 0, 'pct_vence_quando_agiu': None}}, 'ETFs B3': {'ativos': 2, 'realizar': {'cenario': 182, 'referencia': 197, 'pct_agiu': 88, 'pct_vence_quando_agiu': 26}, 'aumentar': {'cenario': 198, 'referencia': 195, 'pct_agiu': 8, 'pct_vence_quando_agiu': 100}}, 'ETFs EUA': {'ativos': 12, 'realizar': {'cenario': 173, 'referencia': 223, 'pct_agiu': 71, 'pct_vence_quando_agiu': 24}, 'aumentar': {'cenario': 223, 'referencia': 223, 'pct_agiu': 24, 'pct_vence_quando_agiu': 76}}}, 'fiis': {'ativos': 28, 'contexto': 'período de Selic alta', 'nao_reinvestir': 140, 'reinvestir_100': {'cenario': 135, 'pct_perde': 79}, 'compra_na_queda': {'cenario': 140, 'pct_agiu': 57, 'pct_vence_quando_agiu': 57}}}


def carregar_evidencias_notebook():
    return carregar_evidencias(ARQUIVO_EVIDENCIAS) or EVIDENCIAS_PADRAO


import textwrap
import time as _time

_CDI_JANELAS = globals().get("_CDI_JANELAS", {})  # cache da sessão: (início, fim) de cada biênio -> série do CDI


def buscar_cdi_bcb(data_inicial, data_final, tentativas=3):
    """CDI diário (% ao dia útil) da série 12 do SGS/Banco Central.

    A API costuma demorar ou falhar em consultas longas: a consulta é feita em
    biênios (janelas de 2 anos), com até 3 tentativas por janela (esperas de
    2 s e 4 s), e cada biênio fica em cache na sessão — as Seções 11, 12 e 13
    reaproveitam o que já foi baixado. Devolve None se alguma janela falhar."""
    ini0, fim = pd.Timestamp(data_inicial).normalize(), pd.Timestamp(data_final).normalize()
    partes, ano = [], ini0.year - (ini0.year % 2)
    while pd.Timestamp(f"{ano}-01-01") <= fim:
        j_ini, j_fim = pd.Timestamp(f"{ano}-01-01"), min(pd.Timestamp(f"{ano + 1}-12-31"), fim)
        chave = (j_ini, j_fim)
        if chave not in _CDI_JANELAS:
            url = ("https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados?formato=json"
                   f"&dataInicial={j_ini:%d/%m/%Y}&dataFinal={j_fim:%d/%m/%Y}")
            for k in range(tentativas):
                try:
                    with _urlreq.urlopen(url, timeout=45) as resp:
                        dados = _json.loads(resp.read().decode("utf-8"))
                    _CDI_JANELAS[chave] = pd.Series([float(d["valor"]) for d in dados],
                                                    index=pd.to_datetime([d["data"] for d in dados], dayfirst=True))
                    break
                except Exception as e:
                    if k == tentativas - 1:
                        print(f"⚠️ Não foi possível baixar o CDI do Banco Central ({j_ini.year}–{j_fim.year}): {e}")
                        return None
                    _time.sleep(2 * 2 ** k)
        partes.append(_CDI_JANELAS[chave])
        ano += 2
    if not partes:
        return None
    serie = pd.concat(partes).sort_index()
    serie = serie[~serie.index.duplicated()]
    return serie[(serie.index >= ini0) & (serie.index <= fim)]


def descrever_cdi(cdi):
    """Linha que informa, no resultado, de onde veio o CDI usado."""
    return (f"Fonte do CDI: série 12 do Banco Central, de {cdi.index[0]:%d/%m/%Y} a {cdi.index[-1]:%d/%m/%Y} "
            f"({len(cdi)} dias úteis), líquido de IR e IOF.")


def imprimir_texto(texto, prefixo="", largura=100):
    """Imprime texto longo quebrando as linhas (leitura sem rolagem lateral)."""
    print(textwrap.fill(texto, width=largura, initial_indent=prefixo,
                        subsequent_indent=" " * len(prefixo)))


# ===================== cards (mesmo HTML do painel) =====================
# Cor de fundo (gradiente), ícone e cor de destaque de cada situação. Sem cinza
# e sem verde/vermelho puros (que seriam lidos como "compre"/"venda").
_ESTILO_SITUACAO = {
    "realizacao": ("linear-gradient(135deg,#4f6bff,#2f45e0)", "📈", "#2f45e0"),
    "queda": ("linear-gradient(135deg,#f0a93b,#d9831c)", "📉", "#d9831c"),
    "fora": ("linear-gradient(135deg,#2bb3a3,#0f7f73)", "↔️", "#0f7f73"),
}
_NOME_GRUPO = {"Ações": "Ações", "FIIs": "Fundos Imobiliários (FIIs)", "ETF": "ETFs", "Outros": "Outros ativos"}


def _fmt_celula(fmt: str, valor, moeda: str) -> str:
    """Formata uma célula das tabelas de cenário."""
    if valor is None:
        return "—"
    if fmt == "caixa":
        tipo, v = valor
        return ("+" if v > 0 else "−") + formatar_valor(abs(v), moeda) + f" ({tipo})"
    if fmt == "sinal":
        return ("+" if valor > 0 else "−" if valor < 0 else "") + formatar_valor(abs(valor), moeda)
    if fmt == "pct":
        return f"{valor * 100:+.0f}%".replace("-", "−")
    if fmt == "cotas":
        return f"{valor:.0f}" if abs(valor - round(valor)) < 1e-9 else f"{valor:.2f}"
    return formatar_valor(valor, moeda)


def _html_tabela(colunas: list, linhas: list, moeda: str) -> str:
    th = "".join(f"<th style='text-align:right;padding:3px 6px;font-weight:600;white-space:nowrap'>"
                 f"{html.escape(c)}</th>" for c in colunas)
    trs = ""
    for nome, fmt, *valores in linhas:
        tds = "".join(f"<td style='text-align:right;padding:3px 6px;white-space:nowrap'>"
                      f"{html.escape(_fmt_celula(fmt, v, moeda))}</td>" for v in valores)
        trs += f"<tr><td style='padding:3px 6px;opacity:.9;white-space:nowrap'>{html.escape(nome)}</td>{tds}</tr>"
    return ("<table style='width:100%;border-collapse:collapse;font-size:.8em;margin-top:6px;"
            f"background:rgba(0,0,0,.12);border-radius:8px'><tr><th></th>{th}</tr>{trs}</table>")


def _html_card_cenario(card: dict) -> str:
    """Card de um ativo: situação, preços e contexto de mercado; tabela de
    cenários (ações/ETFs na faixa) ou próximas faixas; para FIIs, o
    retrospecto de reinvestimento dos últimos 12 meses."""
    fundo, icone, _ = _ESTILO_SITUACAO[card["situacao"]]
    moeda = card["moeda"]
    var = card["variacao"]
    partes = [
        f"<div style='font-size:1.3em;font-weight:700;margin-top:10px'>{html.escape(card['rotulo_situacao'])}</div>",
        f"<div style='font-size:.82em;opacity:.92'>Preço médio {formatar_valor(card['preco_medio'], moeda)} · "
        f"hoje {formatar_valor(card['preco_atual'], moeda)}</div>",
    ]
    if card.get("abaixo_da_maxima_12m") is not None:
        d = card["abaixo_da_maxima_12m"]
        texto = "na máxima" if d < 0.005 else f"{d * 100:.0f}% abaixo da máxima"
        partes.append(f"<div style='font-size:.82em;opacity:.92'>Mercado: a cota está {texto} dos últimos 12 meses</div>")
    tabela = card.get("tabela")
    if tabela:
        partes.append(f"<div style='margin-top:10px;font-weight:600;font-size:.9em'>{html.escape(tabela['titulo'])}</div>")
        partes.append(_html_tabela(tabela["colunas"], tabela["linhas"], moeda))
    else:
        f = card["faixas"]
        partes.append(f"<div style='margin-top:10px;font-size:.85em'>Próximas faixas: realização a partir de "
                      f"{formatar_valor(f['realizacao'], moeda)} (+25%) · aumento abaixo de "
                      f"{formatar_valor(f['queda'], moeda)} (−15%).</div>")
        if card.get("renda_mensal") and not card.get("retrospecto"):
            partes.append(f"<div style='margin-top:6px;font-size:.85em'>💵 Renda estimada: cerca de "
                          f"{formatar_valor(card['renda_mensal'], moeda)} por mês.</div>")
    for nota in card.get("notas", []):
        partes.append(f"<div style='margin-top:6px;font-size:.82em'>ℹ️ {html.escape(nota)}</div>")
    for alerta in card.get("alertas", []):
        partes.append(f"<div style='margin-top:6px;font-size:.82em'>⚠️ Atenção: {html.escape(alerta)}.</div>")
    retro = card.get("retrospecto")
    if retro:
        periodo = "nos últimos 12 meses" if retro["meses"] >= 12 else f"desde a compra ({retro['meses']} meses)"
        partes.append("<div style='margin-top:10px;padding-top:8px;border-top:1px solid rgba(255,255,255,.3);"
                      f"font-weight:600;font-size:.9em'>Rendimentos {html.escape(periodo)}: e se tivesse reinvestido?</div>")
        partes.append(_html_tabela(retro["colunas"], retro["linhas"], moeda))
        q = retro["compra_na_queda"]
        if q["atingida"]:
            texto_queda = (f"Compra na queda: a faixa (abaixo de {formatar_valor(q['preco_faixa'], moeda)}) foi atingida; "
                           f"comprando com a reserva do CDI, hoje seriam {q['cotas']:.0f} cotas, renda de "
                           f"{formatar_valor(q['renda_mensal'], moeda)}/mês e patrimônio de "
                           f"{formatar_valor(q['patrimonio'], moeda)}.")
        else:
            texto_queda = (f"Compra na queda: faixa não atingida no período (abaixo de "
                           f"{formatar_valor(q['preco_faixa'], moeda)}). Reserva no CDI: {formatar_valor(q['reserva'], moeda)}.")
        partes.append(f"<div style='font-size:.78em;margin-top:6px'>{html.escape(texto_queda)}</div>")
    if card.get("evidencia"):
        partes.append("<div style='margin-top:10px;padding:6px 8px;background:rgba(255,255,255,.15);"
                      f"border-radius:8px;font-size:.8em'>📊 <b>O que o histórico mostra:</b> {card['evidencia']}</div>")
    partes.append("<div style='margin-top:8px;font-size:.72em;opacity:.85;font-style:italic'>"
                  "Cenários para comparação, pela regra de faixas. A decisão é sempre sua.</div>")
    return (f"<div style='background:{fundo};color:white;border-radius:16px;padding:16px 18px;"
            f"box-shadow:0 6px 18px rgba(0,0,0,.15)'>"
            f"<div style='display:flex;justify-content:space-between;align-items:center'>"
            f"<div style='display:flex;gap:10px;align-items:center'>"
            f"<div style='background:rgba(255,255,255,.22);border-radius:10px;width:36px;height:36px;display:flex;"
            f"align-items:center;justify-content:center;font-size:19px'>{icone}</div>"
            f"<div style='font-weight:700;font-size:1.1em'>{html.escape(card['ticker'])}</div></div>"
            f"<div title='variação sobre o seu preço médio' style='font-size:.85em;font-weight:600;"
            f"background:rgba(255,255,255,.2);border-radius:999px;padding:3px 10px;white-space:nowrap'>"
            f"{var * 100:+.0f}%</div></div>{''.join(partes)}</div>")


def _html_resumo_cenarios(cards: list) -> str:
    """Resumo no topo: quantos ativos em cada situação."""
    blocos = ""
    for situacao, rotulo in (("realizacao", "Faixa de realização de lucro"), ("queda", "Faixa de queda"),
                             ("fora", "Fora das faixas")):
        _, icone, cor = _ESTILO_SITUACAO[situacao]
        n = sum(1 for c in cards if c["situacao"] == situacao)
        blocos += (f"<div style='background:white;color:#1f2937;border-radius:16px;padding:14px 16px;"
                   f"box-shadow:0 4px 14px rgba(0,0,0,.1);border-top:4px solid {cor}'>"
                   f"<div style='background:{cor};border-radius:10px;width:34px;height:34px;display:flex;"
                   f"align-items:center;justify-content:center'>{icone}</div>"
                   f"<div style='font-size:1.9em;font-weight:700;margin-top:8px'>{n}</div>"
                   f"<div style='font-size:.9em;color:#4b5563'>{rotulo}</div></div>")
    return f"<div style='display:grid;grid-template-columns:repeat(3,1fr);gap:14px;margin-bottom:18px'>{blocos}</div>"


def _html_cabecalho_fiis(evidencias) -> str:
    """Bloco único no topo do grupo de FIIs, com o que vale para todos."""
    frase = frase_evidencia_fiis(evidencias)
    historico = f"📊 <b>O que o histórico mostra:</b> {frase}<br>" if frase else ""
    return ("<div style='background:#2a2f3a;color:#f3f4f6;border-radius:12px;padding:10px 14px;margin:0 0 12px;"
            f"font-size:.82em;line-height:1.45'>{historico}"
            "📖 <b>Como ler a tabela de cada FII:</b> a coluna <i>Real</i> é o que aconteceu com a sua posição nos "
            "últimos 12 meses, sem reinvestir; as outras simulam o reinvestimento dos rendimentos, com os preços e "
            "rendimentos reais do período. Só cotas inteiras, compradas na data aproximada de pagamento (a sobra "
            "fica parada). O dinheiro em caixa rende o CDI, como CDB de 100% do CDI, com IR regressivo (22,5% a 15%) "
            "e IOF, como se resgatado hoje; rendimentos de FII são isentos. <i>Compra na queda</i>: os rendimentos "
            "ficam no CDI e só compram cotas se o preço cair 15% abaixo do seu preço médio.</div>")


def _html_cabecalho_evidencias(cards_do_grupo: list) -> str:
    """Bloco único no topo de um grupo de ações/ETFs com o que o histórico
    mostra para cada faixa: a frase é a mesma para todos os ativos do grupo,
    por isso aparece uma vez só (e não repetida em cada card)."""
    rotulos = {"realizacao": "📈 Faixa de realização de lucro", "queda": "📉 Faixa de queda"}
    blocos = []
    for situacao in ("realizacao", "queda"):
        frase = next((c["evidencia"] for c in cards_do_grupo
                      if c["situacao"] == situacao and c.get("evidencia")), None)
        if frase:
            blocos.append(f"<b>{rotulos[situacao]}</b> — {frase}")
    if not blocos:
        return ""
    return ("<div style='background:#2a2f3a;color:#f3f4f6;border-radius:12px;padding:10px 14px;margin:0 0 12px;"
            "font-size:.82em;line-height:1.45'>📊 <b>O que o histórico mostra</b> (vale para os cards deste grupo)"
            "<br>" + "<br>".join(blocos) + "</div>")


def _html_grupos_cenarios(cards: list, evidencias) -> str:
    """Cards agrupados por mercado e tipo de ativo, com um cabeçalho único
    no topo de cada grupo (FIIs: evidências e como ler a tabela; demais:
    evidências por faixa)."""
    grupos: dict = {}
    for card in cards:
        grupos.setdefault((card["mercado"], card["grupo"]), []).append(card)
    partes = []
    for chave, cards_do_grupo in grupos.items():
        partes.append(f"<div style='font-weight:700;font-size:1.05em;margin:6px 0 10px 2px'>"
                      f"{html.escape(chave[0])} · {html.escape(_NOME_GRUPO.get(chave[1], chave[1]))}</div>")
        partes.append(_html_cabecalho_fiis(evidencias) if chave[1] == "FIIs"
                      else _html_cabecalho_evidencias(cards_do_grupo))
        grade = [_html_card_cenario({**c, "evidencia": None}) for c in cards_do_grupo]
        partes.append("<div style='display:grid;grid-template-columns:repeat(auto-fill,minmax(460px,1fr));"
                      "gap:16px;margin-bottom:22px;align-items:start'>" + "".join(grade) + "</div>")
    return "".join(partes)


def _tipo_por_ticker_secao2():
    """Tipo de cada ticker conforme os grupos preenchidos na Seção 2."""
    mapa = {}
    for caixa, tipo in ((caixa_acoes, 'acao'), (caixa_fiis, 'fii'), (caixa_acoes_us, 'acao_us'),
                        (caixa_etf_br, 'etf_br'), (caixa_etf_us, 'etf_us')):
        for t in parse_tickers(caixa.value):
            mapa[t] = tipo
    return mapa


def _fundamentos_para_filtro(ticker, tipo):
    if tipo == 'fii':
        return indicadores_fundamentalistas_fii(ticker)
    if tipo in ('acao', 'acao_us'):
        return indicadores_fundamentalistas_acao(ticker)
    return {}


class _FonteNotebook:
    """Fonte de dados do notebook no formato que o pacote espera."""

    def baixar_precos(self, ticker, periodo):
        tec = indicadores_tecnicos(ticker, periodo)
        return tec["historico"] if tec else pd.Series(dtype=float)

    def baixar_dividendos(self, ticker):
        return _dividendos_brutos_cache(ticker)


CDI_RESERVA_AA = 0.14  # usado só se a API do Banco Central estiver fora do ar
botao_cenarios = widgets.Button(description="🧭 Gerar cenários", button_style='primary', icon='compass')
saida_cenarios = widgets.Output()


def _gerar_cenarios_click(b):
    with saida_cenarios:
        clear_output()
        carteira = globals().get('_carteira_importada')
        if carteira is None:
            print("⚠️ Importe primeiro a sua carteira na Seção 9.3 (botão '📂 Importar e calcular').")
            return
        print("Calculando os cenários de cada ativo... isso pode levar alguns segundos.\n")
        tipos = _tipo_por_ticker_secao2()
        fundamentos = {t: _fundamentos_para_filtro(t, tp) for t, tp in tipos.items()}
        hoje = pd.Timestamp.today().normalize()
        cdi = buscar_cdi_bcb(hoje - pd.Timedelta(days=800), hoje)
        evidencias = carregar_evidencias_notebook()
        cards, avisos = gerar_cenarios(*carteira, _FonteNotebook(), tipos, fundamentos,
                                       indice_cdi_por_data=lambda idx: indice_cdi_acumulado(idx, cdi, cdi_aa=CDI_RESERVA_AA),
                                       evidencias=evidencias)
        clear_output()
        if not cards:
            print("Nenhuma posição aberta encontrada na carteira importada.")
        else:
            display(HTML(
                "<p style='font-size:.9em'>Compare o que acontece em cada caminho. Os números são calculados pela regra "
                "de faixas e não constituem recomendação: a decisão é sempre sua.</p>"
                + _html_resumo_cenarios(cards) + _html_grupos_cenarios(cards, evidencias)))
        if cdi is None and any(c.get("retrospecto") for c in cards):
            print(f"⚠️ CDI do Banco Central indisponível — o retrospecto dos FIIs usa {CDI_RESERVA_AA * 100:.0f}% ao ano.")
        for a in avisos:
            print(f"⚠️ {a}")


botao_cenarios.on_click(_gerar_cenarios_click)
display(widgets.HTML("<h3>11. Cenários por ativo</h3>"))
display(widgets.VBox([botao_cenarios, saida_cenarios]))


## 📊 12. Backtesting dos cenários comparativos

Gera a evidência histórica mostrada nos cards de cenários ("o que o histórico mostra"), testando **cada cenário isoladamente** contra a referência, nos ativos da Seção 2:

- **Ações e ETFs**, comparados com **manter**: *só realizar lucro nas faixas de alta*, *só aumentar a posição nas faixas de queda* e a *regra completa* (as duas coisas juntas).
- **FIIs**, comparados com **não reinvestir**: *reinvestir 50%* ou *100%* dos rendimentos, e *guardar os rendimentos no CDI e comprar cotas na queda* — com duas definições de queda, fixadas antes de ver os resultados: 15% abaixo do preço médio (a faixa da regra do painel) e 10% abaixo da máxima dos últimos 12 meses (referência de mercado).

**Para o resultado não depender da sorte de uma data**, cada ativo é simulado a partir de **até 13 datas de entrada** (cerca de uma a cada 3 meses) e comparado com a referência na mesma data. São investidores simulados: o resultado não depende das datas de compra da carteira real. Ativos com histórico curto entram com menos datas, mantendo ao menos ~1 mês entre elas. O dinheiro guardado rende o **CDI real** (série 12 do Banco Central), tratado como CDB de 100% do CDI, **lote a lote**, com IR regressivo e IOF.

**Limitações**: o filtro de fundamentos não entra (não há histórico gratuito desses dados); custos de corretagem e IR sobre ganho de capital nas vendas não são considerados; o caixa de ativos dos EUA não rende; a escolha dos ativos testados foi feita no presente (para reduzir o viés de sobrevivência, inclua fundos com desempenho ruim).


In [ ]:
#@title 📊 12. Backtesting dos cenários comparativos
# Funções e constantes iguais às do pacote carteira_analise (o mesmo código do painel),
# importadas do pacote em vez de copiadas aqui (Fase 1, 11/2026).
from carteira_analise.backtest_cenarios import (
    TABELA_IOF, aliquota_ir_renda_fixa, aliquota_iof, indice_cdi_acumulado, CaixaRendaFixa,
    DIAS_ATE_PAGAMENTO, _sem_fuso, _pagamentos_por_pregao, _retorno_anual,
    ESPACAMENTO_MINIMO_ENTRADAS, datas_de_entrada, VARIANTES_ACOES, simular_cenario_regra,
    ESTRATEGIAS_FII, QUEDA_SOBRE_PRECO_MEDIO, QUEDA_SOBRE_MAXIMA_12M, simular_rendimentos_fii,
    rodar_em_varias_datas, TOLERANCIA_EMPATE, _referencia_ajustada, resumir_cenarios,
    frase_evidencia, exportar_evidencias,
)


import math


# ---------------------------------------------------------------------------
# Painel da Seção 12 — roda os backtests dos cenários no Colab
# ---------------------------------------------------------------------------
import json as _json
from pathlib import Path
import urllib.request as _urlreq


MINIMO_PREGOES_BACKTEST = 600  # ~2,4 anos: 1 ano antes da 1ª entrada + 1 ano depois da última


def _preparar_series_backtest(ticker):
    """Preço limpo e proventos de um ativo, ou (None, None, motivo) quando ele
    não tem histórico suficiente — com o motivo, para o relatório."""
    bruto = _baixar_fechamento_cache(ticker)
    if bruto is None or bruto.empty:
        return None, None, "sem dados no Yahoo (possivelmente deslistado ou com código alterado)"
    fechamento, _, n_truncados = limpar_outliers_precos(bruto)
    if len(fechamento) < MINIMO_PREGOES_BACKTEST:
        if n_truncados > 0:
            return None, None, (f"histórico cortado em {fechamento.index[0]:%d/%m/%Y} por mudança de patamar de um dia "
                                f"para o outro (provável grupamento/desdobramento não ajustado pelo Yahoo); "
                                f"sobraram {len(fechamento)} pregões")
        return None, None, f"histórico curto: {len(bruto)} pregões (negociado há menos de ~2,4 anos)"
    return fechamento, _dividendos_brutos_cache(ticker), None


botao_backtest_cenarios = widgets.Button(description="📊 Rodar os backtests dos cenários", button_style='warning',
                                         icon='history', layout=widgets.Layout(width='320px'))
saida_backtest_cenarios = widgets.Output()

_NOMES_TIPO_CENARIOS = {'acao': 'Ações B3', 'fii': 'FIIs', 'acao_us': 'Ações EUA',
                        'etf_br': 'ETFs B3', 'etf_us': 'ETFs EUA'}
_ROTULO_TIPO = {'Ações B3': 'ações B3', 'FIIs': 'FIIs', 'Ações EUA': 'ações dos EUA',
                'ETFs B3': 'ETFs B3', 'ETFs EUA': 'ETFs dos EUA'}
_ROTULO_SINGULAR = {'Ações B3': 'ação B3', 'FIIs': 'FII', 'Ações EUA': 'ação dos EUA',
                    'ETFs B3': 'ETF B3', 'ETFs EUA': 'ETF dos EUA'}


def _fmt_tabela_cenarios(df):
    t = df.drop(columns=['pct_supera_referencia'], errors='ignore').rename(columns={
        'tipo': 'Tipo', 'cenario': 'Cenário', 'ativos': 'Ativos', 'casos': 'Casos (ativo × data)',
        'retorno_aa_mediano': 'Retorno anual mediano',
        'pct_vence': 'Vence', 'pct_empata': 'Empata', 'pct_perde': 'Perde',
        'pct_agiu': 'Agiu (% dos casos)', 'pct_vence_quando_agiu': 'Vence quando agiu',
        'valor_de_100_mediano': '100 viraram (cenário)', 'valor_de_100_ref_mediano': '100 viraram (referência)',
        'pct_com_compra_na_queda': 'Faixa de queda atingida',
        'maior_queda_mediana': 'Maior queda (cenário)', 'maior_queda_mediana_ref': 'Maior queda (referência)',
        'pct_queda_menor': 'Queda menor que a referência', 'retorno_aa_p10': 'Retorno anual (10% piores)'})
    for c in ['Maior queda (cenário)', 'Maior queda (referência)', 'Retorno anual (10% piores)']:
        if c in t:
            t[c] = t[c].map(lambda v: '' if pd.isna(v) else f"{v*100:.0f}%")
    if 'Queda menor que a referência' in t:
        t['Queda menor que a referência'] = t['Queda menor que a referência'].map(lambda v: '' if pd.isna(v) else f"{v:.0f}%")
    if 'Retorno anual mediano' in t:
        t['Retorno anual mediano'] = t['Retorno anual mediano'].map(lambda v: '' if pd.isna(v) else f"{v*100:.1f}%")
    for c in [c for c in ['Vence', 'Empata', 'Perde', 'Agiu (% dos casos)', 'Vence quando agiu',
                          'Faixa de queda atingida'] if c in t]:
        t[c] = t[c].map(lambda v: '' if pd.isna(v) else f"{v:.0f}%")
    for c in [c for c in ['100 viraram (cenário)', '100 viraram (referência)'] if c in t]:
        t[c] = t[c].map(lambda v: '' if pd.isna(v) else f"{v:.0f}")
    return t


def _rodar_backtest_cenarios_click(b):
    with saida_backtest_cenarios:
        clear_output()
        tipos = {t: _NOMES_TIPO_CENARIOS[tp] for t, tp in _tipo_por_ticker_secao2().items()}
        if not tipos:
            print("⚠️ Nenhum ticker configurado na Seção 2.")
            return
        series, ignorados = {}, []
        for ticker in tipos:
            fechamento, divs, motivo = _preparar_series_backtest(ticker)
            if fechamento is None:
                ignorados.append((ticker, motivo))
                continue
            series[ticker] = (fechamento, divs)
        if not series:
            print("⚠️ Nenhum ativo com histórico suficiente.")
            return

        # CDI: série real do Banco Central (ou taxa de reserva) para ativos da B3;
        # ativos dos EUA ficam com o caixa sem rendimento (premissa conservadora).
        datas_b3 = [s.index for t, (s, _) in series.items() if t.endswith('.SA')]
        cdi = None
        if datas_b3:
            ini = min(d[0] for d in datas_b3); fim = max(d[-1] for d in datas_b3)
            print(f"Baixando o CDI diário do Banco Central ({ini:%d/%m/%Y} a {fim:%d/%m/%Y})...")
            cdi = buscar_cdi_bcb(ini, fim)
            if cdi is None:
                imprimir_texto("⚠️ Backtest interrompido: o CDI do Banco Central não pôde ser baixado. Uma taxa fixa "
                               "distorceria a comparação entre anos de juros muito diferentes (cerca de 2% ao ano em "
                               "2021 e 14% em 2025). Tente de novo em alguns minutos.")
                return

        res_acoes, res_fii = {}, {}
        for n, (ticker, (fechamento, divs)) in enumerate(series.items(), 1):
            print(f"  [{n}/{len(series)}] {ticker}...", end="\r")
            indice = None
            if ticker.endswith('.SA'):
                indice = indice_cdi_acumulado(fechamento.index, cdi)
            if tipos[ticker] == 'FIIs':
                res_fii[ticker] = rodar_em_varias_datas(simular_rendimentos_fii, fechamento, divs,
                                                        list(ESTRATEGIAS_FII), indice_cdi=indice)
            else:
                res_acoes[ticker] = rodar_em_varias_datas(simular_cenario_regra, fechamento, divs,
                                                          list(VARIANTES_ACOES), indice_cdi=indice)
        clear_output()

        if cdi is not None:
            imprimir_texto(descrever_cdi(cdi) + " Ativos dos EUA: caixa sem rendimento.")
            print()
        if ignorados:
            print(f"⚠️ {len(ignorados)} ativo(s) fora do backtest:")
            for ticker, motivo in ignorados:
                print(f"   • {ticker}: {motivo}")
            print()
        print("=" * 78)
        print("💬 O QUE O HISTÓRICO MOSTRA (frases para os cards)")
        print("=" * 78)
        df_acoes = resumir_cenarios(res_acoes, tipos, 'Manter') if res_acoes else pd.DataFrame()
        df_fii = resumir_cenarios(res_fii, tipos, 'Não reinvestir') if res_fii else pd.DataFrame()
        for tipo in (df_acoes['tipo'].unique() if not df_acoes.empty else []):
            moeda = 'US$' if 'EUA' in tipo else 'R$'
            print(f"\n▶ {tipo}")
            for cen in ('Só realizar lucro nas faixas', 'Só aumentar nas quedas', 'Regra completa'):
                ref = 'manter' if cen == 'Só realizar lucro nas faixas' else 'manter e guardar o mesmo dinheiro no CDI'
                frase = frase_evidencia(df_acoes, tipo, cen, 'Manter', _ROTULO_TIPO[tipo], moeda,
                                        _ROTULO_SINGULAR[tipo], rotulo_referencia=ref)
                if frase:
                    imprimir_texto(frase[0].upper() + frase[1:], prefixo="  • ")
        if not df_fii.empty:
            print("\n▶ FIIs — o que fazer com os rendimentos")
            for cen in list(ESTRATEGIAS_FII)[1:]:
                frase = frase_evidencia(df_fii, 'FIIs', cen, 'Não reinvestir', 'FIIs', 'R$', 'FII',
                                        rotulo_referencia='não reinvestir (guardar no CDI)')
                if frase:
                    imprimir_texto(frase[0].upper() + frase[1:], prefixo="  • ")
            for cen in [c for c in ESTRATEGIAS_FII if 'queda' in c]:
                linha = df_fii[df_fii['cenario'] == cen]
                if not linha.empty and not pd.isna(linha.iloc[0]['pct_com_compra_na_queda']):
                    imprimir_texto(f"\"{cen}\": a faixa de queda foi atingida em "
                                   f"{linha.iloc[0]['pct_com_compra_na_queda']:.0f}% dos casos.", prefixo="  • ")

        print("\n" + "=" * 78)
        print("📋 TABELAS PARA O RELATÓRIO")
        print("=" * 78)
        if not df_acoes.empty:
            print("\nAções e ETFs — cada cenário comparado com manter:")
            display(_fmt_tabela_cenarios(df_acoes.drop(columns=['maior_queda_mediana', 'maior_queda_mediana_ref', 'pct_queda_menor', 'retorno_aa_p10'])).drop(columns=['Faixa de queda atingida']))
            print("\nRisco — maior queda da carteira (ativo + caixa) a partir de um topo, mediana dos casos em que o")
            print("cenário agiu, comparada com a referência na mesma data de entrada:")
            risco = df_acoes[df_acoes['cenario'] != 'Manter'][['tipo', 'cenario', 'maior_queda_mediana',
                                                                'maior_queda_mediana_ref', 'pct_queda_menor']]
            display(_fmt_tabela_cenarios(risco))
        if not df_fii.empty:
            print("\nFIIs — destino dos rendimentos, comparado com não reinvestir:")
            display(_fmt_tabela_cenarios(df_fii.drop(columns=['maior_queda_mediana', 'maior_queda_mediana_ref', 'pct_queda_menor', 'retorno_aa_p10'])))

        # Arquivo de evidências usado pelos cards (Seção 11 e painel)
        inicios = [r["data_inicio"] for res in (res_acoes, res_fii) for por_cen in res.values()
                   for lista in por_cen.values() for r in lista]
        fins = [r["data_fim"] for res in (res_acoes, res_fii) for por_cen in res.values()
                for lista in por_cen.values() for r in lista]
        if inicios:
            periodo = f"{min(inicios).year} e {max(fins).year}"
            fonte_ev = (f"Seção 12 do notebook: backtest dos cenários com {len(series)} ativos; até 13 datas de entrada "
                        f"por ativo; caixa em CDB de 100% do CDI " + "real (Banco Central)" + ", lote a lote, com IR e IOF.")
            evid = exportar_evidencias(df_acoes, df_fii, periodo, pd.Timestamp.today().strftime("%Y-%m-%d"), fonte_ev)
            Path("evidencias_cenarios.json").write_text(_json.dumps(evid, ensure_ascii=False, indent=2), encoding="utf-8")
            print("\n💾 Evidências salvas em 'evidencias_cenarios.json' (painel de arquivos do Colab). A Seção 11 já")
            print("   passa a usá-las nesta sessão. Para atualizar o painel, baixe o arquivo e suba no GitHub em")
            print("   carteira_analise/dados/ (substituindo o atual).")

        print("\n📌 Como a simulação funciona:")
        print("   • Cada ativo é simulado a partir de até 13 datas de entrada (~1 a cada 3 meses), de ~4 anos atrás")
        print("     até ~1 ano atrás; ativos com histórico curto entram com menos datas (mínimo de ~1 mês entre elas).")
        print("     Os investidores são SIMULADOS: o resultado não depende das datas de compra da sua carteira.")
        print("   • Em cada data, R$ 10 mil iniciais; o resultado é comparado com a referência na MESMA data de entrada.")
        print("   • Ações/ETFs: 'só realizar' vende nas faixas de alta; 'só aumentar' compra nas faixas de queda")
        print("     (com dinheiro novo, até 2 vezes por ciclo); 'regra completa' faz as duas coisas.")
        print("   • FIIs: sem dinheiro novo; o rendimento é reinvestido (só cotas inteiras, sobra parada),")
        print("     guardado no CDI, ou guardado no CDI e usado para comprar cotas nos dias de queda.")
        print("   • Preços do Yahoo SEM ajuste por proventos (auto_adjust=False); os proventos entram à parte.")
        print("   • Dinheiro guardado (vendas e proventos): CDB de 100% do CDI, lote a lote, com IR regressivo")
        print("     (22,5% a 15%) e IOF abaixo de 30 dias, como se resgatado no fim. Ativos dos EUA: caixa sem")
        print("     rendimento (premissa conservadora). Proventos de FII e dividendos: isentos.")
        print("   • Comparação justa: quando o cenário usa dinheiro novo (aumentar nas quedas), a referência")
        print("     recebe o MESMO dinheiro, nas mesmas datas, guardado no CDI.")
        print("   • Risco: 'maior queda' = maior perda da carteira (ativo + caixa) a partir de um topo, medida")
        print("     como uma cota (dinheiro novo não conta como valorização). Realizar lucro troca parte do ganho")
        print("     possível por proteção: a tabela de risco mostra o outro lado da comparação de retorno.")
        print("   • Vence/Empata/Perde: empate = o cenário não chegou a agir (a faixa não foi atingida).")
        print("     'Vence quando agiu' considera só os casos em que houve compra ou venda.")
        print("   • 'Retorno anual mediano' anualiza o resultado de cada caso; '100 viraram' usa a entrada mais")
        print("     antiga (~4 anos). Sem custos de corretagem e sem IR sobre ganho de capital nas vendas.")
        print("   • A limpeza de dados só descarta trechos com mudança de patamar de um dia para o outro")
        print("     (grupamento não ajustado); quedas reais são mantidas, para evitar viés de sobrevivência.")


botao_backtest_cenarios.on_click(_rodar_backtest_cenarios_click)
display(widgets.HTML("<h3>12. Backtesting dos cenários comparativos</h3>"))
display(widgets.VBox([botao_backtest_cenarios, saida_backtest_cenarios]))


## 🔍 13. Teste de critérios de "preço atrativo" para FIIs

O sinal técnico (Seção 10) teve desempenho próximo de "não fazer nada". Esta seção testa outra pergunta: **o FII está barato em relação a ele mesmo ou à renda fixa?** — e se, no passado, isso aumentou a chance de o retorno dos 12 meses seguintes superar o **CDI líquido de IR**.

- **Caminho 2 — yield vs. o próprio histórico:** yield dos últimos 12 meses **15% ou mais acima** da média do próprio FII nos 24 meses anteriores.
- **Caminho 3 — prêmio sobre o Tesouro IPCA+:** prêmio (yield − juro real do Tesouro IPCA+) **1 ponto percentual ou mais acima** da média do próprio FII nos 24 meses anteriores. Juro real a partir do arquivo oficial do Tesouro Transparente.
- Nos dois casos, **rendimentos estáveis** são exigidos (queda de no máximo 10% nos 12 meses e nos últimos 3 meses), para evitar a *armadilha do yield*: o yield parece alto porque o preço caiu antecipando um corte de rendimentos.

Os critérios foram **fixados antes de ver os resultados**. O teste baixa ~10 anos de preços. Os rendimentos vêm, por padrão, da página de proventos de cada FII no Fundamentus (valores efetivamente pagos), porque o histórico do Yahoo tem lacunas de vários anos; os informes mensais da CVM (fonte oficial) servem de conferência, e o botão de conferência compara as três fontes. Só janelas de 12 meses completas contam, e o período efetivo de cada FII aparece no quadro de cobertura e compara a taxa de acerto **com o critério** contra a taxa em **todos os casos**, com intervalo de confiança por reamostragem de FIIs. No fim, mostra quais FIIs atendem aos critérios hoje — **para investigar, não como recomendação**.


In [ ]:
#@title 🔍 13. Teste de critérios de "preço atrativo" para FIIs
# Funções e constantes iguais às do pacote carteira_analise (o mesmo código do painel),
# importadas do pacote em vez de copiadas aqui (Fase 1, 11/2026).
from carteira_analise.fontes.tesouro import (
    URL_TESOURO, _sem_acento, _coluna_csv, juro_real_tesouro_ipca, baixar_juro_real_tesouro_ipca,
)
from carteira_analise.fontes.cvm_fii import (
    URL_CVM_FII, _normalizar, _achar, _numero, ler_zip_informes, _ultima_versao, ticker_do_isin,
    montar_base_fii, serie_rendimentos, serie_valor_patrimonial, comparar_com_referencia,
    baixar_informes,
)
from carteira_analise.fontes.rendimentos import (
    ORDEM_PADRAO, NOMES_FONTES, TOLERANCIA_MES, TOLERANCIA_12M, TAXA_MINIMA, MESES_MINIMOS, SELOS,
    rendimentos_por_mes, _somas_12m, _melhor_alinhamento, comparar_mes_a_mes, comparar_somas_12m,
    diagnosticar_escala, _texto_escala, ajustar_por_desdobramentos, _nomes_fontes, _texto_evento,
    reescalar_ate, conferir_fontes, _conferir, resumo_conferencia,
)
from carteira_analise.oportunidades import (
    DESCONTO_MINIMO_YIELD, PREMIO_EXTRA_MINIMO, QUEDA_MAXIMA_RENDIMENTOS, MIN_PAGAMENTOS_12M,
    MESES_MEDIA, IR_CDI_12M, PVP_DESCONTO, PVP_ABAIXO_DA_MEDIA, _vp_com_defasagem, _ipca_por_mes,
    ipca_acumulado, renda_acima_da_inflacao, avaliar_fii_mes_a_mes, _taxa, resumir_criterio,
    frase_resultado,
)
from carteira_analise.contexto import _sem_fuso
from carteira_analise.fundamentalistas import _SEGMENTOS_PAPEL, _SIGLAS_PAPEL


import io
import unicodedata
import urllib.request


import zipfile


# ===================== interface da Seção 13 =====================
_PROVENTOS_FUNDAMENTUS = globals().get("_PROVENTOS_FUNDAMENTUS", {})  # cache da sessão: proventos de FII por papel


def proventos_fii_fundamentus(ticker):
    """Rendimentos pagos por cota de um FII (página pública de proventos do
    Fundamentus), indexados pela data-com. Só o tipo "Rendimento" entra
    (amortizações devolvem capital). Mesmo acesso usado na Seção 6.
    Devolve uma série vazia se a busca falhar."""
    papel = ticker.upper().replace(".SA", "")
    if papel in _PROVENTOS_FUNDAMENTUS:
        return _PROVENTOS_FUNDAMENTUS[papel]
    url = f"https://www.fundamentus.com.br/fii_proventos.php?papel={papel}&tipo=2"
    try:
        req = urllib.request.Request(url, headers={'User-Agent': _USER_AGENT_FUNDAMENTUS})
        with urllib.request.urlopen(req, timeout=20) as resp:
            html = resp.read().decode("latin-1")
        df = _ler_tabela_fundamentus_como_texto(html)
        norm = {c: " ".join(str(c).lower().split()) for c in df.columns}
        c_data = next(c for c, n in norm.items() if "data com" in n)
        c_valor = next(c for c, n in norm.items() if n.startswith("valor"))
        c_tipo = next((c for c, n in norm.items() if n.startswith("tipo")), None)
        if c_tipo:
            df = df[df[c_tipo].astype(str).str.lower().str.contains("rendimento")]
        serie = pd.Series(df[c_valor].map(_parse_numero_br_fundamentus).values,
                          index=pd.to_datetime(df[c_data], dayfirst=True, errors="coerce").values, dtype=float)
        serie = serie[serie.index.notna() & serie.notna() & (serie > 0)].groupby(level=0).sum().sort_index()
    except Exception:
        # falha (site fora do ar, certificado inválido...) NÃO fica em cache:
        # o próximo clique tenta de novo, em vez de repetir a série vazia
        return pd.Series(dtype=float, index=pd.DatetimeIndex([]))
    _PROVENTOS_FUNDAMENTUS[papel] = serie
    _time.sleep(0.3)  # gentileza com o site: uma consulta por vez
    return serie


_CASOS_FII_ULTIMO_TESTE = globals().get("_CASOS_FII_ULTIMO_TESTE", {"casos": None})  # último teste (célula 13b)
_EVENTOS_YAHOO = globals().get("_EVENTOS_YAHOO", {})  # cache da sessão: desdobramentos/grupamentos por FII


def eventos_yahoo(ticker):
    """Desdobramentos e grupamentos registrados pelo Yahoo (data -> fator)."""
    if ticker not in _EVENTOS_YAHOO:
        try:
            ev = yf.Ticker(ticker).splits
            ev = ev if ev is not None else pd.Series(dtype=float)
            if len(ev) and getattr(ev.index, "tz", None) is not None:
                ev.index = ev.index.tz_localize(None)
        except Exception:
            return pd.Series(dtype=float)  # falha não fica em cache
        _EVENTOS_YAHOO[ticker] = ev
    return _EVENTOS_YAHOO[ticker]


_IPCA_BCB = globals().get("_IPCA_BCB", {})  # cache da sessão: IPCA mensal do Banco Central (válido no dia)


def ipca_mensal_bcb():
    """IPCA do mês (SGS 433, Banco Central) em fração, de 2014 até hoje, baixado
    no máximo uma vez por dia na sessão. None se a consulta falhar."""
    import json as _json
    hoje = pd.Timestamp.today().normalize()
    if _IPCA_BCB.get("data") == hoje and _IPCA_BCB.get("serie") is not None:
        return _IPCA_BCB["serie"]
    url = ("https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json"
           f"&dataInicial=01/01/2014&dataFinal={hoje:%d/%m/%Y}")
    for tentativa in range(3):
        try:
            with urllib.request.urlopen(url, timeout=45) as resp:
                dados = _json.loads(resp.read().decode("utf-8"))
            serie = pd.Series([float(d["valor"]) / 100 for d in dados],
                              index=pd.to_datetime([d["data"] for d in dados], dayfirst=True)).sort_index()
            _IPCA_BCB.update(data=hoje, serie=serie)
            return serie
        except Exception:
            _time.sleep(2 * (2 ** tentativa))
    return None


def _segmento_de_papel(seg):
    return (any(c in seg for c in _SEGMENTOS_PAPEL)
            or any(re.search(rf"\b{sigla}\b", seg) for sigla in _SIGLAS_PAPEL))


_IMOVEIS_PL = globals().get("_IMOVEIS_PL", {})  # cache da sessão: Imóveis/PL por FII


def imoveis_pl_fundamentus(ticker):
    """'Imóveis/PL do FII' da página de detalhes do Fundamentus (ex.: 3,2% ->
    0.032), ou None. Mesmo código do painel (fontes/fundamentus.py)."""
    papel = ticker.upper().replace(".SA", "")
    if papel in _IMOVEIS_PL:
        return _IMOVEIS_PL[papel]
    try:
        req = urllib.request.Request(f"https://www.fundamentus.com.br/detalhes.php?papel={papel}",
                                     headers={'User-Agent': _USER_AGENT_FUNDAMENTUS})
        with urllib.request.urlopen(req, timeout=20) as resp:
            html = resp.read().decode("latin-1")
    except Exception:
        return None  # falha não fica em cache
    texto = re.sub(r"&nbsp;|\xa0", " ", re.sub(r"<[^>]+>", "\n", html))
    partes = [p.strip() for p in texto.split("\n") if p.strip()]
    valor = None
    for i, parte in enumerate(partes):
        if re.sub(r"\s+", " ", parte).lower().startswith("imóveis/pl"):
            for seguinte in partes[i + 1:i + 4]:
                m = re.fullmatch(r"(-?[\d.]+,?\d*)\s*%", seguinte)
                if m:
                    numero = _parse_numero_br_fundamentus(m.group(1))
                    valor = numero / 100 if numero is not None else None
                    break
            break
    _IMOVEIS_PL[papel] = valor
    _time.sleep(0.3)
    return valor


def classe_fii_teste(ticker):
    """'papel', 'tijolo', 'hibrido' ou None — mesma regra do painel
    (classificar_fii): pelo SEGMENTO do Fundamentus; com segmento genérico
    ("Multicategoria", "Outros", vazio), pela participação dos imóveis no
    patrimônio (Imóveis/PL: abaixo de 20%, papel; de 20% a 50%, híbrido;
    acima, tijolo — casos RECR11 e MXRF11); sem esse dado, pela quantidade
    de imóveis."""
    fund = buscar_fundamentos_fii_fundamentus(ticker) or {}
    seg = _sem_acento(fund.get("segmento") or "").strip()
    if seg and seg not in ("outros", "nan", "none", "-") and "multicategoria" not in seg:
        if "hibrido" in seg:
            return "hibrido"
        if _segmento_de_papel(seg):
            return "papel"
        # segmento de tijolo conferido pelo Imóveis/PL (05/11/2026, caso MXRF11: "Logística", mas quase só CRIs)
        imoveis_pl = imoveis_pl_fundamentus(ticker)
        if imoveis_pl is not None and imoveis_pl < 0.50:
            return "papel" if imoveis_pl < 0.20 else "hibrido"
        return "tijolo"
    if not fund:
        return None
    imoveis_pl = imoveis_pl_fundamentus(ticker)
    if imoveis_pl is not None:
        return "papel" if imoveis_pl < 0.20 else ("hibrido" if imoveis_pl < 0.50 else "tijolo")
    qtd = fund.get("qtd_imoveis")
    if qtd in (None, ""):
        return None
    return "papel" if qtd in (0, 0.0) else "tijolo"


_CVM_BASE = globals().get("_CVM_BASE", {})  # cache da sessão: base mensal dos informes da CVM


def _base_cvm(mostrar=True):
    """Baixa (uma vez por sessão) os informes mensais de FIIs da CVM, de 2016
    até o ano atual, e monta a base mensal por fundo."""
    if "base" not in _CVM_BASE:
        anos = list(range(2016, pd.Timestamp.today().year + 1))
        tabelas, status = [], []
        for ano in anos:
            if mostrar:
                print(f"  baixando informes da CVM de {ano}...", end="\r")
            t = baixar_informes([ano])
            status.append((ano, "ok" if t else "falhou"))
            tabelas += t
        _CVM_BASE["tabelas"], _CVM_BASE["status"] = tabelas, status
        _CVM_BASE["base"] = montar_base_fii(tabelas) if tabelas else pd.DataFrame()
    return _CVM_BASE["base"]


fonte_rendimentos = widgets.Dropdown(
    options=[("Automática — Fundamentus → CVM → Yahoo, conferida entre as fontes", "auto"),
             ("Fundamentus — proventos pagos", "fundamentus"),
             ("CVM — dividend yield × valor patrimonial", "cvm_dy"),
             ("CVM — rendimentos a distribuir ÷ cotas", "cvm_passivo"),
             ("Yahoo Finance", "yahoo")],
    value="auto", description="Fonte dos rendimentos:", style={'description_width': 'initial'},
    layout=widgets.Layout(width='520px'))
botao_conferir_cvm = widgets.Button(description="🔎 Conferir as fontes de rendimentos", button_style='info',
                                    icon='check', layout=widgets.Layout(width='340px'))
saida_conferir_cvm = widgets.Output()


def _conferir_cvm_click(b):
    with saida_conferir_cvm:
        clear_output()
        print("Baixando os informes mensais de FIIs da CVM (2016 em diante, ~1 arquivo por ano)...")
        try:
            base = _base_cvm()
        except ValueError as e:
            clear_output()
            imprimir_texto(f"⚠️ A estrutura dos arquivos da CVM é diferente da esperada: {e}")
            base = pd.DataFrame()
        clear_output()
        print("ANOS:", ", ".join(f"{a} {s}" for a, s in _CVM_BASE.get("status", [])))
        tabs = _CVM_BASE.get("tabelas", [])
        if tabs:
            print("\nCOLUNAS ENCONTRADAS (primeiro e último ano baixados):")
            for rotulo, t in (("primeiro", tabs[0]), ("último", tabs[-1])):
                for nome, df in t.items():
                    imprimir_texto(f"[{rotulo} · {nome}] " + " | ".join(df.columns), prefixo="  ")
        if base.empty:
            print("\n⚠️ A base da CVM ficou vazia — me envie esta saída para ajuste.")
            return
        print(f"\nBase montada: {base['ticker'].nunique()} FIIs com ISIN de cota, de "
              f"{base['data'].min():%m/%Y} a {base['data'].max():%m/%Y}.")
        print(f"Dividend yield do mês: mediana {base['dy_mes'].median()*100:.2f}% · "
              f"rendimento por cota (dy × VP): mediana R$ {base['rend_cota_dy'].median():.2f}")
        linhas = []
        fiis = [t for t, tp in _tipo_por_ticker_secao2().items() if tp == 'fii']
        for n, ticker in enumerate(fiis, 1):
            print(f"  comparando {ticker} ({n}/{len(fiis)})...", end="\r")
            fund = proventos_fii_fundamentus(ticker)
            yahoo = _dividendos_brutos_cache(ticker)
            dy, pas = serie_rendimentos(base, ticker, "dy"), serie_rendimentos(base, ticker, "passivo")
            fmt = lambda c: "" if c["erro_mediano_12m"] is None else f"{c['erro_mediano_12m']*100:.1f}%"
            meses = lambda s: s.groupby(s.index.to_period("M")).size().size if len(s) else 0
            linhas.append({
                "FII": ticker,
                "Fundamentus desde": f"{fund.index.min():%m/%Y}" if len(fund) else "—",
                "Meses c/ rendimento (Fundamentus)": meses(fund),
                "Meses c/ rendimento (Yahoo)": meses(yahoo),
                "CVM desde": f"{dy.index.min():%m/%Y}" if len(dy) else "—",
                "Erro 12m: Yahoo × Fundamentus": fmt(comparar_com_referencia(yahoo, fund)),
                "Erro 12m: CVM (dy × VP) × Fundamentus": fmt(comparar_com_referencia(dy, fund)),
                "Erro 12m: CVM (a distribuir) × Fundamentus": fmt(comparar_com_referencia(pas, fund))})
        print(" " * 60, end="\r")
        print("\nCONFERÊNCIA ENTRE AS FONTES: soma de 12 meses de rendimentos por cota, comparada com os")
        print("proventos pagos do Fundamentus, nos meses em que as fontes existem (erro relativo mediano;")
        print("quanto menor, mais as fontes concordam). Meses com rendimento mostram as lacunas de cada fonte:")
        display(pd.DataFrame(linhas).sort_values("FII").reset_index(drop=True))


botao_conferir_cvm.on_click(_conferir_cvm_click)

_TESOURO_DIA = globals().get("_TESOURO_DIA", {})  # cache da sessão: juro real do Tesouro IPCA+, válido só no dia em que foi baixado


def juro_real_do_dia():
    """Juro real do Tesouro IPCA+ (arquivo de ~14 MB), baixado no máximo uma
    vez por dia: no mesmo dia, reaproveita o da sessão; no dia seguinte, ou
    depois do botão 'Atualizar dados baixados', baixa de novo.
    Devolve (série ou None, origem): origem 'sessao', 'baixado' ou 'falhou'."""
    hoje = pd.Timestamp.today().normalize()
    if _TESOURO_DIA.get("data") == hoje and _TESOURO_DIA.get("serie") is not None:
        print("♻️ Juro real do Tesouro IPCA+: reaproveitado da sessão (baixado hoje).")
        return _TESOURO_DIA["serie"], "sessao"
    print("Baixando o juro real do Tesouro IPCA+ (arquivo oficial do Tesouro Transparente, ~14 MB)...")
    serie = baixar_juro_real_tesouro_ipca()
    if serie is None:
        return None, "falhou"
    _TESOURO_DIA.update(data=hoje, serie=serie)
    return serie, "baixado"


botao_atualizar_dados = widgets.Button(description="🔄 Atualizar dados baixados", icon="refresh",
                                       layout=widgets.Layout(width='340px'))
saida_atualizar_dados = widgets.Output()


def _atualizar_dados_click(b):
    """Descarta o que a sessão guardou (Tesouro, CDI, proventos do Fundamentus
    e informes da CVM): o próximo teste baixa tudo de novo."""
    with saida_atualizar_dados:
        clear_output()
        _TESOURO_DIA.clear()
        _CDI_JANELAS.clear()
        _PROVENTOS_FUNDAMENTUS.clear()
        _CVM_BASE.clear()
        _EVENTOS_YAHOO.clear()
        _IPCA_BCB.clear()
        _IMOVEIS_PL.clear()
        print("Dados descartados: o próximo teste baixa de novo o Tesouro IPCA+, o CDI, o IPCA, os proventos e "
              "os informes da CVM.")


botao_atualizar_dados.on_click(_atualizar_dados_click)

botao_oportunidades = widgets.Button(description="🔍 Testar critérios de preço atrativo", button_style='warning',
                                     icon='search', layout=widgets.Layout(width='340px'))
saida_oportunidades = widgets.Output()

_CRITERIOS = {
    "criterio_yield": "quando o yield estava 15% ou mais acima da média do próprio FII nos 24 meses anteriores, "
                      "com rendimentos estáveis",
    "criterio_premio": "quando o prêmio sobre o Tesouro IPCA+ estava 1 ponto percentual ou mais acima da média do "
                       "próprio FII nos 24 meses anteriores, com rendimentos estáveis",
    # caminho 4 (03/11/2026): P/VP com o valor patrimonial da CVM, critérios fixados antes de rodar
    "criterio_pvp": "quando o P/VP estava abaixo de 0,95, com rendimentos estáveis",
    "criterio_pvp_historico": "quando o P/VP estava 10% ou mais abaixo da média do próprio FII nos 24 meses "
                              "anteriores, com rendimentos estáveis",
    "criterio_yield_pvp": "quando o caminho 2 estava atendido E o P/VP estava abaixo de 0,95",
    # Etapa C (04/11/2026): nos FIIs de papel, renda estável OU estável descontada a inflação
    "criterio_yield_real": "caminho 2 aceitando, nos FIIs de papel, renda estável descontada a inflação",
    "criterio_pvp_real": "caminho 4a aceitando, nos FIIs de papel, renda estável descontada a inflação",
    "criterio_yield_pvp_real": "caminho 4c aceitando, nos FIIs de papel, renda estável descontada a inflação",
}
_ROTULOS_CRITERIOS = {
    "criterio_yield": "Caminho 2 — yield vs. próprio histórico",
    "criterio_premio": "Caminho 3 — prêmio sobre o Tesouro IPCA+",
    "criterio_pvp": "Caminho 4a — P/VP abaixo de 0,95",
    "criterio_pvp_historico": "Caminho 4b — P/VP abaixo da própria média",
    "criterio_yield_pvp": "Caminho 4c — caminho 2 com P/VP abaixo de 0,95",
    "criterio_yield_real": "Caminho 2R — caminho 2 com renda real (papel)",
    "criterio_pvp_real": "Caminho 4aR — caminho 4a com renda real (papel)",
    "criterio_yield_pvp_real": "Caminho 4cR — caminho 4c com renda real (papel)",
}
_PARES_ETAPA_C = (("criterio_yield", "criterio_yield_real"), ("criterio_pvp", "criterio_pvp_real"),
                  ("criterio_yield_pvp", "criterio_yield_pvp_real"))


def _bloco_etapa_c(casos):
    """Etapa C: os casos que a regra da renda descontada a inflação ACRESCENTA
    (variante real atendida, original não) e o teste só nos FIIs de papel."""
    resumir_sem_gancho = globals().get("_resumir_criterio_original", resumir_criterio)
    papel = casos[casos["classe"] == "papel"]
    print("\n" + "=" * 78)
    print(f"🧪 ETAPA C — RENDA DESCONTADA A INFLAÇÃO ({papel['ticker'].nunique()} FIIs de papel: "
          + ", ".join(sorted(papel["ticker"].unique())) + ")")
    print("=" * 78)
    if papel.empty:
        print("   Nenhum FII de papel no teste.")
        return
    so_real = (papel["estavel_real"] == 1) & ~papel["estavel"].astype(bool)
    imprimir_texto(f"Meses em que a renda nominal caiu mais de 10% mas a renda acima da inflação ficou estável: "
                   f"{int(so_real.sum())} de {len(papel)}.", prefixo="   ")
    print("  ⏳ calculando...", end="\r")
    linhas = []
    for original, real in _PARES_ETAPA_C:
        aval = casos[casos[original].notna() & casos[real].notna()].copy()
        aval["_novo"] = ((aval[real] == 1) & (aval[original] == 0)).astype(float)
        for rotulo, base, coluna in (
                (_ROTULOS_CRITERIOS[original].split(" — ")[0] + " (só FIIs de papel)", papel, original),
                (_ROTULOS_CRITERIOS[real].split(" — ")[0] + " (só FIIs de papel)", papel, real),
                ("  ↳ só os casos NOVOS que a regra acrescenta", aval[aval["classe"] == "papel"], "_novo")):
            r = resumir_sem_gancho(base, coluna)
            if not r:
                continue
            ic = r["ic90_diferenca"]
            linhas.append({
                "Critério": rotulo, "FIIs": r["fiis"], "Casos": r["casos"],
                "Casos com o critério": r["casos_com_criterio"], "FIIs com o critério": r["fiis_com_criterio"],
                "Superou o CDI (com o critério)": (f"{r['taxa_com_criterio']*100:.0f}%"
                                                   if r["casos_com_criterio"] else "—"),
                "Superou o CDI (todos os casos)": f"{r['taxa_base']*100:.0f}%",
                "Diferença": (f"{(r['taxa_com_criterio']-r['taxa_base'])*100:+.0f} p.p."
                              if r["casos_com_criterio"] else "—"),
                "IC 90% da diferença": f"{ic[0]*100:+.0f} a {ic[1]*100:+.0f} p.p." if ic else ""})
    print(" " * 30, end="\r")
    display(pd.DataFrame(linhas))
    print("   Leitura: a regra nova vale a pena se os casos NOVOS superarem o CDI pelo menos tanto quanto os")
    print("   casos do critério original — sinal de que a queda nominal desses fundos era inflação, não piora.")


TOLERANCIAS_ROBUSTEZ = (0.05, 0.10, 0.15)  # 10% = regra fixada; 5% e 15% só para checar


def _checagem_robustez(casos, entradas):
    """Refaz o teste com a tolerância de queda dos rendimentos em 5% e 15%
    (a regra fixada é 10%). Não escolhe a melhor: mostra se a conclusão se
    mantém. Se só valer em 10%, o resultado é frágil."""
    resumir_sem_gancho = globals().get("_resumir_criterio_original", resumir_criterio)
    print("\n" + "=" * 78)
    print("🧷 CHECAGEM DE ROBUSTEZ — 'rendimentos estáveis' com queda tolerada de 5%, 10% (regra) e 15%")
    print("=" * 78)
    por_tol = {}
    for tol in TOLERANCIAS_ROBUSTEZ:
        if abs(tol - QUEDA_MAXIMA_RENDIMENTOS) < 1e-9:
            por_tol[tol] = casos
            continue
        partes = []
        for n, (ticker, e) in enumerate(entradas.items(), 1):
            print(f"  ⏳ recalculando com queda tolerada de {tol*100:.0f}%: {ticker} ({n}/{len(entradas)})...    ",
                  end="\r")
            e = dict(e)
            classe = e.pop("classe")
            df = avaliar_fii_mes_a_mes(**e, queda_maxima=tol)
            if not df.empty:
                df["ticker"], df["classe"] = ticker, classe
                partes.append(df)
        por_tol[tol] = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()
    print(" " * 90, end="\r")
    print("  ⏳ calculando os intervalos de confiança (bootstrap), cerca de 1 minuto...", end="\r")
    linhas, leituras = [], []
    for coluna in _CRITERIOS:
        linha = {"Critério": _ROTULOS_CRITERIOS[coluna]}
        positivos, com_ic, poucos = 0, 0, False
        for tol in TOLERANCIAS_ROBUSTEZ:
            r = resumir_sem_gancho(por_tol[tol], coluna) if not por_tol[tol].empty else None
            rot = f"Queda até {tol*100:.0f}%" + (" (regra)" if abs(tol - QUEDA_MAXIMA_RENDIMENTOS) < 1e-9 else "")
            if not r or not r["casos_com_criterio"]:
                linha[rot] = "sem casos"
                continue
            ic = r["ic90_diferenca"]
            dif = (r["taxa_com_criterio"] - r["taxa_base"]) * 100
            linha[rot] = (f"{r['taxa_com_criterio']*100:.0f}% vs {r['taxa_base']*100:.0f}% ({dif:+.0f} p.p.; "
                          f"{r['casos_com_criterio']} casos" + (f"; IC {ic[0]*100:+.0f} a {ic[1]*100:+.0f}" if ic else "")
                          + ")")
            poucos = poucos or r["casos_com_criterio"] < 20 or r["fiis_com_criterio"] < 5
            if ic:
                com_ic += 1
                positivos += ic[0] > 0
        linha["Leitura"] = ("— sem casos" if not com_ic else
                            "poucos casos (menos de 20 ou de 5 FIIs) — não conclusivo" if poucos else
                            "✅ robusto: vantagem nas 3 tolerâncias" if positivos == 3 else
                            "❌ sem vantagem em nenhuma" if positivos == 0 else
                            f"⚠️ frágil: vantagem em {positivos} de {com_ic}")
        linhas.append(linha)
    print(" " * 90, end="\r")
    display(pd.DataFrame(linhas))
    print("   Leitura: 'robusto' = o intervalo de confiança fica acima de zero com 5%, 10% e 15%. A regra")
    print("   continua sendo a de 10%, fixada antes do teste; esta tabela só mostra se ela depende desse número.")


def _rodar_oportunidades_click(b):
    with saida_oportunidades:
        clear_output()
        fiis = [t for t, tp in _tipo_por_ticker_secao2().items() if tp == 'fii']
        if not fiis:
            print("⚠️ Nenhum FII configurado na Seção 2.")
            return
        juro, origem_juro = juro_real_do_dia()
        if juro is None:
            print("⚠️ Não foi possível baixar as taxas do Tesouro — o caminho 3 (prêmio) fica de fora nesta rodada.")
        # A CVM entra sempre: é a fonte oficial que confere as outras (baixada uma vez por sessão).
        print("Carregando os informes mensais da CVM (rendimentos por cota)...")
        try:
            base_cvm = _base_cvm(mostrar=False)
        except ValueError:
            base_cvm = pd.DataFrame()
        if base_cvm.empty:
            base_cvm = None
            if fonte_rendimentos.value.startswith("cvm"):
                print("⚠️ Informes da CVM indisponíveis — rode a conferência da CVM para ver o motivo.")
                return
        print(f"Baixando ~10 anos de preços de {len(fiis)} FIIs e o CDI do Banco Central...")
        hoje = pd.Timestamp.today().normalize()
        cdi = buscar_cdi_bcb(hoje - pd.DateOffset(years=11), hoje)
        if cdi is None:
            imprimir_texto("⚠️ Teste interrompido: o CDI do Banco Central não pôde ser baixado. Uma taxa fixa "
                           "distorceria a comparação ao longo de 10 anos, com juros de cerca de 2% ao ano em 2021 a "
                           "14% em 2025. Tente de novo em alguns minutos.")
            return
        ipca = ipca_mensal_bcb()
        casos, hoje_por_fii, ignorados, cobertura = [], [], [], []
        conferencias, entradas = {}, {}  # entradas: para refazer a conta na checagem de robustez
        for n, ticker in enumerate(fiis, 1):
            print(f"  [{n}/{len(fiis)}] {ticker}...", end="\r")
            bruto = _baixar_fechamento_cache(ticker, periodo='10y')
            if bruto is None or bruto.empty:
                ignorados.append((ticker, "sem dados no Yahoo"))
                continue
            precos, _, _ = limpar_outliers_precos(bruto)
            indice = indice_cdi_acumulado(precos.index, cdi)
            fontes_fii = {
                "fundamentus": proventos_fii_fundamentus(ticker),
                "cvm": (serie_rendimentos(base_cvm, ticker, "passivo" if fonte_rendimentos.value == "cvm_passivo"
                                          else "dy") if base_cvm is not None else None),
                "yahoo": _dividendos_brutos_cache(ticker),
            }
            eventos = eventos_yahoo(ticker)
            if fonte_rendimentos.value == "auto":
                conf = conferir_fontes(fontes_fii, eventos=eventos)
            else:  # fonte escolhida na lista: só ela alimenta o teste, as outras só conferem
                escolhida = "cvm" if fonte_rendimentos.value.startswith("cvm") else fonte_rendimentos.value
                conf = conferir_fontes(fontes_fii, ordem=(escolhida,) + tuple(f for f in ORDEM_PADRAO
                                                                               if f != escolhida),
                                       eventos=eventos)
                if conf["usada"] != escolhida:
                    conf = conferir_fontes({escolhida: None})  # a fonte escolhida não respondeu
            conferencias[ticker] = conf
            divs_fii = conf["serie"] if conf["usada"] else pd.Series(dtype=float)
            # valor patrimonial por cota (CVM), corrigido pelos desdobramentos/grupamentos do Yahoo
            vp_fii = serie_valor_patrimonial(base_cvm, ticker) if base_cvm is not None else None
            if vp_fii is not None and len(vp_fii):
                vp_fii, _ = ajustar_por_desdobramentos(vp_fii, eventos)
            classe = classe_fii_teste(ticker)
            df = avaliar_fii_mes_a_mes(precos, divs_fii, indice, juro, valor_patrimonial=vp_fii,
                                       ipca_mensal=ipca, papel=(classe == "papel"))
            cobertura.append({**({"FII": ticker, "Classe": classe or "?"}), **resumo_conferencia(conf), **{
                "Preços desde": f"{precos.index[0]:%m/%Y}",
                "Rendimentos desde": f"{divs_fii.index[0]:%m/%Y}" if len(divs_fii) else "—",
                "P/VP desde": (f"{df.loc[df['pvp'].notna(), 'data'].min():%m/%Y}"
                               if not df.empty and df['pvp'].notna().any() else "—"),
                "Primeiro mês avaliado": (f"{df.loc[df['criterio_yield'].notna() & df['superou_cdi'].notna(), 'data'].min():%m/%Y}"
                                          if not df.empty and (df['criterio_yield'].notna() & df['superou_cdi'].notna()).any() else "—"),
                "Casos": int((df['criterio_yield'].notna() & df['superou_cdi'].notna()).sum()) if not df.empty else 0,
                "Meses com lacuna de rendimentos": int(df['yield'].isna().sum()) if not df.empty else 0,
                "Casos excluídos por lacuna à frente": int(df['lacuna_futuro'].sum()) if not df.empty else 0}})
            if df.empty:
                ignorados.append((ticker, "histórico curto ou sem rendimentos"))
                continue
            df["ticker"] = ticker
            df["classe"] = classe or "?"
            casos.append(df)
            entradas[ticker] = dict(precos=precos, dividendos=divs_fii, indice_cdi=indice, juro_real=juro,
                                    valor_patrimonial=vp_fii, ipca_mensal=ipca, papel=(classe == "papel"),
                                    classe=classe or "?")
            hoje_por_fii.append(df.iloc[-1])
        clear_output()
        if not casos:
            print("⚠️ Nenhum FII com histórico suficiente.")
            return
        casos = pd.concat(casos, ignore_index=True)
        _CASOS_FII_ULTIMO_TESTE["casos"] = casos  # a célula 13b (regimes de juros) lê daqui
        if ignorados:
            print("⚠️ Fora do teste: " + "; ".join(f"{t} ({m})" for t, m in ignorados) + "\n")
        imprimir_texto(descrever_cdi(cdi))
        imprimir_texto("Fonte dos rendimentos: " + dict((v, k) for k, v in fonte_rendimentos.options)[fonte_rendimentos.value]
                       + {"auto": " (a primeira que respondeu para cada FII; veja as colunas de fontes no quadro de "
                                  "cobertura).",
                          "fundamentus": " (página pública de proventos de cada FII, fundamentus.com.br).",
                          "cvm_dy": " (Informes Mensais Estruturados de FIIs, Portal de Dados Abertos da CVM).",
                          "cvm_passivo": " (Informes Mensais Estruturados de FIIs, Portal de Dados Abertos da CVM).",
                          "yahoo": "."}[fonte_rendimentos.value])
        usadas = {}
        for t, cf in conferencias.items():
            if cf["usada"]:
                usadas.setdefault(NOMES_FONTES[cf["usada"]], []).append(t)
        selos = pd.Series([cf["selo"] for cf in conferencias.values()]).value_counts()
        imprimir_texto("Conferência dos rendimentos entre as fontes: " + ", ".join(
            f"{SELOS[k]}: {selos.get(k, 0)}" for k in ("confirmado", "divergente", "fonte_unica", "sem_dados")
            if selos.get(k, 0)) + " FII(s).")
        for t, cf in sorted(conferencias.items()):
            for obs in cf.get("observacoes", []):
                imprimir_texto(f"{t}: {obs}.", prefixo="   🔍 ")
        if fonte_rendimentos.value == "auto" and len(usadas) > 1:
            imprimir_texto("⚠️ Fontes misturadas nesta rodada (" + "; ".join(
                f"{f}: {len(ts)} FII(s)" for f, ts in usadas.items()) + "). Para o relatório, prefira uma "
                "rodada com fonte única — escolha a fonte na lista acima.")
        # As mensagens de download somem com o clear_output() acima: a origem do
        # juro real (baixado agora, reaproveitado da sessão ou indisponível) vai aqui.
        if juro is not None:
            imprimir_texto(f"Fonte do juro real: Tesouro IPCA+ (Tesouro Transparente), de {juro.index[0]:%m/%Y} a "
                           f"{juro.index[-1]:%m/%Y}" + (" — ♻️ reaproveitado da sessão (baixado hoje)."
                                                       if origem_juro == "sessao" else " — baixado agora."))
        else:
            imprimir_texto("⚠️ Juro real do Tesouro IPCA+ indisponível nesta rodada (o arquivo do Tesouro Transparente "
                           "não pôde ser baixado): o caminho 3 (prêmio) ficou sem casos. Tente de novo mais tarde.")
        if ipca is not None:
            imprimir_texto(f"IPCA mensal: Banco Central (SGS 433), de {ipca.index[0]:%m/%Y} a {ipca.index[-1]:%m/%Y}.")
        else:
            imprimir_texto("⚠️ IPCA do Banco Central indisponível nesta rodada: os caminhos 2R, 4aR e 4cR ficaram "
                           "iguais aos originais. Tente de novo mais tarde.")
        print()

        print("=" * 78)
        print("💬 O QUE O HISTÓRICO MOSTRA")
        print("=" * 78)
        linhas_tabela = []
        for coluna, descricao in _CRITERIOS.items():
            r = resumir_criterio(casos, coluna)
            print()
            imprimir_texto(frase_resultado(r, descricao), prefixo="• ")
            if r and r["casos_com_criterio"]:
                ic = r["ic90_diferenca"]
                linhas_tabela.append({
                    "Critério": _ROTULOS_CRITERIOS[coluna],
                    "FIIs": r["fiis"], "Casos": r["casos"], "Casos com o critério": r["casos_com_criterio"],
                    "FIIs com o critério": r["fiis_com_criterio"],
                    "Superou o CDI (com o critério)": f"{r['taxa_com_criterio']*100:.0f}%",
                    "Superou o CDI (todos os casos)": f"{r['taxa_base']*100:.0f}%",
                    "Diferença": f"{(r['taxa_com_criterio']-r['taxa_base'])*100:+.0f} p.p.",
                    "IC 90% da diferença": f"{ic[0]*100:+.0f} a {ic[1]*100:+.0f} p.p." if ic else "",
                    "Excesso mediano s/ CDI (com / todos)": f"{r['excesso_mediano_com_criterio']*100:+.1f}% / "
                                                            f"{r['excesso_mediano_base']*100:+.1f}%",
                })
        if linhas_tabela:
            print("\n📋 TABELA PARA O RELATÓRIO")
            display(pd.DataFrame(linhas_tabela))

        confirmados = [t for t, cf in conferencias.items() if cf["selo"] == "confirmado"]
        so_confirmados = casos[casos["ticker"].isin(confirmados)]
        print("\n🔁 MESMO TESTE SÓ COM OS FIIs DE RENDIMENTOS CONFIRMADOS POR 2 FONTES "
              f"({so_confirmados['ticker'].nunique()} de {casos['ticker'].nunique()} FIIs):")
        if so_confirmados.empty or so_confirmados["ticker"].nunique() == casos["ticker"].nunique():
            print("   (todos os FIIs do teste foram confirmados — o resultado acima já é o dos confirmados)"
                  if not so_confirmados.empty else "   (nenhum FII confirmado nesta rodada)")
        else:
            # versão sem o "gancho" da célula 13b, que guarda só os casos do teste completo
            resumir_sem_gancho = globals().get("_resumir_criterio_original", resumir_criterio)
            for coluna, descricao in _CRITERIOS.items():
                r = resumir_sem_gancho(so_confirmados, coluna)
                rotulo = _ROTULOS_CRITERIOS[coluna].split(" — ")[0]
                if not r or not r["casos_com_criterio"]:
                    print(f"   • {rotulo}: sem casos com o critério.")
                    continue
                ic = r["ic90_diferenca"]
                print(f"   • {rotulo}: {r['taxa_com_criterio']*100:.0f}% contra {r['taxa_base']*100:.0f}% "
                      f"({(r['taxa_com_criterio']-r['taxa_base'])*100:+.0f} p.p.; {r['casos_com_criterio']} de "
                      f"{r['casos']} casos" + (f"; IC 90% {ic[0]*100:+.0f} a {ic[1]*100:+.0f} p.p." if ic else "")
                      + ")")

        _bloco_etapa_c(casos)
        _checagem_robustez(casos, entradas)

        print("\n🔎 CRITÉRIOS ATENDIDOS HOJE (para investigar — não é recomendação):")
        atual = pd.DataFrame(hoje_por_fii)
        tabela_hoje = pd.DataFrame({
            "FII": atual["ticker"],
            "Yield 12m": (atual["yield"] * 100).map(lambda v: "" if pd.isna(v) else f"{v:.1f}%"),
            "Média 24m": (atual["yield_medio_24m"] * 100).map(lambda v: "" if pd.isna(v) else f"{v:.1f}%"),
            "Prêmio s/ IPCA+": (atual["premio"] * 100).map(lambda v: "" if pd.isna(v) else f"{v:.1f} p.p."),
            "Classe": atual["classe"],
            "Rendimentos estáveis": [("" if pd.isna(y) else ("sim" if e else "não"))
                                     for e, y in zip(atual["estavel"], atual["yield"])],
            "Renda acima da inflação": (atual["renda_real_12m"] * 100).map(
                lambda v: "" if pd.isna(v) else f"{v:.1f}%"),
            "Estável descontada a inflação": atual["estavel_real"].map(
                lambda v: "" if pd.isna(v) else ("sim" if v == 1 else "não")),
            "Caminho 2": atual["criterio_yield"].map(lambda v: "✅" if v == 1 else ("" if pd.isna(v) else "—")),
            "Caminho 3": atual["criterio_premio"].map(lambda v: "✅" if v == 1 else ("" if pd.isna(v) else "—")),
            "P/VP": atual["pvp"].map(lambda v: "" if pd.isna(v) else f"{v:.2f}"),
            "P/VP médio 24m": atual["pvp_medio_24m"].map(lambda v: "" if pd.isna(v) else f"{v:.2f}"),
            **{_ROTULOS_CRITERIOS[c].split(" — ")[0]: atual[c].map(lambda v: "✅" if v == 1 else ("" if pd.isna(v) else "—"))
               for c in ("criterio_pvp", "criterio_pvp_historico", "criterio_yield_pvp",
                         "criterio_yield_real", "criterio_pvp_real", "criterio_yield_pvp_real")},
        }).sort_values("FII").reset_index(drop=True)
        display(tabela_hoje)

        print("\n🗂️ COBERTURA DOS DADOS POR FII (início do histórico e casos avaliados):")
        display(pd.DataFrame(cobertura).sort_values("FII").reset_index(drop=True))

        print("\n📌 Como o teste funciona:")
        print("   • Um caso = um FII num fim de mês. Resultado: o retorno dos 12 meses seguintes (preço + rendimentos")
        print("     recebidos, sem reinvestir) superou o CDI do mesmo período, líquido de IR (17,5%)?")
        print("   • Compara a taxa de acerto nos casos COM o critério com a taxa em TODOS os casos (taxa-base).")
        print("   • Critérios fixados antes de ver os resultados. 'Rendimentos estáveis': queda de no máximo 10% nos")
        print("     12 meses E nos últimos 3 meses (anualizados) — evita a armadilha do yield alto por corte recente.")
        print("   • O intervalo de confiança reamostra FIIs inteiros (bootstrap por grupo), porque os casos de um")
        print("     mesmo FII se sobrepõem no tempo. Se ele inclui zero, a diferença pode ser acaso.")
        print("   • Juro real: mediana das taxas dos títulos Tesouro IPCA+ com vencimento entre 5 e 15 anos.")
        print("   • P/VP (caminho 4): valor patrimonial por cota dos informes da CVM, do mês ANTERIOR (o informe sai")
        print("     cerca de um mês depois), corrigido pelos desdobramentos/grupamentos. Critérios fixados antes de")
        print("     rodar; a taxa-base de cada um é a dos casos com P/VP disponível.")
        print("   • Etapa C (2R, 4aR, 4cR), só FIIs de papel (segmento do Fundamentus): num fundo de CRI atrelado ao")
        print("     IPCA, o rendimento cai quando a inflação desacelera, sem que o fundo piore. Renda acima da")
        print("     inflação = rendimentos de 12 meses ÷ valor patrimonial (CVM) − IPCA de 12 meses com 2 meses de")
        print("     defasagem. As variantes aceitam renda estável OU renda acima da inflação estável (mesma régua de")
        print("     10%, nos 12 e nos últimos 3 meses). Regra fixada antes de rodar; nos demais FIIs = original.")
        print("   • Conferência das fontes: Fundamentus × Yahoo mês a mês (diferença de até 5%); com a CVM, pela soma")
        print("     de 12 meses (até 10%), porque o informe registra o mês de competência. Aceita 1 mês de defasagem.")
        print("     ✅ = outra fonte concorda em 80%+ de 12 ou mais meses/janelas em comum; ⚠️ = discordam; ◻️ = só")
        print("     uma fonte. 🔍 = fator constante entre as fontes num trecho (desdobramento/grupamento): como os preços")
        print("     vêm ajustados do Yahoo, os rendimentos do Fundamentus e da CVM daquele trecho são corrigidos.")
        print("     Desdobramentos/grupamentos registrados no Yahoo também corrigem a escala, se a própria série")
        print("     mostrar o salto na data do evento (o rendimento por cota cai ~10x num desdobramento de 1 para 10).")
        print("   • ~10 anos de preços; preços sem ajuste por proventos. O histórico de RENDIMENTOS do Yahoo para FIIs")
        print("     começa mais tarde e tem lacunas: só janelas de 12 meses com ao menos 10 pagamentos contam, e a média")
        print("     de 24 meses exige 24 meses seguidos válidos. Veja o período efetivo no quadro de cobertura.")
        print("\n✅ Teste concluído. Agora pode rodar a célula 13b (regimes de juros).")


botao_oportunidades.on_click(_rodar_oportunidades_click)
display(widgets.HTML("<h3>13. Teste de critérios de preço atrativo (FIIs)</h3>"))
display(widgets.HTML("<b>1. Conferir as fontes de rendimentos</b> (Fundamentus, CVM e Yahoo, lado a lado)"))
display(widgets.VBox([botao_conferir_cvm, saida_conferir_cvm]))
display(widgets.HTML("<b>2. Rodar o teste</b>"))
display(widgets.VBox([fonte_rendimentos, botao_oportunidades, saida_oportunidades]))
display(widgets.HTML("<b>Dados guardados na sessão</b> — o Tesouro IPCA+ vale só no dia em que foi baixado; o CDI "
                     "dos anos anteriores fica guardado e o do ano corrente é atualizado a cada dia. Rodar esta célula "
                     "de novo ou clicar outra vez no teste reaproveita o que já foi baixado. Para forçar "
                     "um novo download de tudo:"))
display(widgets.VBox([botao_atualizar_dados, saida_atualizar_dados]))


In [ ]:
#@title 🔍 13b. Preço atrativo (FIIs) por regime de juros
# Cole esta célula logo DEPOIS da célula 13 e rode-a uma vez.
# Depois: (1) clique em "🔍 Testar critérios de preço atrativo" (célula 13), como sempre;
#         (2) clique em "📊 Separar por regime de juros" (abaixo).
# Não altera o teste original: só guarda os casos que ele avaliou e os reparte
# pelo CDI médio dos 12 meses em que o resultado de cada caso foi medido.

FAIXAS_REGIME = [  # CDI bruto ao ano nos 12 meses do resultado
    ("Juros baixos (CDI até 6% a.a.)", 0.00, 0.06),
    ("Juros intermediários (6% a 11% a.a.)", 0.06, 0.11),
    ("Juros altos (acima de 11% a.a.)", 0.11, 10.0),
]

# Guarda os casos que a célula 13 avaliou, sem mudar o cálculo dela.
if "resumir_criterio" not in globals():
    raise RuntimeError("Rode antes as células 1 a 13 (menu Ambiente de execução → Executar células anteriores) "
                       "e depois rode esta célula de novo.")
if "_resumir_criterio_original" not in globals():
    _resumir_criterio_original = resumir_criterio
# a célula 13 já guarda os casos do último teste: a ordem de execução das duas células não importa mais
_CASOS_FII_ULTIMO_TESTE = globals().get("_CASOS_FII_ULTIMO_TESTE", {"casos": None})

def resumir_criterio(casos, coluna, *args, **kwargs):
    _CASOS_FII_ULTIMO_TESTE["casos"] = casos
    return _resumir_criterio_original(casos, coluna, *args, **kwargs)


def _tabela_por_regime(casos):
    df = casos[casos["superou_cdi"].notna()].copy()
    if df.empty or (df["cdi_liquido_12m"] == 0).all():
        print("⚠️ Sem a série do CDI do Banco Central no teste: não dá para separar por regime.")
        return None
    df["cdi_bruto_12m"] = df["cdi_liquido_12m"] / (1 - IR_CDI_12M)
    linhas = []
    for nome, lo, hi in FAIXAS_REGIME:
        sub = df[(df["cdi_bruto_12m"] > lo) & (df["cdi_bruto_12m"] <= hi)]
        if sub.empty:
            continue
        for coluna in _CRITERIOS:
            r = _resumir_criterio_original(sub, coluna)
            criterio = globals().get("_ROTULOS_CRITERIOS", {}).get(coluna, coluna)
            if not r:
                continue
            ic = r["ic90_diferenca"]
            com = r["taxa_com_criterio"]
            if not r["casos_com_criterio"]:
                leitura = "critério não ocorreu"
            elif r["casos_com_criterio"] < 20 or r["fiis_com_criterio"] < 5:
                leitura = "poucos casos — não conclusivo"
            elif ic and ic[0] > 0:
                leitura = "vantagem (IC acima de zero)"
            elif ic and ic[1] < 0:
                leitura = "desvantagem (IC abaixo de zero)"
            else:
                leitura = "inconclusivo (IC inclui zero)"
            linhas.append({
                "Regime de juros": nome,
                "Período das decisões": f"{r['periodo'][0]:%m/%Y} a {r['periodo'][1]:%m/%Y}",
                "Critério": criterio,
                "FIIs": r["fiis"], "Casos": r["casos"],
                "Casos com o critério": r["casos_com_criterio"],
                "Superou o CDI (com o critério)": "" if com is None else f"{com*100:.0f}%",
                "Superou o CDI (todos os casos)": f"{r['taxa_base']*100:.0f}%",
                "Diferença": "" if com is None else f"{(com - r['taxa_base'])*100:+.0f} p.p.",
                "IC 90% da diferença": f"{ic[0]*100:+.0f} a {ic[1]*100:+.0f} p.p." if ic else "",
                "Leitura": leitura,
            })
    return pd.DataFrame(linhas)


saida_regime = widgets.Output()
botao_regime = widgets.Button(description="📊 Separar por regime de juros", button_style="info",
                              layout=widgets.Layout(width="320px"))

def _rodar_regime_click(b):
    with saida_regime:
        clear_output()
        casos = _CASOS_FII_ULTIMO_TESTE["casos"]
        if casos is None:
            print("⚠️ Rode primeiro o teste da célula 13 (botão 🔍 Testar critérios de preço atrativo).")
            return
        print(f"⏳ Separando os {len(casos)} meses avaliados por regime de juros e recalculando os intervalos de "
              "confiança de cada critério (bootstrap). Leva de 1 a 3 minutos — a tabela aparece aqui ao terminar.")
        tabela = _tabela_por_regime(casos)
        clear_output()
        if tabela is None or tabela.empty:
            print("⚠️ Nenhum caso com o CDI do período para separar por regime.")
            return
        print("📋 PREÇO ATRATIVO (FIIs) POR REGIME DE JUROS — TABELA PARA O RELATÓRIO")
        print("Regime = CDI médio (bruto, ao ano) nos 12 meses em que o resultado de cada caso foi medido.")
        print("A taxa-base é recalculada dentro de cada regime; o IC reamostra FIIs inteiros, como na célula 13.")
        display(tabela)
        print("\nLeitura: com menos casos em cada regime, os intervalos ficam mais largos. 'Inconclusivo' não é")
        print("resultado negativo — significa que, nesse regime, não dá para separar o efeito do critério do acaso.")

botao_regime.on_click(_rodar_regime_click)
display(widgets.HTML("<h3>13b. Preço atrativo por regime de juros</h3>"))
display(widgets.VBox([botao_regime, saida_regime]))

## 📏 14. Regularidade dos dividendos das ações

Testa o selo **"Os dividendos são regulares?"** que aparece nos cards de ações do painel. Ele é calculado com o DY de cada um dos 5 anos fechados (proventos do ano ÷ preço médio do ano):
- **Regular:** pagou nos 5 anos e o pior ano rendeu pelo menos metade da mediana;
- **Oscilante:** pagou nos 5, mas algum ano rendeu menos da metade;
- **Irregular:** ficou pelo menos 1 ano sem pagar.

**Pergunta (fixada antes de rodar, 04/11/2026):** em cada fim de ano, com o selo calculado só com os 5 anos até ali, a renda **se manteve no ano seguinte** (DY do ano seguinte de pelo menos metade da mediana dos 5 anos, e maior que zero) mais vezes nas ações "Regulares" do que nas demais? O selo é útil se a diferença for positiva, com o intervalo de confiança acima de zero.

Também compara três previsões do DY do ano seguinte: a **mediana** dos 5 anos, a **média** e o **último ano**. Se a mediana errar menos que a média, a escolha do painel (olhar a mediana, por causa dos anos extraordinários) fica confirmada.


In [ ]:
#@title 📏 14. Teste da regularidade dos dividendos (ações)
# Funções e constantes iguais às do pacote carteira_analise (o mesmo código do painel),
# importadas do pacote em vez de copiadas aqui (Fase 1, 11/2026).
from carteira_analise.fundamentalistas import (
    ANOS_REGULARIDADE, FRACAO_ANO_FRACO, FATOR_ANO_EXTRAORDINARIO, _sem_fuso_serie, dy_por_ano,
    regularidade_dividendos, precisa_conferir, texto_regularidade,
)


ACOES_AMPLIACAO = ("PETR4.SA VALE3.SA ITUB4.SA BBDC4.SA BBAS3.SA ITSA4.SA ABEV3.SA B3SA3.SA WEGE3.SA TAEE11.SA "
                   "CMIG4.SA CPLE6.SA EGIE3.SA SAPR11.SA SBSP3.SA VIVT3.SA TIMS3.SA KLBN11.SA SUZB3.SA GGBR4.SA "
                   "CSNA3.SA USIM5.SA BBSE3.SA CXSE3.SA PSSA3.SA RENT3.SA LREN3.SA MGLU3.SA RADL3.SA BRFS3.SA "
                   "MRVE3.SA CYRE3.SA EZTC3.SA GOAU4.SA CSMG3.SA EMBR3.SA HYPE3.SA POMO4.SA ABCB4.SA").split()
ANOS_DE_CORTE = range(2021, 2025)  # selo com os 5 anos até Y; resultado no ano Y+1 (10 anos de preços)

ampliar_amostra = widgets.Checkbox(value=True, description="Incluir também ~40 ações grandes do Ibovespa "
                                   "(pagadoras e não pagadoras) para ampliar a amostra",
                                   layout=widgets.Layout(width='720px'))
botao_regularidade = widgets.Button(description="📏 Testar a regularidade dos dividendos", button_style='warning',
                                    icon='search', layout=widgets.Layout(width='340px'))
saida_regularidade = widgets.Output()


def _casos_regularidade(tickers):
    casos, sem_dados = [], []
    for n, t in enumerate(tickers, 1):
        print(f"  ⏳ {t} ({n}/{len(tickers)})...        ", end="\r")
        try:
            precos = _baixar_fechamento_cache(t, periodo='10y')
            divs = _dividendos_brutos_cache(t)
        except Exception:
            precos, divs = pd.Series(dtype=float), pd.Series(dtype=float)
        if precos is None or precos.empty or divs is None or len(divs) == 0:
            sem_dados.append(t)
            continue
        precos = _sem_fuso_serie(precos)
        divs = _sem_fuso_serie(divs)
        for ano in ANOS_DE_CORTE:
            ref = pd.Timestamp(f"{ano + 1}-01-01")
            reg = regularidade_dividendos(divs[divs.index < ref], precos[precos.index < ref], data_ref=ref)
            proximo = dy_por_ano(divs, precos, [ano + 1])[ano + 1]
            if reg is None or proximo is None:
                continue
            ultimo = reg["anos"][-1]["dy"]
            casos.append({"ticker": t, "ano": ano, "selo": reg["selo"], "mediana": reg["mediana"],
                          "media": reg["media"], "ultimo": ultimo, "dy_seguinte": proximo,
                          "extraordinario": bool(reg["extraordinarios"]),
                          "manteve": bool(proximo > 0 and proximo >= FRACAO_ANO_FRACO * reg["mediana"])})
    print(" " * 60, end="\r")
    return pd.DataFrame(casos), sem_dados


def _ic_diferenca(df, n=2000, semente=42):
    """IC 90% da diferença (taxa de 'manteve' nas Regulares − nas demais), reamostrando ações inteiras."""
    rng = np.random.default_rng(semente)
    por = df.assign(reg=df["selo"].eq("regular").astype(float), m=df["manteve"].astype(float))
    por = por.assign(m_reg=por["m"] * por["reg"], n_out=1 - por["reg"], m_out=por["m"] * (1 - por["reg"]))
    g = por.groupby("ticker")[["reg", "m_reg", "n_out", "m_out"]].sum()
    a = g.to_numpy()
    difs = []
    for _ in range(n):
        s = a[rng.choice(len(a), size=len(a), replace=True)].sum(axis=0)
        if s[0] and s[2]:
            difs.append(s[1] / s[0] - s[3] / s[2])
    return (float(np.percentile(difs, 5)), float(np.percentile(difs, 95))) if difs else None


def _rodar_regularidade_click(b):
    with saida_regularidade:
        clear_output()
        tickers = list(dict.fromkeys(parse_tickers(caixa_acoes.value)
                                     + (ACOES_AMPLIACAO if ampliar_amostra.value else [])))
        if not tickers:
            print("⚠️ Nenhuma ação configurada na Seção 2.")
            return
        print(f"⏳ Baixando 10 anos de preços e proventos de {len(tickers)} ações (fica em cache na sessão)...")
        casos, sem_dados = _casos_regularidade(tickers)
        clear_output()
        if casos.empty:
            print("⚠️ Nenhuma ação com 6 anos de histórico de preços e proventos.")
            return
        _CASOS_REGULARIDADE["casos"] = casos
        if sem_dados:
            imprimir_texto("Fora do teste (sem preços ou sem nenhum provento registrado): " + ", ".join(sem_dados) + ".")
        print("=" * 78)
        print("📏 A RENDA SE MANTEVE NO ANO SEGUINTE? (por selo, calculado só com os 5 anos até o corte)")
        print("=" * 78)
        linhas = []
        for selo in ("regular", "oscilante", "irregular"):
            sub = casos[casos["selo"] == selo]
            if sub.empty:
                continue
            linhas.append({"Selo": selo.capitalize(), "Ações": sub["ticker"].nunique(), "Casos (ação × ano)": len(sub),
                           "Renda se manteve no ano seguinte": f"{sub['manteve'].mean()*100:.0f}%",
                           "DY mediano no ano seguinte": f"{sub['dy_seguinte'].median()*100:.1f}%"})
        display(pd.DataFrame(linhas))
        reg, demais = casos[casos["selo"] == "regular"], casos[casos["selo"] != "regular"]
        if len(reg) and len(demais):
            dif = reg["manteve"].mean() - demais["manteve"].mean()
            ic = _ic_diferenca(casos)
            frase = (f"Regulares: a renda se manteve em {reg['manteve'].mean()*100:.0f}% dos {len(reg)} casos, contra "
                     f"{demais['manteve'].mean()*100:.0f}% nos {len(demais)} casos Oscilantes ou Irregulares "
                     f"(diferença de {dif*100:+.0f} p.p.)")
            if ic:
                frase += f"; IC 90% de {ic[0]*100:+.0f} a {ic[1]*100:+.0f} p.p."
                frase += (" 👉 O selo separa, com margem estatística, as rendas que se mantêm." if ic[0] > 0 else
                          " 👉 Sem diferença confiável: o selo pode não acrescentar informação." if ic[1] >= 0 else
                          " 👉 O selo funcionou ao contrário do esperado.")
            imprimir_texto(frase, prefixo="• ")

        print("\n🎯 QUAL REFERÊNCIA PREVÊ MELHOR O DY DO ANO SEGUINTE? (erro médio absoluto, em pontos percentuais)")
        linhas = []
        for rotulo, sub in (("Todas", casos), ("Com ano extraordinário nos 5 anos", casos[casos["extraordinario"]]),
                            ("Sem ano extraordinário", casos[~casos["extraordinario"]])):
            if sub.empty:
                continue
            erro = lambda col: f"{(sub[col] - sub['dy_seguinte']).abs().mean()*100:.2f}"
            linhas.append({"Casos": rotulo, "N": len(sub), "Mediana dos 5 anos": erro("mediana"),
                           "Média dos 5 anos": erro("media"), "Último ano": erro("ultimo")})
        display(pd.DataFrame(linhas))
        print("\n📌 Como o teste funciona:")
        print("   • Um caso = uma ação num fim de ano (2021 a 2024). O selo usa só os 5 anos até ali; o resultado é")
        print("     o DY do ano seguinte (2022 a 2025). DY = proventos do ano (dividendos e JCP brutos, Yahoo) ÷ preço")
        print("     médio do ano (sem ajuste por proventos).")
        print("   • 'Renda se manteve': DY do ano seguinte maior que zero e de pelo menos metade da mediana dos 5 anos.")
        print("   • O intervalo de confiança reamostra ações inteiras (bootstrap por grupo).")
        print("   • Limitação: a lista ampliada usa ações que existem hoje (viés de sobrevivência) — empresas que")
        print("     deixaram a bolsa ficam de fora. Ações sem nenhum provento no Yahoo ficam fora do teste, e anos")
        print("     sem registro no Yahoo contam como 'não pagou' (no painel, eles são conferidos no Fundamentus);")
        print("     isso pode classificar pagadoras como Irregulares e, se algo, reduz a diferença medida.")
        print("\n✅ Teste concluído.")


_CASOS_REGULARIDADE = globals().get("_CASOS_REGULARIDADE", {"casos": None})
botao_regularidade.on_click(_rodar_regularidade_click)
display(widgets.HTML("<h3>14. Regularidade dos dividendos (ações)</h3>"))
display(widgets.VBox([ampliar_amostra, botao_regularidade, saida_regularidade]))


## 🗄️ 15. Base de cotações da B3 (anos encerrados)

O Yahoo Finance não tem cotações antigas de alguns papéis (o AFHI11, por exemplo, só a partir de 01/2024), e sem o preço desses meses o painel não calcula o yield nem a média do próprio fundo. Esta célula baixa os **arquivos oficiais de cotações da B3 (COTAHIST)** dos últimos anos — do ano corrente, só os meses já encerrados — e gera um resumo mensal pequeno — fechamento do último pregão e média dos fechamentos de cada mês — de ações, units, FIIs e ETFs.

**Como usar:** clique no botão (leva alguns minutos: cada ano é um arquivo de dezenas de MB), baixe o arquivo `cotacoes_b3_mensais.csv.gz` gerado e suba no GitHub, na pasta **`dados/`** do repositório (crie a pasta se não existir). O painel passa a usar a base sozinho quando o Yahoo começar tarde. Rode de novo de tempos em tempos (por exemplo, uma vez por trimestre) para incluir os meses que encerraram.


In [ ]:
#@title 🗄️ 15. Base de cotações mensais da B3 (COTAHIST)
# Funções e constantes iguais às do pacote carteira_analise (o mesmo código do painel),
# importadas do pacote em vez de copiadas aqui (Fase 1, 11/2026).
from carteira_analise.fontes.b3_cotahist import (
    URL_COTAHIST, MERCADO_A_VISTA, CODBDI_FRACIONARIO, ler_linhas_cotahist, ler_zip_cotahist,
    _PADRAO_PAPEL, filtrar_papeis, resumo_mensal,
)

import io
import os
import zipfile


ANO_CORRENTE_B3 = pd.Timestamp.today().year         # do ano corrente, só os meses já encerrados
anos_b3 = widgets.IntRangeSlider(value=[ANO_CORRENTE_B3 - 10, ANO_CORRENTE_B3], min=2010, max=ANO_CORRENTE_B3, step=1,
                                 description="Anos:", layout=widgets.Layout(width='520px'))
botao_b3 = widgets.Button(description="🗄️ Gerar base de cotações da B3", button_style='warning', icon='database',
                          layout=widgets.Layout(width='340px'))
saida_b3 = widgets.Output()
_B3_ANOS = globals().get("_B3_ANOS", {})  # cache da sessão: resumo mensal de cada ano já baixado


def _baixar_ano_b3(ano, tentativas=3):
    if ano in _B3_ANOS:
        return _B3_ANOS[ano]
    url = URL_COTAHIST.format(ano=ano)
    for k in range(tentativas):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=300) as resp:
                conteudo = resp.read()
            resumo = resumo_mensal(filtrar_papeis(ler_zip_cotahist(conteudo)))
            if ano == ANO_CORRENTE_B3:
                # o arquivo do ano corrente é atualizado a cada pregão: só os meses encerrados entram
                resumo = resumo[resumo["mes"] < pd.Timestamp.today().to_period("M").strftime("%Y-%m")]
            _B3_ANOS[ano] = resumo
            return resumo
        except Exception as e:
            erro = e
            _time.sleep(3 * (k + 1))
    raise RuntimeError(f"{ano}: {erro}")


def _gerar_base_b3_click(b):
    with saida_b3:
        clear_output()
        ini, fim = anos_b3.value
        partes, falhas = [], []
        print(f"⏳ Iniciando: {fim - ini + 1} arquivos anuais da B3 ({ini} a {fim}). Cada um tem dezenas de MB e "
              "leva cerca de 1 minuto — uma linha aparece a cada ano concluído. Não clique de novo.")
        for ano in range(ini, fim + 1):
            inicio = _time.time()
            ja_tinha = ano in _B3_ANOS
            try:
                resumo = _baixar_ano_b3(ano)
                partes.append(resumo)
                print(f"   ✔ {ano}: {resumo['ticker'].nunique()} papéis"
                      + (" (já estava na sessão)" if ja_tinha else f" — {_time.time() - inicio:.0f} s"))
            except Exception as e:
                falhas.append(str(e))
                print(f"   ✖ {ano}: falhou ({e})")
        if not partes:
            print("⚠️ Nenhum ano pôde ser baixado: " + "; ".join(falhas))
            return
        base = pd.concat(partes, ignore_index=True).sort_values(["ticker", "data"])
        caminho = "cotacoes_b3_mensais.csv.gz"
        base.to_csv(caminho, index=False, compression="gzip", date_format="%Y-%m-%d")
        tam = os.path.getsize(caminho) / 1e6
        print(f"✅ Base gerada: {base['ticker'].nunique()} papéis, {len(base)} linhas (papel × mês), "
              f"{base['mes'].min()} a {base['mes'].max()}, {tam:.1f} MB.")
        if falhas:
            print("⚠️ Anos que falharam (rode de novo para tentar): " + "; ".join(falhas))
        exemplo = base[base["ticker"] == "AFHI11"]
        if len(exemplo):
            print(f"   AFHI11: {len(exemplo)} meses, de {exemplo['mes'].min()} a {exemplo['mes'].max()}; "
                  f"fechamento do primeiro mês R$ {exemplo['fechamento'].iloc[0]:.2f}.")
        print("\n👉 Agora baixe o arquivo e suba no GitHub, na pasta dados/ do repositório:")
        try:
            from google.colab import files
            files.download(caminho)
        except Exception:
            print(f"   (arquivo salvo em {os.path.abspath(caminho)})")


botao_b3.on_click(_gerar_base_b3_click)
display(widgets.HTML("<h3>15. Base de cotações mensais da B3</h3>"))
display(widgets.VBox([anos_b3, botao_b3, saida_b3]))


## 🏷️ 16. Preço-teto (método Bazin com a régua do painel)

Testa se vale mostrar nos cards um **preço-teto**: o preço até o qual os dividendos ainda rendem o mínimo exigido.

> **preço-teto = dividendos por ação no ano ÷ yield mínimo**

O yield mínimo é a mesma régua do painel: **80% do CDI líquido** dos 12 meses anteriores, com piso. O Bazin original usa 6% fixos; aqui o mínimo acompanha os juros de cada época. Duas versões:
- **12 meses:** dividendos dos últimos 12 meses (é o mesmo que "yield acima da régua");
- **5 anos:** média anual dos dividendos dos últimos 5 anos, como no Bazin, que amortece um ano extraordinário ou ruim.

**Pergunta (fixada antes de rodar, 13/11/2026):** em cada fim de mês dos últimos ~10 anos, comprar com o preço **até o preço-teto** superou o CDI líquido nos 12 meses seguintes (preço + dividendos) mais vezes do que a média de todos os meses? O preço-teto vale para o card se a diferença for positiva, com o intervalo de confiança de 90% (reamostrando ativos inteiros) acima de zero. Ações e FIIs são avaliados separadamente.

**Limitações:** os proventos das ações vêm do Yahoo, que tem lacunas (Seção 14); nos FIIs, do Fundamentus quando a Seção 13 já rodou nesta sessão, e do Yahoo nos demais casos. Os preços antigos que faltam no Yahoo vêm da base da B3 (Seção 15). O teste não usa fundamentos (lucro, dívida): uma ação "abaixo do teto" pode estar barata por um bom motivo.

In [ ]:
#@title 🏷️ 16. Teste do preço-teto (ações e FIIs)
# Funções do pacote carteira_analise (as mesmas do painel)
from carteira_analise.oportunidades import avaliar_preco_teto_mes_a_mes, preco_teto_hoje, resumir_criterio
from carteira_analise.backtest_cenarios import indice_cdi_acumulado
from carteira_analise.contexto import cdi_liquido_12m
from carteira_analise.fontes.b3_cotahist import carregar_base as carregar_base_b3, completar_precos

_CASOS_TETO = globals().get("_CASOS_TETO", {"casos": None})  # último teste (para consulta depois)

ampliar_teto = widgets.Checkbox(value=True, description="Incluir também ~40 ações grandes do Ibovespa "
                                "(as mesmas da Seção 14) para ampliar a amostra",
                                layout=widgets.Layout(width='720px'))
botao_teto = widgets.Button(description="🏷️ Testar o preço-teto", button_style='warning', icon='search',
                            layout=widgets.Layout(width='300px'))
saida_teto = widgets.Output()

_ROTULOS_TETO = {"criterio_teto_12m": "preço até o teto de 12 meses",
                 "criterio_teto_5a": "preço até o teto de 5 anos (Bazin)",
                 "criterio_teto_ambos": "preço até os dois tetos"}

_AMPLIACAO_PADRAO = ("PETR4.SA VALE3.SA ITUB4.SA BBDC4.SA BBAS3.SA ITSA4.SA ABEV3.SA B3SA3.SA WEGE3.SA TAEE11.SA "
                     "CMIG4.SA CPLE6.SA EGIE3.SA SAPR11.SA SBSP3.SA VIVT3.SA TIMS3.SA KLBN11.SA SUZB3.SA GGBR4.SA "
                     "CSNA3.SA USIM5.SA BBSE3.SA CXSE3.SA PSSA3.SA RENT3.SA LREN3.SA MGLU3.SA RADL3.SA BRFS3.SA "
                     "MRVE3.SA CYRE3.SA EZTC3.SA GOAU4.SA CSMG3.SA EMBR3.SA HYPE3.SA POMO4.SA ABCB4.SA").split()


def _proventos_para_teto(ticker, tipo):
    """FIIs: Fundamentus (se a Seção 13 já rodou nesta sessão), senão Yahoo. Ações: Yahoo."""
    if tipo == "fii" and "proventos_fii_fundamentus" in globals():
        try:
            serie = proventos_fii_fundamentus(ticker)
            if serie is not None and len(serie):
                return serie, "Fundamentus"
        except Exception:
            pass
    return _dividendos_brutos_cache(ticker), "Yahoo"


def _precos_para_teto(ticker, base_b3):
    precos = _baixar_fechamento_cache(ticker, periodo='10y')
    if precos is None or precos.empty:
        return None
    precos, _, _ = limpar_outliers_precos(precos)
    if base_b3 is not None:
        eventos = eventos_yahoo(ticker) if "eventos_yahoo" in globals() else None
        precos, _ = completar_precos(precos, base_b3, ticker, eventos)
    return precos


def _imprimir_resumo_teto(casos, rotulo_grupo):
    ativos = casos["ticker"].nunique()
    print(f"\n{'=' * 70}\n{rotulo_grupo}: {ativos} ativo(s)\n{'=' * 70}")
    for coluna, rotulo in _ROTULOS_TETO.items():
        r = resumir_criterio(casos, coluna)
        if r is None or r["taxa_com_criterio"] is None:
            print(f"• {rotulo}: sem casos.")
            continue
        ic = r["ic90_diferenca"]
        dif = r["taxa_com_criterio"] - r["taxa_base"]
        veredito = ("✅ vantagem (IC acima de zero)" if ic and ic[0] > 0 else
                    "❌ desvantagem (IC abaixo de zero)" if ic and ic[1] < 0 else "➖ sem vantagem comprovada")
        print(f"• {rotulo}: superou o CDI em {r['taxa_com_criterio']:.0%} de {r['casos_com_criterio']} casos "
              f"({r['fiis_com_criterio']} ativo{'s' if r['fiis_com_criterio'] != 1 else ''}), contra {r['taxa_base']:.0%} de todos os {r['casos']} casos — "
              f"diferença de {dif * 100:+.0f} p.p."
              + (f" (IC 90%: {ic[0] * 100:+.0f} a {ic[1] * 100:+.0f} p.p.)" if ic else "") + f" → {veredito}")
        pct = lambda v: f"{v * 100:+.1f}%".replace(".", ",")
        print(f"    excesso mediano sobre o CDI em 12 meses: {pct(r['excesso_mediano_com_criterio'])} "
              f"com o critério, {pct(r['excesso_mediano_base'])} em todos os casos.")
    # perto do teto x longe dele (5 anos)
    faixas = pd.cut(casos["preco_sobre_teto_5a"], [0, 0.8, 1.0, 1.2, 1.5, np.inf],
                    labels=["até 80% do teto", "80% a 100%", "100% a 120%", "120% a 150%", "acima de 150%"])
    tabela = (casos.dropna(subset=["superou_cdi"]).assign(faixa=faixas)
              .groupby("faixa", observed=True)["superou_cdi"].agg(["mean", "size"]))
    if len(tabela):
        print("\n  Preço sobre o teto de 5 anos x superou o CDI em 12 meses:")
        for faixa, linha in tabela.iterrows():
            print(f"    {faixa:<16} {linha['mean']:.0%} de {int(linha['size'])} casos")


def _rodar_teto_click(b):
    with saida_teto:
        clear_output()
        tipos_secao2 = _tipo_por_ticker_secao2()
        acoes = [t for t, tp in tipos_secao2.items() if tp == "acao"]
        if ampliar_teto.value:
            acoes = list(dict.fromkeys(acoes + list(globals().get("ACOES_AMPLIACAO", _AMPLIACAO_PADRAO))))
        fiis = [t for t, tp in tipos_secao2.items() if tp == "fii"]
        if not acoes and not fiis:
            print("⚠️ Nenhuma ação ou FII configurado na Seção 2.")
            return
        hoje = pd.Timestamp.today().normalize()
        print("⏳ Baixando o CDI do Banco Central (11 anos)...", flush=True)
        try:
            cdi = buscar_cdi_bcb(hoje - pd.DateOffset(years=11), hoje)
        except Exception:
            cdi = None
        if cdi is None or len(cdi) == 0:
            print("⚠️ O CDI do Banco Central não pôde ser baixado. Tente de novo em alguns minutos.")
            return
        try:
            cdi_hoje = cdi_liquido_12m(cdi)  # mesmo CDI do teste: últimos 12 meses, líquido de IR
        except Exception:
            cdi_hoje = None
        base_b3 = carregar_base_b3()
        print(f"⏳ Baixando ~10 anos de preços e proventos de {len(acoes)} ações e {len(fiis)} FIIs "
              "(fica em cache na sessão)...", flush=True)
        casos, hoje_linhas, sem_dados, fontes = [], [], [], {}
        lista = [(t, "acao") for t in acoes] + [(t, "fii") for t in fiis]
        for n, (ticker, tipo) in enumerate(lista, 1):
            print(f"  [{n}/{len(lista)}] {ticker}", flush=True)
            try:
                precos = _precos_para_teto(ticker, base_b3)
                divs, fonte = _proventos_para_teto(ticker, tipo)
            except Exception:
                precos, divs, fonte = None, None, None
            if precos is None or divs is None or len(divs) == 0:
                sem_dados.append(ticker)
                continue
            fontes[fonte] = fontes.get(fonte, 0) + 1
            df = avaliar_preco_teto_mes_a_mes(precos, divs, indice_cdi_acumulado(precos.index, cdi), tipo)
            if df.empty:
                sem_dados.append(ticker)
                continue
            casos.append(df.assign(ticker=ticker, tipo=tipo))
            if ticker in tipos_secao2:
                t = preco_teto_hoje(precos, divs, cdi_hoje, tipo)
                if t:
                    hoje_linhas.append({"ticker": ticker, **t})
        clear_output()
        if not casos:
            print("⚠️ Nenhum ativo com histórico suficiente.")
            return
        casos = pd.concat(casos, ignore_index=True)
        _CASOS_TETO["casos"] = casos
        periodo = casos.dropna(subset=["superou_cdi"])["data"]
        print(f"🏷️ Teste do preço-teto — meses de {periodo.min():%m/%Y} a {periodo.max():%m/%Y} "
              f"(resultado nos 12 meses seguintes). Proventos: "
              + ", ".join(f"{f} ({n} ativos)" for f, n in fontes.items()) + ".")
        if sem_dados:
            imprimir_texto("Fora do teste (sem preços, sem proventos ou com menos de 1 ano de histórico): "
                           + ", ".join(sem_dados) + ".")
        for tipo, rotulo in (("acao", "AÇÕES B3"), ("fii", "FIIs")):
            parte = casos[casos["tipo"] == tipo]
            if len(parte):
                _imprimir_resumo_teto(parte, rotulo)
        if hoje_linhas:
            print(f"\n{'=' * 70}\nPREÇO-TETO HOJE (ativos da Seção 2) — yield mínimo: 80% do CDI líquido"
                  + (f" de {cdi_hoje * 100:.1f}%".replace('.', ',') if cdi_hoje else " (sem o CDI: limite fixo)")
                  + f"\n{'=' * 70}")
            tabela = pd.DataFrame(hoje_linhas).set_index("ticker")
            fmt = lambda v: "—" if v is None or v != v else f"{v:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
            for t, l in tabela.iterrows():
                dist = (f"{(l['preco'] / l['teto_5a'] - 1) * 100:+.0f}% do teto de 5 anos"
                        if l["teto_5a"] and l["teto_5a"] == l["teto_5a"] else "sem 5 anos de proventos")
                print(f"  {t:<10} preço {fmt(l['preco']):>8} · teto 12m {fmt(l['teto_12m']):>8} · "
                      f"teto 5 anos {fmt(l['teto_5a']):>8} · {dist}")
        print("\n📌 O preço-teto descreve o histórico de proventos; não é recomendação. Uma ação pode estar abaixo "
              "do teto por um bom motivo (lucro em queda, dividendo que não vai se repetir).")


botao_teto.on_click(_rodar_teto_click)
display(ampliar_teto, botao_teto, saida_teto)

## 📌 Notas finais

**O que os testes mostraram (e como isso mudou a ferramenta)**

- **Sinais técnicos (Seção 10).** MM200, RSI e distância do piso/topo ficaram próximos de "não fazer nada" no backtest. Por isso, desde 11/2026, a **leitura combinada** do painel e do notebook considera só **fundamentos e renda**; os sinais técnicos continuam na pontuação completa, como informação. Quando o piso e o topo usuais do período estão a menos de 10% um do outro (ativo pouco volátil), nenhum dos dois pontua.
- **Preço atrativo dos FIIs (Seção 13).** Yield acima da régua e desconto no P/VP, com renda estável, superaram o CDI nos 12 meses seguintes com mais frequência que a média, sobretudo com juros intermediários e altos. Nos FIIs de papel o sinal mais forte foi o desconto no P/VP. O critério "renda acima da inflação" (Etapa C) **não** mostrou vantagem e foi descartado. Os resultados se mantiveram com tolerâncias de queda de 5%, 10% e 15%.
- **Regularidade dos dividendos das ações (Seção 14).** Com o selo "Regular" (DY por ano civil nos 5 anos fechados), a renda se manteve no ano seguinte em 99% dos casos, contra 82% nas demais. O selo substitui a contagem de pagamentos na nota de renda das ações.

**Réguas e classificações**

- A régua de **dividend yield** de ações, FIIs e ETFs brasileiros é **80% do CDI líquido** do momento (com um piso mínimo), e acompanha os juros sozinha. Ações dos EUA usam limite fixo.
- FIIs são classificados como **tijolo, papel ou híbrido** pelo segmento e conferidos pela proporção de imóveis no patrimônio (Imóveis/PL, Fundamentus): abaixo de 20%, papel; abaixo de 50%, híbrido.

**Fontes de dados e limitações**

- Cotações e proventos vêm do **Yahoo Finance**; fundamentos, rendimentos de FIIs e proventos de ações, do **Fundamentus**; CDI e IPCA, do **Banco Central (SGS)**. Quando o Yahoo tem lacunas nos proventos, o pacote confere no Fundamentus.
- O Yahoo começa tarde para alguns papéis (ex.: AFHI11, só desde 01/2024). A **base da B3 (Seção 15)** completa os meses anteriores com o fechamento oficial; ela precisa ser regerada de tempos em tempos para incluir os meses encerrados.
- Fundamentos incompletos simplesmente não pontuam (são tratados como neutros), o que pode enviesar o resultado. Nenhuma fonte gratuita tem o histórico de fundamentos, então o motor completo não pode ser testado retroativamente — só as partes acima. Se um número parecer estranho, confira na fonte original (RI da empresa/fundo, B3).
- **Notebook e painel usam o mesmo código.** A Seção 1 baixa o pacote `carteira_analise` do repositório do projeto (versão escolhida em `VERSAO_PACOTE`), e as seções importam dele o motor de pontuação, a regularidade dos dividendos, o preço atrativo dos FIIs, os cenários e a base da B3, cobertos pelos testes automatizados do pacote. No notebook ficam a interface (botões e tabelas), a busca de dados com cache da sessão e os testes estatísticos das Seções 10 a 15.

**Uso responsável**

- Os critérios da Seção 6 são **regras de bolso** de mercado: ponto de partida, sem garantia de retorno e sem substituir análise mais profunda (relatórios setoriais, balanços completos, contexto macroeconômico).
- Os cenários da Seção 11 usam uma **regra de faixas de preço** (realizar lucro na alta, aumentar a posição na queda, com limites) só para decidir **quais caminhos comparar** — não é previsão nem recomendação. O resultado histórico de cada cenário (Seção 12) aparece no próprio card.
- Este notebook **não é uma recomendação de investimento** nem substitui um assessor/consultor licenciado (CVM).
